In [1]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.4
NumPy: 2.5.0


In [2]:
# LeadIQ — Historical Timeline

import pandas as pd
import numpy as np

np.random.seed(42)

n = 20000

timestamps = pd.date_range(
    start="2025-01-01",
    end="2026-06-30",
    periods=n
)

print("Number of leads:", len(timestamps))
print("First lead:", timestamps.min())
print("Last lead:", timestamps.max())

print("\nFirst 5 timestamps:")
print(timestamps[:5])

Number of leads: 20000
First lead: 2025-01-01 00:00:00
Last lead: 2026-06-30 00:00:00

First 5 timestamps:
DatetimeIndex([       '2025-01-01 00:00:00', '2025-01-01 00:39:14.517725',
               '2025-01-01 01:18:29.035451', '2025-01-01 01:57:43.553177',
               '2025-01-01 02:36:58.070903'],
              dtype='datetime64[us]', freq=None)


In [3]:
# LeadIQ — Generate inquiry-time features

rng = np.random.default_rng(42)

df_new = pd.DataFrame({
    "timestamp": timestamps,

    "buyer_industry": rng.choice(
        [
            "Manufacturing", "Retail", "Wholesale", "Chemicals",
            "Pharmaceuticals", "Food & Beverage", "Automotive",
            "Electronics", "Construction", "Textiles"
        ],
        n
    ),

    "buyer_location": rng.choice(
        [
            "Mumbai", "Delhi NCR", "Bengaluru", "Hyderabad",
            "Chennai", "Pune", "Kolkata", "Ahmedabad",
            "Jaipur", "Indore"
        ],
        n
    ),

    "buyer_company_size": rng.choice(
        ["Micro", "Small", "Medium", "Large", "Enterprise"],
        n,
        p=[0.15, 0.25, 0.30, 0.20, 0.10]
    ),

    "product_category": rng.choice(
        [
            "Industrial Equipment", "Electrical Components",
            "Packaging", "Chemicals", "Steel & Metal",
            "Food Ingredients", "Textile Machinery",
            "Office Supplies", "Construction Materials", "Auto Parts"
        ],
        n
    ),

    "urgency": rng.choice(
        ["Low", "Medium", "High", "Critical"],
        n,
        p=[0.20, 0.40, 0.30, 0.10]
    ),

    "lead_source": rng.choice(
        [
            "Marketplace Search", "Direct Inquiry", "Referral",
            "Organic Search", "Paid Search", "Email Campaign"
        ],
        n
    ),

    "inquiry_hour": rng.integers(0, 24, n),

    "seller_rating": np.round(
        np.clip(rng.normal(4.15, 0.45, n), 2.5, 5.0), 2
    ),

    "seller_experience": np.round(
        np.clip(rng.gamma(3.0, 2.5, n), 0.5, 20), 1
    )
})

print("Shape:", df_new.shape)
print("\nColumns:")
print(df_new.columns.tolist())

display(df_new.head())

Shape: (20000, 10)

Columns:
['timestamp', 'buyer_industry', 'buyer_location', 'buyer_company_size', 'product_category', 'urgency', 'lead_source', 'inquiry_hour', 'seller_rating', 'seller_experience']


,timestamp,buyer_industry,buyer_location,buyer_company_size,product_category,urgency,lead_source,inquiry_hour,seller_rating,seller_experience
0,2025-01-01 00:00:00.000000,Manufacturing,Mumbai,Large,Chemicals,High,Referral,23,3.83,8.8
1,2025-01-01 00:39:14.517725,Electronics,Ahmedabad,Enterprise,Packaging,Medium,Direct Inquiry,15,5.00,4.6
2,2025-01-01 01:18:29.035451,Automotive,Bengaluru,Micro,Industrial Equipment,High,Email Campaign,0,3.62,6.9
3,2025-01-01 01:57:43.553177,Pharmaceuticals,Ahmedabad,Small,Electrical Components,Low,Direct Inquiry,3,4.38,15.9
4,2025-01-01 02:36:58.070903,Pharmaceuticals,Pune,Medium,Electrical Components,Medium,Marketplace Search,16,3.30,3.2


In [4]:
# LeadIQ — Generate inquiry economics

category_multiplier = {
    "Industrial Equipment": 1.8,
    "Electrical Components": 1.2,
    "Packaging": 0.8,
    "Chemicals": 1.5,
    "Steel & Metal": 1.7,
    "Food Ingredients": 0.9,
    "Textile Machinery": 1.6,
    "Office Supplies": 0.6,
    "Construction Materials": 1.3,
    "Auto Parts": 1.1
}

df_new["quantity"] = np.maximum(
    1,
    rng.poisson(12, n) + 1
)

df_new["inquiry_value"] = np.round(
    df_new["quantity"]
    * rng.uniform(700, 1800, n)
    * df_new["product_category"].map(category_multiplier),
    2
)

print("Quantity:")
print(df_new["quantity"].describe().round(2))

print("\nInquiry value:")
print(df_new["inquiry_value"].describe().round(2))

print("\nSample:")
display(
    df_new[
        [
            "product_category",
            "quantity",
            "inquiry_value"
        ]
    ].head(10)
)

Quantity:
count    20000.00
mean        12.93
std          3.44
min          2.00
25%         11.00
50%         13.00
75%         15.00
max         28.00
Name: quantity, dtype: float64

Inquiry value:
count    20000.00
mean     20205.91
std      10032.38
min       1945.41
25%      12691.64
50%      18337.78
75%      25878.90
max      75864.58
Name: inquiry_value, dtype: float64

Sample:


,product_category,quantity,inquiry_value
0,Chemicals,9,16951.55
1,Packaging,15,8538.26
2,Industrial Equipment,20,55079.57
3,Electrical Components,10,20853.88
4,Electrical Components,13,23376.75
5,Electrical Components,17,18753.26
6,Auto Parts,12,20468.96
7,Industrial Equipment,9,11479.84
8,Chemicals,13,24684.85
9,Auto Parts,18,31233.66


In [5]:
# LeadIQ — Create buyers and historical inquiry counts

# Assign each lead to a buyer
n_buyers = 5000

df_new["buyer_id"] = rng.integers(
    1,
    n_buyers + 1,
    size=n
)

# Count how many inquiries this buyer had made BEFORE the current lead
df_new["previous_inquiries"] = (
    df_new
    .groupby("buyer_id")
    .cumcount()
)

# Repeat buyer flag
df_new["repeat_buyer"] = (
    df_new["previous_inquiries"] > 0
).astype(int)

print("Unique buyers:", df_new["buyer_id"].nunique())

print("\nPrevious inquiries:")
print(df_new["previous_inquiries"].describe().round(2))

print("\nRepeat buyer distribution:")
print(df_new["repeat_buyer"].value_counts())

print("\nSample:")
display(
    df_new[
        [
            "buyer_id",
            "previous_inquiries",
            "repeat_buyer"
        ]
    ].head(15)
)

Unique buyers: 4915

Previous inquiries:
count    20000.00
mean         1.99
std          1.80
min          0.00
25%          1.00
50%          2.00
75%          3.00
max         13.00
Name: previous_inquiries, dtype: float64

Repeat buyer distribution:
repeat_buyer
1    15085
0     4915
Name: count, dtype: int64

Sample:


,buyer_id,previous_inquiries,repeat_buyer
0,3692,0,0
1,3147,0,0
2,3611,0,0
3,965,0,0
4,4382,0,0
5,620,0,0
6,2662,0,0
7,4351,0,0
8,384,0,0
9,4093,0,0


In [6]:
n = 20000

timestamps = pd.date_range(
    start="2025-01-01",
    end="2026-06-30",
    periods=n
)

print("Number of timestamps:", len(timestamps))
print("First:", timestamps.min())
print("Last:", timestamps.max())

Number of timestamps: 20000
First: 2025-01-01 00:00:00
Last: 2026-06-30 00:00:00


In [7]:
n_buyers = 5000
n_sellers = 1000

rng = np.random.default_rng(42)

df_new = pd.DataFrame({
    "timestamp": timestamps,
    "buyer_id": rng.integers(1, n_buyers + 1, size=n),
    "seller_id": rng.integers(1, n_sellers + 1, size=n)
})

print("Shape:", df_new.shape)
print("Unique buyers:", df_new["buyer_id"].nunique())
print("Unique sellers:", df_new["seller_id"].nunique())
print(df_new.head())

Shape: (20000, 3)
Unique buyers: 4901
Unique sellers: 1000
                   timestamp  buyer_id  seller_id
0 2025-01-01 00:00:00.000000       447         35
1 2025-01-01 00:39:14.517725      3870        721
2 2025-01-01 01:18:29.035451      3273        214
3 2025-01-01 01:57:43.553177      2195        712
4 2025-01-01 02:36:58.070903      2166        565


In [8]:
df_new["buyer_industry"] = rng.choice(
    ["Manufacturing", "Retail", "Construction", "Healthcare", "Automotive"],
    size=n
)

df_new["buyer_location"] = rng.choice(
    ["Mumbai", "Delhi", "Bengaluru", "Pune", "Ahmedabad", "Chennai", "Hyderabad"],
    size=n
)

df_new["buyer_company_size"] = rng.choice(
    ["Micro", "Small", "Medium", "Large", "Enterprise"],
    size=n,
    p=[0.10, 0.25, 0.30, 0.20, 0.15]
)

df_new["product_category"] = rng.choice(
    [
        "Industrial Equipment",
        "Electrical Components",
        "Packaging",
        "Chemicals",
        "Steel & Metal",
        "Food Ingredients",
        "Textile Machinery",
        "Office Supplies",
        "Construction Materials",
        "Auto Parts"
    ],
    size=n
)

df_new["urgency"] = rng.choice(
    ["Low", "Medium", "High", "Critical"],
    size=n,
    p=[0.20, 0.40, 0.30, 0.10]
)

df_new["lead_source"] = rng.choice(
    [
        "Marketplace Search",
        "Direct Inquiry",
        "Referral",
        "Organic Search",
        "Paid Search",
        "Email Campaign"
    ],
    size=n
)

df_new["inquiry_hour"] = df_new["timestamp"].dt.hour

print("Shape:", df_new.shape)
print(df_new.head())

Shape: (20000, 10)
                   timestamp  buyer_id  seller_id buyer_industry  \
0 2025-01-01 00:00:00.000000       447         35     Automotive   
1 2025-01-01 00:39:14.517725      3870        721     Healthcare   
2 2025-01-01 01:18:29.035451      3273        214         Retail   
3 2025-01-01 01:57:43.553177      2195        712     Automotive   
4 2025-01-01 02:36:58.070903      2166        565     Automotive   

  buyer_location buyer_company_size        product_category urgency  \
0         Mumbai              Small    Industrial Equipment  Medium   
1      Bengaluru              Small  Construction Materials  Medium   
2        Chennai              Small   Electrical Components  Medium   
3      Bengaluru              Micro         Office Supplies  Medium   
4          Delhi         Enterprise    Industrial Equipment  Medium   

          lead_source  inquiry_hour  
0  Marketplace Search             0  
1      Organic Search             0  
2  Marketplace Search          

In [9]:
df_new["seller_rating"] = np.clip(
    rng.normal(4.1, 0.45, n),
    2.5,
    5.0
).round(2)

df_new["seller_experience"] = np.clip(
    rng.normal(6.5, 3.0, n),
    0.5,
    20
).round(1)

print(df_new[["seller_rating", "seller_experience"]].describe())

       seller_rating  seller_experience
count   20000.000000       20000.000000
mean        4.094590           6.529180
std         0.441013           2.945051
min         2.500000           0.500000
25%         3.790000           4.500000
50%         4.100000           6.500000
75%         4.400000           8.500000
max         5.000000          20.000000


In [10]:
df_new = df_new.sort_values("timestamp").reset_index(drop=True)

df_new["previous_inquiries"] = (
    df_new.groupby("buyer_id").cumcount()
)

df_new["repeat_buyer"] = (
    df_new["previous_inquiries"] > 0
).astype(int)

print(df_new[["previous_inquiries", "repeat_buyer"]].describe())
print("\nRepeat buyers:", df_new["repeat_buyer"].sum())
print("First-time inquiries:", (df_new["repeat_buyer"] == 0).sum())

       previous_inquiries  repeat_buyer
count        20000.000000  20000.000000
mean             1.994600      0.754950
std              1.815994      0.430128
min              0.000000      0.000000
25%              1.000000      1.000000
50%              2.000000      1.000000
75%              3.000000      1.000000
max             12.000000      1.000000

Repeat buyers: 15099
First-time inquiries: 4901


In [11]:
buyer_propensity = pd.DataFrame({
    "buyer_id": np.arange(1, n_buyers + 1),
    "buyer_propensity": rng.beta(2.5, 4.5, n_buyers)
})

df_new = df_new.merge(
    buyer_propensity,
    on="buyer_id",
    how="left"
)

print(df_new["buyer_propensity"].describe())

count    20000.000000
mean         0.355745
std          0.173023
min          0.013737
25%          0.220769
50%          0.340439
75%          0.470985
max          0.907145
Name: buyer_propensity, dtype: float64


In [12]:
df_new["previous_orders"] = np.where(
    df_new["previous_inquiries"] == 0,
    0,
    rng.binomial(
        df_new["previous_inquiries"],
        df_new["buyer_propensity"]
    )
)

print(df_new["previous_orders"].describe())
print("\nPrevious orders vs previous inquiries:")
print(
    df_new[["previous_inquiries", "previous_orders"]]
    .head(10)
)

count    20000.000000
mean         0.706950
std          1.009318
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          8.000000
Name: previous_orders, dtype: float64

Previous orders vs previous inquiries:
   previous_inquiries  previous_orders
0                   0                0
1                   0                0
2                   0                0
3                   0                0
4                   0                0
5                   0                0
6                   0                0
7                   0                0
8                   0                0
9                   0                0


In [13]:
df_new["quantity"] = np.maximum(
    1,
    rng.poisson(12, n) + 1
)

category_multiplier = {
    "Industrial Equipment": 1.8,
    "Electrical Components": 1.2,
    "Packaging": 0.8,
    "Chemicals": 1.5,
    "Steel & Metal": 1.7,
    "Food Ingredients": 0.9,
    "Textile Machinery": 1.6,
    "Office Supplies": 0.6,
    "Construction Materials": 1.3,
    "Auto Parts": 1.1
}

df_new["inquiry_value"] = (
    df_new["quantity"]
    * rng.uniform(700, 1800, n)
    * df_new["product_category"].map(category_multiplier)
).round(2)

print(df_new[["quantity", "inquiry_value"]].describe())

           quantity  inquiry_value
count  20000.000000   20000.000000
mean      12.936800   20282.694962
std        3.423798   10050.606491
min        2.000000    2089.620000
25%       11.000000   12677.485000
50%       13.000000   18540.210000
75%       15.000000   26113.510000
max       27.000000   68614.990000


In [14]:
df_new["buyer_seller_previous_inquiries"] = (
    df_new
    .groupby(["buyer_id", "seller_id"])
    .cumcount()
)

df_new["buyer_seller_previous_orders"] = 0

print(
    df_new[
        [
            "buyer_id",
            "seller_id",
            "buyer_seller_previous_inquiries",
            "buyer_seller_previous_orders"
        ]
    ].head(10)
)

print(
    "\nRepeat buyer-seller inquiries:",
    (df_new["buyer_seller_previous_inquiries"] > 0).sum()
)

   buyer_id  seller_id  buyer_seller_previous_inquiries  \
0       447         35                                0   
1      3870        721                                0   
2      3273        214                                0   
3      2195        712                                0   
4      2166        565                                0   
5      4293        203                                0   
6       430        470                                0   
7      3487         37                                0   
8      1008        702                                0   
9       471        304                                0   

   buyer_seller_previous_orders  
0                             0  
1                             0  
2                             0  
3                             0  
4                             0  
5                             0  
6                             0  
7                             0  
8                             0  
9         

In [15]:
repeat_rate = (
    df_new["buyer_seller_previous_inquiries"] > 0
).mean()

print(f"Buyer-seller repeat inquiry rate: {repeat_rate:.2%}")
print(
    "Unique buyer-seller pairs:",
    df_new[["buyer_id", "seller_id"]].drop_duplicates().shape[0]
)

Buyer-seller repeat inquiry rate: 0.19%
Unique buyer-seller pairs: 19962


In [16]:
seller_quality = pd.DataFrame({
    "seller_id": np.arange(1, n_sellers + 1),
    "seller_quality": rng.beta(5, 2, n_sellers)
})

df_new = df_new.merge(
    seller_quality,
    on="seller_id",
    how="left"
)

print(df_new["seller_quality"].describe())

count    20000.000000
mean         0.711181
std          0.162584
min          0.191628
25%          0.608883
50%          0.734876
75%          0.835774
max          0.992500
Name: seller_quality, dtype: float64


In [17]:
urgency_effect = df_new["urgency"].map({
    "Low": -0.25,
    "Medium": 0.00,
    "High": 0.20,
    "Critical": 0.35
})

company_effect = df_new["buyer_company_size"].map({
    "Micro": -0.15,
    "Small": -0.05,
    "Medium": 0.00,
    "Large": 0.10,
    "Enterprise": 0.18
})

logit = (
    -2.2
    + 1.4 * df_new["buyer_propensity"]
    + 0.8 * df_new["seller_quality"]
    + 0.20 * np.log1p(df_new["previous_orders"])
    + 0.15 * np.log1p(df_new["inquiry_value"])
    + urgency_effect
    + company_effect
    + 0.10 * df_new["repeat_buyer"]
)

conversion_probability = 1 / (1 + np.exp(-logit))

df_new["converted"] = (
    rng.random(n) < conversion_probability
).astype(int)

print(df_new["converted"].value_counts())
print("\nConversion rate:", df_new["converted"].mean())

converted
1    12618
0     7382
Name: count, dtype: int64

Conversion rate: 0.6309


In [18]:
print("Average predicted probability:", conversion_probability.mean())
print("Min probability:", conversion_probability.min())
print("Max probability:", conversion_probability.max())
print("\nProbability quartiles:")
print(conversion_probability.quantile([0.25, 0.50, 0.75]))

Average predicted probability: 0.6308736180383885
Min probability: 0.3100488772103088
Max probability: 0.8924460015093457

Probability quartiles:
0.25    0.568787
0.50    0.632373
0.75    0.694606
dtype: float64


In [19]:
# LeadIQ API input contract

api_schema = {
    "lead_id": "string",
    "timestamp": "datetime",
    "buyer_id": "string",
    "seller_id": "string",
    
    "buyer_industry": "string",
    "buyer_location": "string",
    "buyer_company_size": "string",
    
    "product_category": "string",
    "inquiry_value": "float",
    "quantity": "integer",
    "urgency": "string",
    "lead_source": "string",
    
    "seller_rating": "float",
    "seller_experience": "float"
}

print("LeadIQ API fields:")
for field, dtype in api_schema.items():
    print(f"{field:25} → {dtype}")

print("\nTotal API fields:", len(api_schema))

LeadIQ API fields:
lead_id                   → string
timestamp                 → datetime
buyer_id                  → string
seller_id                 → string
buyer_industry            → string
buyer_location            → string
buyer_company_size        → string
product_category          → string
inquiry_value             → float
quantity                  → integer
urgency                   → string
lead_source               → string
seller_rating             → float
seller_experience         → float

Total API fields: 14


In [20]:
def generate_lead(lead_id, timestamp, rng):
    lead = {
        "lead_id": lead_id,
        "timestamp": timestamp,
        "buyer_id": f"B{rng.integers(1, 5001):04d}",
        "seller_id": f"S{rng.integers(1, 1001):04d}",
        
        "buyer_industry": rng.choice([
            "Manufacturing", "Retail", "Construction",
            "Healthcare", "Automotive"
        ]),
        
        "buyer_location": rng.choice([
            "Mumbai", "Delhi", "Bengaluru", "Pune",
            "Ahmedabad", "Chennai", "Hyderabad"
        ]),
        
        "buyer_company_size": rng.choice(
            ["Micro", "Small", "Medium", "Large", "Enterprise"],
            p=[0.10, 0.25, 0.30, 0.20, 0.15]
        ),
        
        "product_category": rng.choice([
            "Industrial Equipment", "Electrical Components",
            "Packaging", "Chemicals", "Steel & Metal",
            "Food Ingredients", "Textile Machinery",
            "Office Supplies", "Construction Materials",
            "Auto Parts"
        ]),
        
        "inquiry_value": round(rng.uniform(2000, 70000), 2),
        "quantity": int(rng.integers(1, 30)),
        
        "urgency": rng.choice(
            ["Low", "Medium", "High", "Critical"],
            p=[0.20, 0.40, 0.30, 0.10]
        ),
        
        "lead_source": rng.choice([
            "Marketplace Search", "Direct Inquiry", "Referral",
            "Organic Search", "Paid Search", "Email Campaign"
        ]),
        
        "seller_rating": round(
            np.clip(rng.normal(4.1, 0.45), 2.5, 5.0), 2
        ),
        
        "seller_experience": round(
            np.clip(rng.normal(6.5, 3.0), 0.5, 20), 1
        )
    }
    
    return lead

In [21]:
test_lead = generate_lead(
    lead_id="L000001",
    timestamp=pd.Timestamp.now(),
    rng=np.random.default_rng(42)
)

test_lead

{'lead_id': 'L000001',
 'timestamp': Timestamp('2026-10-07 14:31:44.871709'),
 'buyer_id': 'B0447',
 'seller_id': 'S0774',
 'buyer_industry': np.str_('Healthcare'),
 'buyer_location': np.str_('Pune'),
 'buyer_company_size': np.str_('Enterprise'),
 'product_category': np.str_('Industrial Equipment'),
 'inquiry_value': 8404.06,
 'quantity': 21,
 'urgency': np.str_('Critical'),
 'lead_source': np.str_('Paid Search'),
 'seller_rating': np.float64(3.96),
 'seller_experience': np.float64(6.4)}

In [22]:
# Persistent marketplace profiles

rng = np.random.default_rng(42)

buyers = pd.DataFrame({
    "buyer_id": [f"B{i:04d}" for i in range(1, 5001)],
    "buyer_propensity": rng.beta(2.5, 4.5, 5000)
})

sellers = pd.DataFrame({
    "seller_id": [f"S{i:04d}" for i in range(1, 1001)],
    "seller_quality": rng.beta(5, 2, 1000)
})

print("Buyers:", buyers.shape)
print("Sellers:", sellers.shape)

print("\nBuyer profile sample:")
print(buyers.head())

print("\nSeller profile sample:")
print(sellers.head())

Buyers: (5000, 2)
Sellers: (1000, 2)

Buyer profile sample:
  buyer_id  buyer_propensity
0    B0001          0.309909
1    B0002          0.363532
2    B0003          0.464739
3    B0004          0.371112
4    B0005          0.495580

Seller profile sample:
  seller_id  seller_quality
0     S0001        0.847238
1     S0002        0.922438
2     S0003        0.633296
3     S0004        0.839752
4     S0005        0.706277


In [23]:
def generate_lead(lead_id, timestamp, rng):
    
    # Select existing marketplace participants
    buyer_id = rng.choice(buyers["buyer_id"])
    seller_id = rng.choice(sellers["seller_id"])
    
    buyer = buyers.loc[buyers["buyer_id"] == buyer_id].iloc[0]
    seller = sellers.loc[sellers["seller_id"] == seller_id].iloc[0]

    lead = {
        "lead_id": lead_id,
        "timestamp": timestamp,
        "buyer_id": buyer_id,
        "seller_id": seller_id,

        "buyer_industry": rng.choice([
            "Manufacturing", "Retail", "Construction",
            "Healthcare", "Automotive"
        ]),

        "buyer_location": rng.choice([
            "Mumbai", "Delhi", "Bengaluru", "Pune",
            "Ahmedabad", "Chennai", "Hyderabad"
        ]),

        "buyer_company_size": rng.choice(
            ["Micro", "Small", "Medium", "Large", "Enterprise"],
            p=[0.10, 0.25, 0.30, 0.20, 0.15]
        ),

        "product_category": rng.choice([
            "Industrial Equipment", "Electrical Components",
            "Packaging", "Chemicals", "Steel & Metal",
            "Food Ingredients", "Textile Machinery",
            "Office Supplies", "Construction Materials",
            "Auto Parts"
        ]),

        "inquiry_value": round(rng.uniform(2000, 70000), 2),
        "quantity": int(rng.integers(1, 30)),

        "urgency": rng.choice(
            ["Low", "Medium", "High", "Critical"],
            p=[0.20, 0.40, 0.30, 0.10]
        ),

        "lead_source": rng.choice([
            "Marketplace Search", "Direct Inquiry", "Referral",
            "Organic Search", "Paid Search", "Email Campaign"
        ]),

        "seller_rating": round(
            np.clip(rng.normal(4.1, 0.45), 2.5, 5.0), 2
        ),

        "seller_experience": round(
            np.clip(rng.normal(6.5, 3.0), 0.5, 20), 1
        )
    }

    return lead

In [24]:
# Historical state for buyers, sellers, and buyer-seller relationships

buyer_history = {
    buyer_id: {
        "previous_inquiries": 0,
        "previous_orders": 0,
        "converted_leads": 0
    }
    for buyer_id in buyers["buyer_id"]
}

seller_history = {
    seller_id: {
        "total_leads": 0,
        "converted_leads": 0
    }
    for seller_id in sellers["seller_id"]
}

buyer_seller_history = {}

print("Buyer history records:", len(buyer_history))
print("Seller history records:", len(seller_history))
print("Buyer-seller relationships:", len(buyer_seller_history))

Buyer history records: 5000
Seller history records: 1000
Buyer-seller relationships: 0


In [25]:
def enrich_lead_with_history(lead):
    buyer_id = lead["buyer_id"]
    seller_id = lead["seller_id"]
    
    buyer = buyer_history[buyer_id]
    seller = seller_history[seller_id]
    
    pair_key = (buyer_id, seller_id)
    pair = buyer_seller_history.get(
        pair_key,
        {
            "previous_inquiries": 0,
            "previous_orders": 0
        }
    )
    
    previous_inquiries = buyer["previous_inquiries"]
    previous_orders = buyer["previous_orders"]
    
    # Conversion rate is calculated only from past leads
    if previous_inquiries > 0:
        previous_conversion_rate = (
            buyer["converted_leads"] / previous_inquiries
        )
    else:
        previous_conversion_rate = 0.0
    
    enriched_lead = lead.copy()
    
    enriched_lead.update({
        "previous_orders": previous_orders,
        "previous_inquiries": previous_inquiries,
        "repeat_buyer": int(previous_inquiries > 0),
        "previous_conversion_rate": round(
            previous_conversion_rate, 4
        ),
        "buyer_seller_previous_inquiries": pair[
            "previous_inquiries"
        ],
        "buyer_seller_previous_orders": pair[
            "previous_orders"
        ]
    })
    
    return enriched_lead

In [26]:
lead = generate_lead(
    lead_id="L000001",
    timestamp=pd.Timestamp.now(),
    rng=rng
)

enriched_lead = enrich_lead_with_history(lead)

print(enriched_lead)

{'lead_id': 'L000001', 'timestamp': Timestamp('2026-10-07 14:33:56.407604'), 'buyer_id': 'B1255', 'seller_id': 'S0978', 'buyer_industry': np.str_('Automotive'), 'buyer_location': np.str_('Delhi'), 'buyer_company_size': np.str_('Medium'), 'product_category': np.str_('Packaging'), 'inquiry_value': 42845.87, 'quantity': 7, 'urgency': np.str_('High'), 'lead_source': np.str_('Marketplace Search'), 'seller_rating': np.float64(3.91), 'seller_experience': np.float64(9.4), 'previous_orders': 0, 'previous_inquiries': 0, 'repeat_buyer': 0, 'previous_conversion_rate': 0.0, 'buyer_seller_previous_inquiries': 0, 'buyer_seller_previous_orders': 0}


In [27]:
def update_history(lead, converted):
    buyer_id = lead["buyer_id"]
    seller_id = lead["seller_id"]
    pair_key = (buyer_id, seller_id)

    # Update buyer history
    buyer_history[buyer_id]["previous_inquiries"] += 1
    buyer_history[buyer_id]["converted_leads"] += converted
    buyer_history[buyer_id]["previous_orders"] += converted

    # Update seller history
    seller_history[seller_id]["total_leads"] += 1
    seller_history[seller_id]["converted_leads"] += converted

    # Update buyer-seller relationship
    if pair_key not in buyer_seller_history:
        buyer_seller_history[pair_key] = {
            "previous_inquiries": 0,
            "previous_orders": 0
        }

    buyer_seller_history[pair_key]["previous_inquiries"] += 1
    buyer_seller_history[pair_key]["previous_orders"] += converted

    return True

In [28]:
# First inquiry
lead_1 = generate_lead(
    lead_id="L000001",
    timestamp=pd.Timestamp.now(),
    rng=rng
)

enriched_1 = enrich_lead_with_history(lead_1)

print("BEFORE outcome:")
print({
    "buyer_id": enriched_1["buyer_id"],
    "previous_inquiries": enriched_1["previous_inquiries"],
    "previous_orders": enriched_1["previous_orders"],
    "repeat_buyer": enriched_1["repeat_buyer"]
})

# Assume this lead converted
update_history(enriched_1, converted=1)

# Same buyer appears again
lead_2 = lead_1.copy()
lead_2["lead_id"] = "L000002"

enriched_2 = enrich_lead_with_history(lead_2)

print("\nAFTER previous conversion:")
print({
    "buyer_id": enriched_2["buyer_id"],
    "previous_inquiries": enriched_2["previous_inquiries"],
    "previous_orders": enriched_2["previous_orders"],
    "repeat_buyer": enriched_2["repeat_buyer"],
    "previous_conversion_rate": enriched_2["previous_conversion_rate"]
})

BEFORE outcome:
{'buyer_id': 'B3285', 'previous_inquiries': 0, 'previous_orders': 0, 'repeat_buyer': 0}

AFTER previous conversion:
{'buyer_id': 'B3285', 'previous_inquiries': 1, 'previous_orders': 1, 'repeat_buyer': 1, 'previous_conversion_rate': 1.0}


In [29]:
# Simulate a continuous stream of marketplace inquiries

n_events = 20000
start_time = pd.Timestamp("2025-01-01")

event_records = []

for i in range(n_events):
    timestamp = start_time + pd.Timedelta(minutes=i * 30)

    lead = generate_lead(
        lead_id=f"L{i+1:06d}",
        timestamp=timestamp,
        rng=rng
    )

    enriched_lead = enrich_lead_with_history(lead)

    event_records.append(enriched_lead)

print("Events generated:", len(event_records))

historical_api_df = pd.DataFrame(event_records)

print("Shape:", historical_api_df.shape)
print("\nFirst event:")
print(historical_api_df.iloc[0])

print("\nLast event:")
print(historical_api_df.iloc[-1])

Events generated: 20000
Shape: (20000, 20)

First event:
lead_id                                        L000001
timestamp                          2025-01-01 00:00:00
buyer_id                                         B2610
seller_id                                        S0087
buyer_industry                                  Retail
buyer_location                               Bengaluru
buyer_company_size                               Small
product_category                         Steel & Metal
inquiry_value                                 55937.68
quantity                                             9
urgency                                           High
lead_source                             Organic Search
seller_rating                                     3.65
seller_experience                                  2.1
previous_orders                                      0
previous_inquiries                                   0
repeat_buyer                                         0
previous

In [30]:
# Reset historical state

buyer_history = {
    buyer_id: {
        "previous_inquiries": 0,
        "previous_orders": 0,
        "converted_leads": 0
    }
    for buyer_id in buyers["buyer_id"]
}

seller_history = {
    seller_id: {
        "total_leads": 0,
        "converted_leads": 0
    }
    for seller_id in sellers["seller_id"]
}

buyer_seller_history = {}

event_records = []

n_events = 20000
start_time = pd.Timestamp("2025-01-01")

for i in range(n_events):

    timestamp = start_time + pd.Timedelta(minutes=i * 30)

    # 1. New marketplace inquiry
    lead = generate_lead(
        lead_id=f"L{i+1:06d}",
        timestamp=timestamp,
        rng=rng
    )

    # 2. Enrich using ONLY information available before this inquiry
    enriched_lead = enrich_lead_with_history(lead)

    event_records.append(enriched_lead)

    # 3. Temporary outcome mechanism
    buyer_propensity = buyers.loc[
        buyers["buyer_id"] == lead["buyer_id"],
        "buyer_propensity"
    ].iloc[0]

    seller_quality = sellers.loc[
        sellers["seller_id"] == lead["seller_id"],
        "seller_quality"
    ].iloc[0]

    urgency_effect = {
        "Low": -0.20,
        "Medium": 0.00,
        "High": 0.18,
        "Critical": 0.30
    }[lead["urgency"]]

    company_effect = {
        "Micro": -0.12,
        "Small": -0.04,
        "Medium": 0.00,
        "Large": 0.08,
        "Enterprise": 0.14
    }[lead["buyer_company_size"]]

    logit = (
        -3.6
        + 1.0 * buyer_propensity
        + 0.55 * seller_quality
        + 0.15 * np.log1p(
            enriched_lead["previous_orders"]
        )
        + 0.10 * np.log1p(
            enriched_lead["inquiry_value"]
        )
        + urgency_effect
        + company_effect
        + 0.08 * enriched_lead["repeat_buyer"]
    )

    probability = 1 / (1 + np.exp(-logit))

    converted = int(rng.random() < probability)

    # 4. Outcome happens AFTER the inquiry
    update_history(
        enriched_lead,
        converted=converted
    )

# Final training/event dataset
historical_api_df = pd.DataFrame(event_records)

print("Shape:", historical_api_df.shape)

print("\nFirst 5 events:")
print(
    historical_api_df[
        ["lead_id", "buyer_id", "previous_inquiries",
         "previous_orders", "repeat_buyer"]
    ].head()
)

print("\nLast 5 events:")
print(
    historical_api_df[
        ["lead_id", "buyer_id", "previous_inquiries",
         "previous_orders", "repeat_buyer"]
    ].tail()
)

Shape: (20000, 20)

First 5 events:
   lead_id buyer_id  previous_inquiries  previous_orders  repeat_buyer
0  L000001    B2899                   0                0             0
1  L000002    B0749                   0                0             0
2  L000003    B1196                   0                0             0
3  L000004    B3711                   0                0             0
4  L000005    B1751                   0                0             0

Last 5 events:
       lead_id buyer_id  previous_inquiries  previous_orders  repeat_buyer
19995  L019996    B4915                   5                0             1
19996  L019997    B3706                   2                0             1
19997  L019998    B1514                   2                0             1
19998  L019999    B1549                   5                2             1
19999  L020000    B4502                   3                0             1


In [31]:
print("Repeat buyer rate:")
print(historical_api_df["repeat_buyer"].mean())

print("\nPrevious inquiries:")
print(historical_api_df["previous_inquiries"].describe())

print("\nPrevious orders:")
print(historical_api_df["previous_orders"].describe())

print("\nPrevious conversion rate:")
print(historical_api_df["previous_conversion_rate"].describe())

print("\nBuyer-seller repeat inquiry rate:")
print(
    (historical_api_df["buyer_seller_previous_inquiries"] > 0).mean()
)

Repeat buyer rate:
0.7553

Previous inquiries:
count    20000.000000
mean         2.009450
std          1.837506
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         12.000000
Name: previous_inquiries, dtype: float64

Previous orders:
count    20000.000000
mean         0.323200
std          0.613645
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          5.000000
Name: previous_orders, dtype: float64

Previous conversion rate:
count    20000.000000
mean         0.118757
std          0.242721
min          0.000000
25%          0.000000
50%          0.000000
75%          0.142900
max          1.000000
Name: previous_conversion_rate, dtype: float64

Buyer-seller repeat inquiry rate:
0.00165


In [32]:
# Reset history again

buyer_history = {
    buyer_id: {
        "previous_inquiries": 0,
        "previous_orders": 0,
        "converted_leads": 0
    }
    for buyer_id in buyers["buyer_id"]
}

seller_history = {
    seller_id: {
        "total_leads": 0,
        "converted_leads": 0
    }
    for seller_id in sellers["seller_id"]
}

buyer_seller_history = {}

event_records = []

n_events = 20000
start_time = pd.Timestamp("2025-01-01")

for i in range(n_events):

    timestamp = start_time + pd.Timedelta(minutes=i * 30)

    # Incoming API event
    lead = generate_lead(
        lead_id=f"L{i+1:06d}",
        timestamp=timestamp,
        rng=rng
    )

    # Historical features available BEFORE prediction
    enriched_lead = enrich_lead_with_history(lead)

    # Hidden variables used ONLY to simulate the eventual outcome
    buyer_propensity = buyers.loc[
        buyers["buyer_id"] == lead["buyer_id"],
        "buyer_propensity"
    ].iloc[0]

    seller_quality = sellers.loc[
        sellers["seller_id"] == lead["seller_id"],
        "seller_quality"
    ].iloc[0]

    urgency_effect = {
        "Low": -0.20,
        "Medium": 0.00,
        "High": 0.18,
        "Critical": 0.30
    }[lead["urgency"]]

    company_effect = {
        "Micro": -0.12,
        "Small": -0.04,
        "Medium": 0.00,
        "Large": 0.08,
        "Enterprise": 0.14
    }[lead["buyer_company_size"]]

    logit = (
        -3.6
        + 1.0 * buyer_propensity
        + 0.55 * seller_quality
        + 0.15 * np.log1p(enriched_lead["previous_orders"])
        + 0.10 * np.log1p(enriched_lead["inquiry_value"])
        + urgency_effect
        + company_effect
        + 0.08 * enriched_lead["repeat_buyer"]
    )

    probability = 1 / (1 + np.exp(-logit))

    converted = int(rng.random() < probability)

    # Store outcome AFTER the prediction point
    enriched_lead["converted"] = converted

    event_records.append(enriched_lead)

    # Update marketplace history AFTER outcome
    update_history(
        enriched_lead,
        converted=converted
    )

historical_api_df = pd.DataFrame(event_records)

print("Shape:", historical_api_df.shape)
print("\nConversion rate:", historical_api_df["converted"].mean())

print("\nColumns:")
print(historical_api_df.columns.tolist())

Shape: (20000, 21)

Conversion rate: 0.15945

Columns:
['lead_id', 'timestamp', 'buyer_id', 'seller_id', 'buyer_industry', 'buyer_location', 'buyer_company_size', 'product_category', 'inquiry_value', 'quantity', 'urgency', 'lead_source', 'seller_rating', 'seller_experience', 'previous_orders', 'previous_inquiries', 'repeat_buyer', 'previous_conversion_rate', 'buyer_seller_previous_inquiries', 'buyer_seller_previous_orders', 'converted']


In [33]:
historical_api_df["month"] = (
    historical_api_df["timestamp"].dt.to_period("M").astype(str)
)

monthly_conversion = (
    historical_api_df
    .groupby("month")["converted"]
    .agg(["count", "mean"])
)

print(monthly_conversion)

print("\nOverall conversion rate:",
      historical_api_df["converted"].mean())

         count      mean
month                   
2025-01   1488  0.149194
2025-02   1344  0.150298
2025-03   1488  0.152554
2025-04   1440  0.147222
2025-05   1488  0.148522
2025-06   1440  0.160417
2025-07   1488  0.175403
2025-08   1488  0.188172
2025-09   1440  0.153472
2025-10   1488  0.155242
2025-11   1440  0.183333
2025-12   1488  0.151882
2026-01   1488  0.161962
2026-02    992  0.151210

Overall conversion rate: 0.15945


In [34]:
 # Remove diagnostic column
historical_api_df = historical_api_df.drop(columns=["month"])

# Ensure chronological ordering
historical_api_df = historical_api_df.sort_values(
    "timestamp"
).reset_index(drop=True)

# Temporal split
n = len(historical_api_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = historical_api_df.iloc[:train_end].copy()
validation_df = historical_api_df.iloc[
    train_end:validation_end
].copy()
test_df = historical_api_df.iloc[
    validation_end:
].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nDate ranges:")
print("Train:", train_df["timestamp"].min(), "→", train_df["timestamp"].max())
print("Validation:", validation_df["timestamp"].min(), "→", validation_df["timestamp"].max())
print("Test:", test_df["timestamp"].min(), "→", test_df["timestamp"].max())

print("\nConversion rates:")
print("Train:", train_df["converted"].mean())
print("Validation:", validation_df["converted"].mean())
print("Test:", test_df["converted"].mean())

Train: (14000, 21)
Validation: (3000, 21)
Test: (3000, 21)

Date ranges:
Train: 2025-01-01 00:00:00 → 2025-10-19 15:30:00
Validation: 2025-10-19 16:00:00 → 2025-12-21 03:30:00
Test: 2025-12-21 04:00:00 → 2026-02-21 15:30:00

Conversion rates:
Train: 0.15842857142857142
Validation: 0.168
Test: 0.15566666666666668


In [35]:
feature_cols = [
    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "product_category",
    "inquiry_value",
    "quantity",
    "urgency",
    "lead_source",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

target_col = "converted"

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_validation = validation_df[feature_cols]
y_validation = validation_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Number of features:", len(feature_cols))
print("\nFeatures:")
for i, feature in enumerate(feature_cols, 1):
    print(f"{i}. {feature}")

print("\nTrain:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

Number of features: 15

Features:
1. buyer_industry
2. buyer_location
3. buyer_company_size
4. previous_orders
5. previous_inquiries
6. repeat_buyer
7. product_category
8. inquiry_value
9. quantity
10. urgency
11. lead_source
12. seller_rating
13. seller_experience
14. previous_conversion_rate
15. buyer_seller_previous_inquiries

Train: (14000, 15)
Validation: (3000, 15)
Test: (3000, 15)


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

categorical_features = [
    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "product_category",
    "urgency",
    "lead_source"
]

numeric_features = [
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "inquiry_value",
    "quantity",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

validation_probability = logistic_model.predict_proba(
    X_validation
)[:, 1]

validation_prediction = (
    validation_probability >= 0.5
).astype(int)

print("Validation Results")
print("------------------")
print("ROC-AUC:",
      roc_auc_score(y_validation, validation_probability))
print("Accuracy:",
      accuracy_score(y_validation, validation_prediction))
print("Precision:",
      precision_score(y_validation, validation_prediction))
print("Recall:",
      recall_score(y_validation, validation_prediction))
print("F1:",
      f1_score(y_validation, validation_prediction))

Validation Results
------------------
ROC-AUC: 0.5610254184472935
Accuracy: 0.504
Precision: 0.19134253450439148
Recall: 0.6051587301587301
F1: 0.29075309818875117


In [37]:
# Check conversion rate by important historical features

print("Conversion by repeat buyer:")
print(
    train_df.groupby("repeat_buyer")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by previous orders:")
print(
    train_df.groupby("previous_orders")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by previous inquiries:")
print(
    train_df.groupby("previous_inquiries")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by urgency:")
print(
    train_df.groupby("urgency")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by company size:")
print(
    train_df.groupby("buyer_company_size")["converted"]
    .agg(["count", "mean"])
)

Conversion by repeat buyer:
              count      mean
repeat_buyer                 
0              4689  0.141181
1              9311  0.167114

Conversion by previous orders:
                 count      mean
previous_orders                 
0                11376  0.154536
1                 2207  0.174445
2                  372  0.188172
3                   41  0.121951
4                    4  0.000000

Conversion by previous inquiries:
                    count      mean
previous_inquiries                 
0                    4689  0.141181
1                    3814  0.163608
2                    2630  0.172624
3                    1552  0.163015
4                     772  0.190415
5                     357  0.156863
6                     119  0.142857
7                      46  0.086957
8                      12  0.000000
9                       7  0.142857
10                      1  0.000000
11                      1  0.000000

Conversion by urgency:
          count      mean


In [38]:
print("Conversion by repeat buyer:")
print(
    train_df.groupby("repeat_buyer")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by previous orders:")
print(
    train_df.groupby("previous_orders")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by previous conversion rate:")
print(
    train_df.groupby("previous_conversion_rate")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by urgency:")
print(
    train_df.groupby("urgency")["converted"]
    .agg(["count", "mean"])
)

print("\nConversion by company size:")
print(
    train_df.groupby("buyer_company_size")["converted"]
    .agg(["count", "mean"])
)

Conversion by repeat buyer:
              count      mean
repeat_buyer                 
0              4689  0.141181
1              9311  0.167114

Conversion by previous orders:
                 count      mean
previous_orders                 
0                11376  0.154536
1                 2207  0.174445
2                  372  0.188172
3                   41  0.121951
4                    4  0.000000

Conversion by previous conversion rate:
                          count      mean
previous_conversion_rate                 
0.0000                    11376  0.154536
0.1111                        4  0.250000
0.1250                        5  0.000000
0.1429                       17  0.058824
0.1667                       45  0.177778
0.2000                      148  0.175676
0.2500                      288  0.159722
0.2857                       15  0.133333
0.3333                      531  0.165725
0.3750                        1  0.000000
0.4000                       53  0.132075
0.

In [39]:
# Reset histories

buyer_history = {
    buyer_id: {
        "previous_inquiries": 0,
        "previous_orders": 0,
        "converted_leads": 0
    }
    for buyer_id in buyers["buyer_id"]
}

seller_history = {
    seller_id: {
        "total_leads": 0,
        "converted_leads": 0
    }
    for seller_id in sellers["seller_id"]
}

buyer_seller_history = {}

event_records = []

n_events = 20000
start_time = pd.Timestamp("2025-01-01")

for i in range(n_events):

    timestamp = start_time + pd.Timedelta(minutes=i * 30)

    lead = generate_lead(
        lead_id=f"L{i+1:06d}",
        timestamp=timestamp,
        rng=rng
    )

    enriched_lead = enrich_lead_with_history(lead)

    # Observable behavioral signals
    order_signal = np.log1p(
        enriched_lead["previous_orders"]
    )

    history_signal = enriched_lead[
        "previous_conversion_rate"
    ]

    inquiry_signal = np.log1p(
        enriched_lead["inquiry_value"]
    )

    urgency_effect = {
        "Low": -0.45,
        "Medium": 0.00,
        "High": 0.35,
        "Critical": 0.65
    }[lead["urgency"]]

    company_effect = {
        "Micro": -0.20,
        "Small": -0.08,
        "Medium": 0.00,
        "Large": 0.12,
        "Enterprise": 0.22
    }[lead["buyer_company_size"]]

    # Outcome generation uses observable signals strongly
    logit = (
        -3.8
        + 0.55 * order_signal
        + 1.10 * history_signal
        + 0.10 * inquiry_signal
        + 0.30 * enriched_lead["repeat_buyer"]
        + 0.25 * enriched_lead["seller_rating"]
        + 0.035 * enriched_lead["seller_experience"]
        + urgency_effect
        + company_effect
    )

    probability = 1 / (1 + np.exp(-logit))

    converted = int(rng.random() < probability)

    enriched_lead["converted"] = converted

    event_records.append(enriched_lead)

    update_history(
        enriched_lead,
        converted=converted
    )

historical_api_df = pd.DataFrame(event_records)

print("Shape:", historical_api_df.shape)
print("Conversion rate:", historical_api_df["converted"].mean())

Shape: (20000, 21)
Conversion rate: 0.32625


In [40]:
historical_api_df = (
    historical_api_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

n = len(historical_api_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = historical_api_df.iloc[:train_end].copy()
validation_df = historical_api_df.iloc[
    train_end:validation_end
].copy()
test_df = historical_api_df.iloc[
    validation_end:
].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nConversion rates:")
print("Train:", round(train_df["converted"].mean(), 4))
print("Validation:", round(validation_df["converted"].mean(), 4))
print("Test:", round(test_df["converted"].mean(), 4))

Train: (14000, 21)
Validation: (3000, 21)
Test: (3000, 21)

Conversion rates:
Train: 0.2989
Validation: 0.3707
Test: 0.4097


In [41]:
# Reset histories

buyer_history = {
    buyer_id: {
        "previous_inquiries": 0,
        "previous_orders": 0,
        "converted_leads": 0
    }
    for buyer_id in buyers["buyer_id"]
}

seller_history = {
    seller_id: {
        "total_leads": 0,
        "converted_leads": 0
    }
    for seller_id in sellers["seller_id"]
}

buyer_seller_history = {}

event_records = []

n_events = 20000
start_time = pd.Timestamp("2025-01-01")

for i in range(n_events):

    timestamp = start_time + pd.Timedelta(minutes=i * 30)

    lead = generate_lead(
        lead_id=f"L{i+1:06d}",
        timestamp=timestamp,
        rng=rng
    )

    enriched_lead = enrich_lead_with_history(lead)

    # Stable observable signals
    order_signal = np.log1p(
        enriched_lead["previous_orders"]
    )

    history_signal = enriched_lead[
        "previous_conversion_rate"
    ]

    inquiry_signal = np.log1p(
        enriched_lead["inquiry_value"]
    )

    urgency_effect = {
        "Low": -0.35,
        "Medium": 0.00,
        "High": 0.30,
        "Critical": 0.55
    }[lead["urgency"]]

    company_effect = {
        "Micro": -0.15,
        "Small": -0.05,
        "Medium": 0.00,
        "Large": 0.08,
        "Enterprise": 0.15
    }[lead["buyer_company_size"]]

    # Stronger observable signal, but less dependence on
    # accumulated history
    logit = (
        -2.8
        + 0.30 * order_signal
        + 0.65 * history_signal
        + 0.12 * inquiry_signal
        + 0.18 * enriched_lead["repeat_buyer"]
        + 0.22 * enriched_lead["seller_rating"]
        + 0.025 * enriched_lead["seller_experience"]
        + urgency_effect
        + company_effect
    )

    probability = 1 / (1 + np.exp(-logit))

    converted = int(rng.random() < probability)

    enriched_lead["converted"] = converted

    event_records.append(enriched_lead)

    update_history(
        enriched_lead,
        converted=converted
    )

historical_api_df = pd.DataFrame(event_records)

print("Shape:", historical_api_df.shape)
print("Overall conversion rate:",
      round(historical_api_df["converted"].mean(), 4))

Shape: (20000, 21)
Overall conversion rate: 0.5235


In [42]:
temp_check = historical_api_df.copy()

temp_check["month"] = (
    temp_check["timestamp"]
    .dt.to_period("M")
    .astype(str)
)

monthly_conversion = (
    temp_check
    .groupby("month")["converted"]
    .agg(["count", "mean"])
)

print(monthly_conversion)

print(
    "\nMinimum monthly conversion:",
    monthly_conversion["mean"].min()
)

print(
    "Maximum monthly conversion:",
    monthly_conversion["mean"].max()
)

         count      mean
month                   
2025-01   1488  0.412634
2025-02   1344  0.435268
2025-03   1488  0.463038
2025-04   1440  0.478472
2025-05   1488  0.515457
2025-06   1440  0.496528
2025-07   1488  0.540995
2025-08   1488  0.531586
2025-09   1440  0.558333
2025-10   1488  0.576613
2025-11   1440  0.577778
2025-12   1488  0.594086
2026-01   1488  0.579301
2026-02    992  0.579637

Minimum monthly conversion: 0.41263440860215056
Maximum monthly conversion: 0.5940860215053764


In [43]:
# Reset histories

buyer_history = {
    buyer_id: {
        "previous_inquiries": 0,
        "previous_orders": 0,
        "converted_leads": 0
    }
    for buyer_id in buyers["buyer_id"]
}

seller_history = {
    seller_id: {
        "total_leads": 0,
        "converted_leads": 0
    }
    for seller_id in sellers["seller_id"]
}

buyer_seller_history = {}

event_records = []

n_events = 20000
start_time = pd.Timestamp("2025-01-01")

for i in range(n_events):

    timestamp = start_time + pd.Timedelta(minutes=i * 30)

    lead = generate_lead(
        lead_id=f"L{i+1:06d}",
        timestamp=timestamp,
        rng=rng
    )

    enriched_lead = enrich_lead_with_history(lead)

    # Cap historical signals so accumulated history
    # cannot create an artificial time trend.
    order_signal = min(
        np.log1p(enriched_lead["previous_orders"]),
        np.log1p(2)
    )

    history_signal = min(
        enriched_lead["previous_conversion_rate"],
        0.50
    )

    inquiry_signal = np.log1p(
        enriched_lead["inquiry_value"]
    )

    urgency_effect = {
        "Low": -0.45,
        "Medium": 0.00,
        "High": 0.35,
        "Critical": 0.65
    }[lead["urgency"]]

    company_effect = {
        "Micro": -0.12,
        "Small": -0.04,
        "Medium": 0.00,
        "Large": 0.08,
        "Enterprise": 0.15
    }[lead["buyer_company_size"]]

    logit = (
        -4.0
        + 0.35 * order_signal
        + 0.70 * history_signal
        + 0.08 * inquiry_signal
        + 0.18 * enriched_lead["repeat_buyer"]
        + 0.20 * enriched_lead["seller_rating"]
        + 0.025 * enriched_lead["seller_experience"]
        + urgency_effect
        + company_effect
    )

    probability = 1 / (1 + np.exp(-logit))

    converted = int(rng.random() < probability)

    enriched_lead["converted"] = converted

    event_records.append(enriched_lead)

    update_history(
        enriched_lead,
        converted=converted
    )

historical_api_df = pd.DataFrame(event_records)

print("Shape:", historical_api_df.shape)
print(
    "Overall conversion rate:",
    round(historical_api_df["converted"].mean(), 4)
)

Shape: (20000, 21)
Overall conversion rate: 0.1436


In [44]:
temp_check = historical_api_df.copy()

temp_check["month"] = (
    temp_check["timestamp"]
    .dt.to_period("M")
    .astype(str)
)

monthly_conversion = (
    temp_check
    .groupby("month")["converted"]
    .agg(["count", "mean"])
)

print(monthly_conversion)

print(
    "\nMinimum monthly conversion:",
    round(monthly_conversion["mean"].min(), 4)
)

print(
    "Maximum monthly conversion:",
    round(monthly_conversion["mean"].max(), 4)
)

print(
    "Range:",
    round(
        monthly_conversion["mean"].max()
        - monthly_conversion["mean"].min(),
        4
    )
)

         count      mean
month                   
2025-01   1488  0.130376
2025-02   1344  0.139881
2025-03   1488  0.126344
2025-04   1440  0.132639
2025-05   1488  0.125672
2025-06   1440  0.136111
2025-07   1488  0.156586
2025-08   1488  0.147849
2025-09   1440  0.138889
2025-10   1488  0.154570
2025-11   1440  0.156944
2025-12   1488  0.160618
2026-01   1488  0.151882
2026-02    992  0.156250

Minimum monthly conversion: 0.1257
Maximum monthly conversion: 0.1606
Range: 0.0349


In [45]:
historical_api_df = (
    historical_api_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

n = len(historical_api_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = historical_api_df.iloc[:train_end].copy()
validation_df = historical_api_df.iloc[
    train_end:validation_end
].copy()
test_df = historical_api_df.iloc[
    validation_end:
].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nConversion rates:")
print("Train:", round(train_df["converted"].mean(), 4))
print("Validation:", round(validation_df["converted"].mean(), 4))
print("Test:", round(test_df["converted"].mean(), 4))


Train: (14000, 21)
Validation: (3000, 21)
Test: (3000, 21)

Conversion rates:
Train: 0.1386
Validation: 0.1553
Test: 0.1553


In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

feature_cols = [
    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "product_category",
    "inquiry_value",
    "quantity",
    "urgency",
    "lead_source",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

# Remove duplicate if any
feature_cols = list(dict.fromkeys(feature_cols))

target_col = "converted"

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_validation = validation_df[feature_cols]
y_validation = validation_df[target_col]

categorical_features = [
    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "product_category",
    "urgency",
    "lead_source"
]

numeric_features = [
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "inquiry_value",
    "quantity",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

validation_probability = logistic_model.predict_proba(
    X_validation
)[:, 1]

validation_prediction = (
    validation_probability >= 0.5
).astype(int)

print("Validation Results")
print("------------------")
print("ROC-AUC:",
      round(roc_auc_score(
          y_validation,
          validation_probability
      ), 4))

print("Accuracy:",
      round(accuracy_score(
          y_validation,
          validation_prediction
      ), 4))

print("Precision:",
      round(precision_score(
          y_validation,
          validation_prediction
      ), 4))

print("Recall:",
      round(recall_score(
          y_validation,
          validation_prediction
      ), 4))

print("F1:",
      round(f1_score(
          y_validation,
          validation_prediction
      ), 4))

Validation Results
------------------
ROC-AUC: 0.6167
Accuracy: 0.5097
Precision: 0.1915
Recall: 0.6695
F1: 0.2979


In [47]:
def add_engineered_features(df):
    df = df.copy()

    # 1. Potential order value
    df["opportunity_value"] = (
        df["inquiry_value"] * df["quantity"]
    )

    # 2. Log transforms for highly skewed monetary/count variables
    df["log_inquiry_value"] = np.log1p(
        df["inquiry_value"]
    )

    df["log_previous_orders"] = np.log1p(
        df["previous_orders"]
    )

    df["log_previous_inquiries"] = np.log1p(
        df["previous_inquiries"]
    )

    # 3. Buyer maturity
    df["buyer_order_rate"] = (
        df["previous_orders"] /
        df["previous_inquiries"].replace(0, np.nan)
    ).fillna(0)

    # 4. Seller quality interaction
    df["seller_strength"] = (
        df["seller_rating"] *
        np.log1p(df["seller_experience"])
    )

    # 5. Buyer-seller relationship signal
    df["buyer_seller_repeat"] = (
        df["buyer_seller_previous_inquiries"] > 0
    ).astype(int)

    # 6. High-value urgent opportunity
    df["urgent_opportunity"] = (
        (df["urgency"].isin(["High", "Critical"])) &
        (df["inquiry_value"] >= df["inquiry_value"].median())
    ).astype(int)

    # 7. Buyer maturity × seller strength
    df["buyer_seller_strength"] = (
        df["buyer_order_rate"] *
        df["seller_rating"]
    )

    return df


train_fe = add_engineered_features(train_df)
validation_fe = add_engineered_features(validation_df)

print("Original features:", len(feature_cols))
print("Engineered dataset:", train_fe.shape)

print("\nNew features:")
new_features = [
    c for c in train_fe.columns
    if c not in train_df.columns
]

print(new_features)

Original features: 15
Engineered dataset: (14000, 30)

New features:
['opportunity_value', 'log_inquiry_value', 'log_previous_orders', 'log_previous_inquiries', 'buyer_order_rate', 'seller_strength', 'buyer_seller_repeat', 'urgent_opportunity', 'buyer_seller_strength']


In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Features to exclude
exclude_cols = [
    "converted",
    "timestamp",
    "lead_id",
    "buyer_id",
    "seller_id",
    "buyer_propensity",
    "seller_quality"
]

model_features = [
    c for c in train_fe.columns
    if c not in exclude_cols
]

X_train = train_fe[model_features]
y_train = train_fe["converted"]

X_val = validation_fe[model_features]
y_val = validation_fe["converted"]

# Identify categorical and numeric columns
categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_cols = [
    c for c in model_features
    if c not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numeric_cols
        )
    ]
)

logistic_fe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

logistic_fe.fit(X_train, y_train)

train_prob = logistic_fe.predict_proba(X_train)[:, 1]
val_prob = logistic_fe.predict_proba(X_val)[:, 1]

print("Train ROC-AUC:", round(
    roc_auc_score(y_train, train_prob), 4
))

print("Validation ROC-AUC:", round(
    roc_auc_score(y_val, val_prob), 4
))

Train ROC-AUC: 0.6287
Validation ROC-AUC: 0.6108


/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/2051363224.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [49]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Use engineered features
X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)

# Convert sparse matrix to dense
X_train_enc = X_train_enc.toarray()
X_val_enc = X_val_enc.toarray()

hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=42
)

hgb.fit(X_train_enc, y_train)

train_prob_hgb = hgb.predict_proba(X_train_enc)[:, 1]
val_prob_hgb = hgb.predict_proba(X_val_enc)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, train_prob_hgb), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_prob_hgb), 4))

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

preprocessor_hgb = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols
        ),
        (
            "num",
            "passthrough",
            numeric_cols
        )
    ]
)

X_train_hgb = preprocessor_hgb.fit_transform(X_train)
X_val_hgb = preprocessor_hgb.transform(X_val)

hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=42
)

hgb.fit(X_train_hgb, y_train)

train_prob_hgb = hgb.predict_proba(X_train_hgb)[:, 1]
val_prob_hgb = hgb.predict_proba(X_val_hgb)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, train_prob_hgb), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_prob_hgb), 4))

Train ROC-AUC: 0.7015
Validation ROC-AUC: 0.6185


In [51]:
# Cell 28 — Feature signal diagnostic

signal_features = [
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "inquiry_value",
    "quantity",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

for col in signal_features:
    temp = train_df.groupby("converted")[col].mean()

    print(f"\n{col}")
    print("  Non-converted:", round(temp.get(0, np.nan), 4))
    print("  Converted:    ", round(temp.get(1, np.nan), 4))


previous_orders
  Non-converted: 0.1731
  Converted:     0.2787

previous_inquiries
  Non-converted: 1.3817
  Converted:     1.5574

repeat_buyer
  Non-converted: 0.6555
  Converted:     0.7197

inquiry_value
  Non-converted: 35795.3496
  Converted:     37512.0101

quantity
  Non-converted: 15.177
  Converted:     15.3843

seller_rating
  Non-converted: 4.0969
  Converted:     4.1312

seller_experience
  Non-converted: 6.4853
  Converted:     6.6025

previous_conversion_rate
  Non-converted: 0.0778
  Converted:     0.129

buyer_seller_previous_inquiries
  Non-converted: 0.0019
  Converted:     0.001


In [52]:
# Cell 29 — Conversion rate by strongest observable signals

print("Previous conversion rate:")
print(
    train_df.groupby(
        pd.cut(
            train_df["previous_conversion_rate"],
            bins=[-0.001, 0.05, 0.15, 0.30, 0.50, 1.0]
        )
    )["converted"].mean().round(3)
)

print("\nPrevious orders:")
print(
    train_df.groupby("previous_orders")["converted"]
    .mean()
    .round(3)
)

print("\nRepeat buyer:")
print(
    train_df.groupby("repeat_buyer")["converted"]
    .mean()
    .round(3)
)

print("\nUrgency:")
print(
    train_df.groupby("urgency")["converted"]
    .mean()
    .round(3)
)

print("\nCompany size:")
print(
    train_df.groupby("buyer_company_size")["converted"]
    .mean()
    .round(3)
)

Previous conversion rate:
previous_conversion_rate
(-0.001, 0.05]    0.126
(0.05, 0.15]      0.316
(0.15, 0.3]       0.172
(0.3, 0.5]        0.223
(0.5, 1.0]        0.203
Name: converted, dtype: float64

Previous orders:
previous_orders
0    0.126
1    0.210
2    0.218
3    0.085
4    0.250
Name: converted, dtype: float64

Repeat buyer:
repeat_buyer
0    0.116
1    0.150
Name: converted, dtype: float64

Urgency:
urgency
Critical    0.216
High        0.169
Low         0.083
Medium      0.126
Name: converted, dtype: float64

Company size:
buyer_company_size
Enterprise    0.162
Large         0.138
Medium        0.136
Micro         0.132
Small         0.131
Name: converted, dtype: float64


In [53]:
# Cell 30 — stronger observable-signal outcome

rng = np.random.default_rng(42)

order_signal = np.minimum(
    np.log1p(df_new["previous_orders"]),
    np.log1p(3)
)

history_signal = np.minimum(
    df_new["previous_conversion_rate"],
    0.50
)

inquiry_signal = np.log1p(df_new["inquiry_value"])

urgency_effect = df_new["urgency"].map({
    "Low": -0.80,
    "Medium": -0.20,
    "High": 0.45,
    "Critical": 0.85
})

company_effect = df_new["buyer_company_size"].map({
    "Micro": -0.20,
    "Small": -0.10,
    "Medium": 0.00,
    "Large": 0.12,
    "Enterprise": 0.25
})

source_effect = df_new["lead_source"].map({
    "Paid Search": -0.15,
    "Marketplace Search": -0.05,
    "Email Campaign": 0.00,
    "Direct Inquiry": 0.08,
    "Organic Search": 0.12,
    "Referral": 0.20
})

logit = (
    -4.2
    + 0.75 * order_signal
    + 1.80 * history_signal
    + 0.10 * inquiry_signal
    + 0.40 * df_new["repeat_buyer"]
    + 0.25 * df_new["seller_rating"]
    + 0.03 * df_new["seller_experience"]
    + urgency_effect
    + company_effect
    + source_effect
)

probability = 1 / (1 + np.exp(-logit))

df_new["converted"] = (
    rng.random(len(df_new)) < probability
).astype(int)

print("Overall conversion:",
      round(df_new["converted"].mean(), 4))

print("\nConversion by urgency:")
print(
    df_new.groupby("urgency")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by repeat buyer:")
print(
    df_new.groupby("repeat_buyer")["converted"]
    .mean()
    .round(3)
)

KeyError: 'previous_conversion_rate'

In [54]:
print(df_new.shape)
print(df_new.columns.tolist())

(20000, 22)
['timestamp', 'buyer_id', 'seller_id', 'buyer_industry', 'buyer_location', 'buyer_company_size', 'product_category', 'urgency', 'lead_source', 'inquiry_hour', 'seller_rating', 'seller_experience', 'previous_inquiries', 'repeat_buyer', 'buyer_propensity', 'previous_orders', 'quantity', 'inquiry_value', 'buyer_seller_previous_inquiries', 'buyer_seller_previous_orders', 'seller_quality', 'converted']


In [55]:
# Cell 30B

rng = np.random.default_rng(42)

order_signal = np.minimum(
    np.log1p(df_new["previous_orders"]),
    np.log1p(3)
)

history_signal = np.minimum(
    df_new["previous_conversion_rate"],
    0.50
)

inquiry_signal = np.log1p(df_new["inquiry_value"])

urgency_effect = df_new["urgency"].map({
    "Low": -0.80,
    "Medium": -0.20,
    "High": 0.45,
    "Critical": 0.85
})

company_effect = df_new["buyer_company_size"].map({
    "Micro": -0.20,
    "Small": -0.10,
    "Medium": 0.00,
    "Large": 0.12,
    "Enterprise": 0.25
})

source_effect = df_new["lead_source"].map({
    "Paid Search": -0.15,
    "Marketplace Search": -0.05,
    "Email Campaign": 0.00,
    "Direct Inquiry": 0.08,
    "Organic Search": 0.12,
    "Referral": 0.20
})

logit = (
    -4.2
    + 0.75 * order_signal
    + 1.80 * history_signal
    + 0.10 * inquiry_signal
    + 0.40 * df_new["repeat_buyer"]
    + 0.25 * df_new["seller_rating"]
    + 0.03 * df_new["seller_experience"]
    + urgency_effect
    + company_effect
    + source_effect
)

probability = 1 / (1 + np.exp(-logit))

df_new["converted"] = (
    rng.random(len(df_new)) < probability
).astype(int)

print("Overall conversion:",
      round(df_new["converted"].mean(), 4))

print("\nConversion by urgency:")
print(
    df_new.groupby("urgency")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by repeat buyer:")
print(
    df_new.groupby("repeat_buyer")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by company size:")
print(
    df_new.groupby("buyer_company_size")["converted"]
    .mean()
    .round(3)
)

KeyError: 'previous_conversion_rate'

In [56]:
# Cell 30C

df_new["previous_conversion_rate"] = (
    df_new["previous_orders"] /
    df_new["previous_inquiries"].replace(0, np.nan)
).fillna(0)

print(
    df_new[
        ["previous_orders",
         "previous_inquiries",
         "previous_conversion_rate"]
    ].head()
)

print("\nMissing values:",
      df_new["previous_conversion_rate"].isna().sum())

   previous_orders  previous_inquiries  previous_conversion_rate
0                0                   0                       0.0
1                0                   0                       0.0
2                0                   0                       0.0
3                0                   0                       0.0
4                0                   0                       0.0

Missing values: 0


In [57]:
# Cell 30E

rng = np.random.default_rng(42)

# Historical buyer signal
order_signal = np.minimum(
    np.log1p(df_new["previous_orders"]),
    np.log1p(3)
)

# Current inquiry signal
inquiry_signal = np.log1p(df_new["inquiry_value"])

# Observable marketplace signals
urgency_effect = df_new["urgency"].map({
    "Low": -0.75,
    "Medium": -0.15,
    "High": 0.45,
    "Critical": 0.80
})

company_effect = df_new["buyer_company_size"].map({
    "Micro": -0.15,
    "Small": -0.08,
    "Medium": 0.00,
    "Large": 0.10,
    "Enterprise": 0.20
})

source_effect = df_new["lead_source"].map({
    "Paid Search": -0.10,
    "Marketplace Search": -0.03,
    "Email Campaign": 0.00,
    "Direct Inquiry": 0.05,
    "Organic Search": 0.10,
    "Referral": 0.15
})

logit = (
    -4.0
    + 0.65 * order_signal
    + 0.08 * inquiry_signal
    + 0.35 * df_new["repeat_buyer"]
    + 0.22 * df_new["seller_rating"]
    + 0.025 * df_new["seller_experience"]
    + urgency_effect
    + company_effect
    + source_effect
)

probability = 1 / (1 + np.exp(-logit))

df_new["converted"] = (
    rng.random(len(df_new)) < probability
).astype(int)

print("Overall conversion:",
      round(df_new["converted"].mean(), 4))

print("\nConversion by urgency:")
print(
    df_new.groupby("urgency")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by repeat buyer:")
print(
    df_new.groupby("repeat_buyer")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by company size:")
print(
    df_new.groupby("buyer_company_size")["converted"]
    .mean()
    .round(3)
)

Overall conversion: 0.1876

Conversion by urgency:
urgency
Critical    0.309
High        0.241
Low         0.091
Medium      0.165
Name: converted, dtype: float64

Conversion by repeat buyer:
repeat_buyer
0    0.118
1    0.210
Name: converted, dtype: float64

Conversion by company size:
buyer_company_size
Enterprise    0.218
Large         0.207
Medium        0.174
Micro         0.177
Small         0.175
Name: converted, dtype: float64


In [58]:
# Cell 30E

rng = np.random.default_rng(42)

# Historical buyer signal
order_signal = np.minimum(
    np.log1p(df_new["previous_orders"]),
    np.log1p(3)
)

# Current inquiry signal
inquiry_signal = np.log1p(df_new["inquiry_value"])

# Observable marketplace signals
urgency_effect = df_new["urgency"].map({
    "Low": -0.75,
    "Medium": -0.15,
    "High": 0.45,
    "Critical": 0.80
})

company_effect = df_new["buyer_company_size"].map({
    "Micro": -0.15,
    "Small": -0.08,
    "Medium": 0.00,
    "Large": 0.10,
    "Enterprise": 0.20
})

source_effect = df_new["lead_source"].map({
    "Paid Search": -0.10,
    "Marketplace Search": -0.03,
    "Email Campaign": 0.00,
    "Direct Inquiry": 0.05,
    "Organic Search": 0.10,
    "Referral": 0.15
})

logit = (
    -4.0
    + 0.65 * order_signal
    + 0.08 * inquiry_signal
    + 0.35 * df_new["repeat_buyer"]
    + 0.22 * df_new["seller_rating"]
    + 0.025 * df_new["seller_experience"]
    + urgency_effect
    + company_effect
    + source_effect
)

probability = 1 / (1 + np.exp(-logit))

df_new["converted"] = (
    rng.random(len(df_new)) < probability
).astype(int)

print("Overall conversion:",
      round(df_new["converted"].mean(), 4))

print("\nConversion by urgency:")
print(
    df_new.groupby("urgency")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by repeat buyer:")
print(
    df_new.groupby("repeat_buyer")["converted"]
    .mean()
    .round(3)
)

print("\nConversion by company size:")
print(
    df_new.groupby("buyer_company_size")["converted"]
    .mean()
    .round(3)
)

Overall conversion: 0.1876

Conversion by urgency:
urgency
Critical    0.309
High        0.241
Low         0.091
Medium      0.165
Name: converted, dtype: float64

Conversion by repeat buyer:
repeat_buyer
0    0.118
1    0.210
Name: converted, dtype: float64

Conversion by company size:
buyer_company_size
Enterprise    0.218
Large         0.207
Medium        0.174
Micro         0.177
Small         0.175
Name: converted, dtype: float64


In [59]:
# Cell 31 — temporal train / validation / test split

df_new = df_new.sort_values("timestamp").reset_index(drop=True)

train_df = df_new.iloc[:14000].copy()
validation_df = df_new.iloc[14000:17000].copy()
test_df = df_new.iloc[17000:20000].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nConversion rates:")
print("Train:", round(train_df["converted"].mean(), 4))
print("Validation:", round(validation_df["converted"].mean(), 4))
print("Test:", round(test_df["converted"].mean(), 4))

print("\nDate ranges:")
print("Train:", train_df["timestamp"].min(), "→", train_df["timestamp"].max())
print("Validation:", validation_df["timestamp"].min(), "→", validation_df["timestamp"].max())
print("Test:", test_df["timestamp"].min(), "→", test_df["timestamp"].max())

Train: (14000, 23)
Validation: (3000, 23)
Test: (3000, 23)

Conversion rates:
Train: 0.1739
Validation: 0.2163
Test: 0.2233

Date ranges:
Train: 2025-01-01 00:00:00 → 2026-01-17 11:48:13.644682
Validation: 2026-01-17 12:27:28.162408 → 2026-04-09 05:54:06.822341
Test: 2026-04-09 06:33:21.340067 → 2026-06-30 00:00:00


In [60]:
# Cell 32 — production feature set

feature_cols = [
    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "product_category",
    "inquiry_value",
    "quantity",
    "urgency",
    "lead_source",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

print("Number of production features:", len(feature_cols))
print("\nFeatures:")
for i, col in enumerate(feature_cols, 1):
    print(i, col)

Number of production features: 15

Features:
1 buyer_industry
2 buyer_location
3 buyer_company_size
4 previous_orders
5 previous_inquiries
6 repeat_buyer
7 product_category
8 inquiry_value
9 quantity
10 urgency
11 lead_source
12 seller_rating
13 seller_experience
14 previous_conversion_rate
15 buyer_seller_previous_inquiries


In [61]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train = train_df[feature_cols]
y_train = train_df["converted"]

X_val = validation_df[feature_cols]
y_val = validation_df["converted"]

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_cols = [
    c for c in feature_cols
    if c not in categorical_cols
]

preprocessor_base = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numeric_cols
        )
    ]
)

baseline_model = Pipeline([
    ("preprocessor", preprocessor_base),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

baseline_model.fit(X_train, y_train)

train_prob = baseline_model.predict_proba(X_train)[:, 1]
val_prob = baseline_model.predict_proba(X_val)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, train_prob), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_prob), 4))

Train ROC-AUC: 0.6811
Validation ROC-AUC: 0.6375


/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/1679503321.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [62]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Dense encoding specifically for HGB
preprocessor_hgb = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols
        ),
        (
            "num",
            "passthrough",
            numeric_cols
        )
    ]
)

X_train_hgb = preprocessor_hgb.fit_transform(X_train)
X_val_hgb = preprocessor_hgb.transform(X_val)

hgb_model = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.04,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=2.0,
    random_state=42
)

hgb_model.fit(X_train_hgb, y_train)

train_prob_hgb = hgb_model.predict_proba(X_train_hgb)[:, 1]
val_prob_hgb = hgb_model.predict_proba(X_val_hgb)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, train_prob_hgb), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_prob_hgb), 4))

Train ROC-AUC: 0.7105
Validation ROC-AUC: 0.6382


In [63]:
# Cell 35 — buyer-seller fit features

def add_fit_features(df):
    df = df.copy()

    # Buyer historical maturity
    df["buyer_order_rate"] = (
        df["previous_orders"] /
        df["previous_inquiries"].replace(0, np.nan)
    ).fillna(0)

    # Seller strength
    df["seller_strength"] = (
        df["seller_rating"] *
        np.log1p(df["seller_experience"])
    )

    # Buyer × seller interaction
    df["buyer_seller_fit"] = (
        df["buyer_order_rate"] *
        df["seller_rating"]
    )

    # Commercial opportunity
    df["log_inquiry_value"] = np.log1p(
        df["inquiry_value"]
    )

    # Urgency × commercial value
    urgency_weight = df["urgency"].map({
        "Low": 0.5,
        "Medium": 1.0,
        "High": 1.5,
        "Critical": 2.0
    })

    df["urgency_value_score"] = (
        urgency_weight * df["log_inquiry_value"]
    )

    # Quantity × urgency
    df["urgent_quantity"] = (
        urgency_weight * np.log1p(df["quantity"])
    )

    return df


train_fit = add_fit_features(train_df)
validation_fit = add_fit_features(validation_df)

print("Original columns:", train_df.shape[1])
print("New columns:", train_fit.shape[1])

print("\nAdded:")
print([
    c for c in train_fit.columns
    if c not in train_df.columns
])

Original columns: 23
New columns: 29

Added:
['buyer_order_rate', 'seller_strength', 'buyer_seller_fit', 'log_inquiry_value', 'urgency_value_score', 'urgent_quantity']


In [64]:
fit_feature_cols = [
    c for c in train_fit.columns
    if c not in [
        "converted",
        "timestamp",
        "lead_id",
        "buyer_id",
        "seller_id",
        "buyer_propensity",
        "seller_quality"
    ]
]

X_train_fit = train_fit[fit_feature_cols]
X_val_fit = validation_fit[fit_feature_cols]

cat_fit = X_train_fit.select_dtypes(
    include=["object"]
).columns.tolist()

num_fit = [
    c for c in fit_feature_cols
    if c not in cat_fit
]

preprocessor_fit = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_fit
        ),
        (
            "num",
            StandardScaler(),
            num_fit
        )
    ]
)

fit_model = Pipeline([
    ("preprocessor", preprocessor_fit),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

fit_model.fit(X_train_fit, y_train)

train_prob_fit = fit_model.predict_proba(X_train_fit)[:, 1]
val_prob_fit = fit_model.predict_proba(X_val_fit)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, train_prob_fit), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_prob_fit), 4))


Train ROC-AUC: 0.6812
Validation ROC-AUC: 0.6383


/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/1570125093.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_fit = X_train_fit.select_dtypes(


In [65]:
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

param_grid = {
    "learning_rate": [0.02, 0.04, 0.07],
    "max_leaf_nodes": [7, 15, 31],
    "min_samples_leaf": [20, 40, 80],
    "l2_regularization": [0.5, 2.0, 5.0]
}

results = []

for params in ParameterGrid(param_grid):
    model = HistGradientBoostingClassifier(
        max_iter=400,
        random_state=42,
        **params
    )

    model.fit(X_train_hgb, y_train)

    val_prob = model.predict_proba(X_val_hgb)[:, 1]

    auc = roc_auc_score(y_val, val_prob)

    results.append({
        **params,
        "validation_auc": auc
    })

results_df = pd.DataFrame(results).sort_values(
    "validation_auc",
    ascending=False
)

print(results_df.head(10).to_string(index=False))

 l2_regularization  learning_rate  max_leaf_nodes  min_samples_leaf  validation_auc
               0.5           0.07               7                80        0.644977
               0.5           0.04               7                80        0.643953
               5.0           0.02               7                80        0.643728
               5.0           0.04               7                40        0.643266
               2.0           0.02               7                80        0.643132
               2.0           0.07               7                40        0.643018
               0.5           0.02               7                80        0.642972
               2.0           0.04               7                40        0.642934
               5.0           0.07               7                80        0.642899
               2.0           0.04               7                80        0.642821


In [66]:
best_params = {
    "learning_rate": 0.07,
    "max_leaf_nodes": 7,
    "min_samples_leaf": 80,
    "l2_regularization": 0.5
}

best_hgb = HistGradientBoostingClassifier(
    max_iter=400,
    random_state=42,
    **best_params
)

best_hgb.fit(X_train_hgb, y_train)

test_X = test_df[feature_cols]
y_test = test_df["converted"]

X_test_hgb = preprocessor_hgb.transform(test_X)

test_prob = best_hgb.predict_proba(X_test_hgb)[:, 1]

print("Validation ROC-AUC:",
      round(
          roc_auc_score(
              y_val,
              best_hgb.predict_proba(X_val_hgb)[:, 1]
          ), 4
      ))

print("Future Test ROC-AUC:",
      round(
          roc_auc_score(y_test, test_prob), 4
      ))

Validation ROC-AUC: 0.645
Future Test ROC-AUC: 0.6525


In [67]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_hgb,
    X_val_hgb,
    y_val,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42
)

feature_names_hgb = preprocessor_hgb.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names_hgb,
    "importance": perm.importances_mean
}).sort_values(
    "importance",
    ascending=False
)

print(importance_df.head(20).to_string(index=False))

                               feature  importance
                      cat__urgency_Low    0.057861
                  num__previous_orders    0.044206
                   cat__urgency_Medium    0.012831
                 cat__urgency_Critical    0.007942
    cat__buyer_company_size_Enterprise    0.002281
                num__seller_experience    0.001955
          cat__lead_source_Paid Search    0.001742
         cat__buyer_company_size_Large    0.001277
               num__previous_inquiries    0.000627
        cat__buyer_industry_Healthcare    0.000383
            cat__buyer_industry_Retail    0.000320
         cat__buyer_company_size_Micro    0.000292
cat__product_category_Food Ingredients    0.000285
            cat__buyer_location_Mumbai    0.000268
                    num__seller_rating    0.000227
           cat__buyer_location_Chennai    0.000151
       cat__lead_source_Direct Inquiry    0.000134
             cat__buyer_location_Delhi    0.000007
        cat__buyer_industry_Aut

In [68]:
# Cell 40 — historical seller performance
# Calculated using information available BEFORE each lead

df_work = df_new.sort_values("timestamp").copy()

df_work["seller_previous_leads"] = (
    df_work.groupby("seller_id").cumcount()
)

df_work["seller_previous_conversions"] = (
    df_work.groupby("seller_id")["converted"]
    .cumsum()
    - df_work["converted"]
)

df_work["seller_historical_conversion_rate"] = (
    df_work["seller_previous_conversions"] /
    df_work["seller_previous_leads"].replace(0, np.nan)
).fillna(0)

print(
    df_work[
        [
            "seller_id",
            "seller_previous_leads",
            "seller_previous_conversions",
            "seller_historical_conversion_rate"
        ]
    ].head(15)
)

    seller_id  seller_previous_leads  seller_previous_conversions  \
0          35                      0                            0   
1         721                      0                            0   
2         214                      0                            0   
3         712                      0                            0   
4         565                      0                            0   
5         203                      0                            0   
6         470                      0                            0   
7          37                      0                            0   
8         702                      0                            0   
9         304                      0                            0   
10        384                      0                            0   
11        158                      0                            0   
12        365                      0                            0   
13        396                     

In [69]:
# Cell 41 — add seller historical performance

feature_cols_v2 = feature_cols + [
    "seller_historical_conversion_rate"
]

train_v2 = df_work.iloc[:14000].copy()
validation_v2 = df_work.iloc[14000:17000].copy()
test_v2 = df_work.iloc[17000:20000].copy()

X_train_v2 = train_v2[feature_cols_v2]
X_val_v2 = validation_v2[feature_cols_v2]

y_train_v2 = train_v2["converted"]
y_val_v2 = validation_v2["converted"]

categorical_v2 = X_train_v2.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_v2 = [
    c for c in feature_cols_v2
    if c not in categorical_v2
]

print("Features:", len(feature_cols_v2))
print("Seller historical conversion rate:")
print(
    train_v2["seller_historical_conversion_rate"]
    .describe()
    .round(3)
)

Features: 16
Seller historical conversion rate:
count    14000.000
mean         0.142
std          0.171
min          0.000
25%          0.000
50%          0.111
75%          0.222
max          1.000
Name: seller_historical_conversion_rate, dtype: float64


/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/1634324010.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_v2 = X_train_v2.select_dtypes(


In [70]:
# Cell 42 — HGB with seller historical performance

preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_v2
        ),
        (
            "num",
            "passthrough",
            numeric_v2
        )
    ]
)

X_train_v2_enc = preprocessor_v2.fit_transform(X_train_v2)
X_val_v2_enc = preprocessor_v2.transform(X_val_v2)

hgb_v2 = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.07,
    max_leaf_nodes=7,
    min_samples_leaf=80,
    l2_regularization=0.5,
    random_state=42
)

hgb_v2.fit(X_train_v2_enc, y_train_v2)

train_prob_v2 = hgb_v2.predict_proba(X_train_v2_enc)[:, 1]
val_prob_v2 = hgb_v2.predict_proba(X_val_v2_enc)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train_v2, train_prob_v2), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val_v2, val_prob_v2), 4))

Train ROC-AUC: 0.6909
Validation ROC-AUC: 0.6446


In [71]:
# Cell 43 — historical seller-category fit

df_work = df_work.sort_values("timestamp").reset_index(drop=True)

pair_key = (
    df_work["seller_id"].astype(str)
    + "_"
    + df_work["product_category"].astype(str)
)

# Prior seller-category inquiries
df_work["seller_category_previous_leads"] = (
    df_work.groupby(pair_key).cumcount()
)

# Prior seller-category conversions
df_work["seller_category_previous_conversions"] = (
    df_work.groupby(pair_key)["converted"].cumsum()
    - df_work["converted"]
)

# Historical conversion rate for this seller + category
df_work["seller_category_conversion_rate"] = (
    df_work["seller_category_previous_conversions"]
    /
    df_work["seller_category_previous_leads"].replace(0, np.nan)
).fillna(0)

print(
    df_work[
        [
            "seller_id",
            "product_category",
            "seller_category_previous_leads",
            "seller_category_conversion_rate"
        ]
    ].head(15)
)

print("\nNon-zero category history:",
      (df_work["seller_category_previous_leads"] > 0).mean())

    seller_id        product_category  seller_category_previous_leads  \
0          35    Industrial Equipment                               0   
1         721  Construction Materials                               0   
2         214   Electrical Components                               0   
3         712         Office Supplies                               0   
4         565    Industrial Equipment                               0   
5         203  Construction Materials                               0   
6         470               Packaging                               0   
7          37              Auto Parts                               0   
8         702           Steel & Metal                               0   
9         304        Food Ingredients                               0   
10        384           Steel & Metal                               0   
11        158         Office Supplies                               0   
12        365               Packaging              

In [72]:
# Cell 44

df_work["seller_category_experience"] = (
    np.log1p(df_work["seller_category_previous_leads"])
)

df_work["seller_category_success"] = (
    df_work["seller_category_previous_conversions"]
    /
    df_work["seller_category_previous_leads"].replace(0, np.nan)
).fillna(0)

print(
    df_work[
        [
            "seller_category_previous_leads",
            "seller_category_experience",
            "seller_category_success"
        ]
    ].describe().round(3)
)

print(
    "\nLeads with seller-category history:",
    round(
        (df_work["seller_category_previous_leads"] > 0).mean(),
        4
    )
)

       seller_category_previous_leads  seller_category_experience  \
count                       20000.000                   20000.000   
mean                            1.000                       0.546   
std                             1.152                       0.533   
min                             0.000                       0.000   
25%                             0.000                       0.000   
50%                             1.000                       0.693   
75%                             2.000                       1.099   
max                             7.000                       2.079   

       seller_category_success  
count                20000.000  
mean                     0.093  
std                      0.249  
min                      0.000  
25%                      0.000  
50%                      0.000  
75%                      0.000  
max                      1.000  

Leads with seller-category history: 0.5674


In [73]:
# Cell 45

feature_cols_v3 = feature_cols + [
    "seller_category_experience",
    "seller_category_success"
]

train_v3 = df_work.iloc[:14000].copy()
validation_v3 = df_work.iloc[14000:17000].copy()

X_train_v3 = train_v3[feature_cols_v3]
X_val_v3 = validation_v3[feature_cols_v3]

y_train_v3 = train_v3["converted"]
y_val_v3 = validation_v3["converted"]

categorical_v3 = X_train_v3.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_v3 = [
    c for c in feature_cols_v3
    if c not in categorical_v3
]

preprocessor_v3 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_v3
        ),
        (
            "num",
            "passthrough",
            numeric_v3
        )
    ]
)

X_train_v3_enc = preprocessor_v3.fit_transform(X_train_v3)
X_val_v3_enc = preprocessor_v3.transform(X_val_v3)

hgb_v3 = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.07,
    max_leaf_nodes=7,
    min_samples_leaf=80,
    l2_regularization=0.5,
    random_state=42
)

hgb_v3.fit(X_train_v3_enc, y_train_v3)

train_prob_v3 = hgb_v3.predict_proba(X_train_v3_enc)[:, 1]
val_prob_v3 = hgb_v3.predict_proba(X_val_v3_enc)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train_v3, train_prob_v3), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val_v3, val_prob_v3), 4))

/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/3435277446.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_v3 = X_train_v3.select_dtypes(


Train ROC-AUC: 0.6934
Validation ROC-AUC: 0.643


In [74]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_hgb, y_train)

rf_train_prob = rf.predict_proba(X_train_hgb)[:, 1]
rf_val_prob = rf.predict_proba(X_val_hgb)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, rf_train_prob), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, rf_val_prob), 4))

Train ROC-AUC: 0.7323
Validation ROC-AUC: 0.6458


In [75]:
rf_regularized = RandomForestClassifier(
    n_estimators=700,
    max_depth=5,
    min_samples_leaf=50,
    max_features=0.7,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_regularized.fit(X_train_hgb, y_train)

rf_reg_train = rf_regularized.predict_proba(
    X_train_hgb
)[:, 1]

rf_reg_val = rf_regularized.predict_proba(
    X_val_hgb
)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train, rf_reg_train), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, rf_reg_val), 4))


Train ROC-AUC: 0.6942
Validation ROC-AUC: 0.6404


In [76]:
# Cell 48 — buyer recency feature

df_work = df_work.sort_values("timestamp").reset_index(drop=True)

last_buyer_inquiry = (
    df_work.groupby("buyer_id")["timestamp"]
    .shift(1)
)

df_work["days_since_buyer_inquiry"] = (
    df_work["timestamp"] - last_buyer_inquiry
).dt.total_seconds() / (24 * 3600)

# New buyers have no previous inquiry
df_work["days_since_buyer_inquiry"] = (
    df_work["days_since_buyer_inquiry"]
    .fillna(999)
)

# Recency signal: higher = more recently active
df_work["buyer_recency_score"] = (
    np.exp(
        -df_work["days_since_buyer_inquiry"] / 30
    )
)

print(
    df_work[
        [
            "buyer_id",
            "days_since_buyer_inquiry",
            "buyer_recency_score"
        ]
    ].head(20)
)

print("\nRecency statistics:")
print(
    df_work["days_since_buyer_inquiry"]
    .describe()
    .round(2)
)

    buyer_id  days_since_buyer_inquiry  buyer_recency_score
0        447                     999.0         3.451388e-15
1       3870                     999.0         3.451388e-15
2       3273                     999.0         3.451388e-15
3       2195                     999.0         3.451388e-15
4       2166                     999.0         3.451388e-15
5       4293                     999.0         3.451388e-15
6        430                     999.0         3.451388e-15
7       3487                     999.0         3.451388e-15
8       1008                     999.0         3.451388e-15
9        471                     999.0         3.451388e-15
10      2633                     999.0         3.451388e-15
11      4879                     999.0         3.451388e-15
12      3679                     999.0         3.451388e-15
13      3806                     999.0         3.451388e-15
14      3588                     999.0         3.451388e-15
15      3931                     999.0  

In [77]:
# Cell 49 — corrected buyer recency

df_work["buyer_has_history"] = (
    df_work["previous_inquiries"] > 0
).astype(int)

df_work["buyer_recency_score"] = np.where(
    df_work["buyer_has_history"] == 1,
    np.exp(
        -df_work["days_since_buyer_inquiry"] / 30
    ),
    0
)

print(
    df_work[
        [
            "previous_inquiries",
            "days_since_buyer_inquiry",
            "buyer_has_history",
            "buyer_recency_score"
        ]
    ].head(20)
)

print("\nBuyer history rate:",
      round(df_work["buyer_has_history"].mean(), 4))

print("\nRecency score:")
print(
    df_work["buyer_recency_score"]
    .describe()
    .round(4)
)

    previous_inquiries  days_since_buyer_inquiry  buyer_has_history  \
0                    0                     999.0                  0   
1                    0                     999.0                  0   
2                    0                     999.0                  0   
3                    0                     999.0                  0   
4                    0                     999.0                  0   
5                    0                     999.0                  0   
6                    0                     999.0                  0   
7                    0                     999.0                  0   
8                    0                     999.0                  0   
9                    0                     999.0                  0   
10                   0                     999.0                  0   
11                   0                     999.0                  0   
12                   0                     999.0                  0   
13    

In [78]:
# Cell 50 — add buyer recency features

feature_cols_v4 = feature_cols + [
    "buyer_has_history",
    "buyer_recency_score"
]

train_v4 = df_work.iloc[:14000].copy()
validation_v4 = df_work.iloc[14000:17000].copy()

X_train_v4 = train_v4[feature_cols_v4]
X_val_v4 = validation_v4[feature_cols_v4]

y_train_v4 = train_v4["converted"]
y_val_v4 = validation_v4["converted"]

categorical_v4 = X_train_v4.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_v4 = [
    c for c in feature_cols_v4
    if c not in categorical_v4
]

preprocessor_v4 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_v4
        ),
        (
            "num",
            "passthrough",
            numeric_v4
        )
    ]
)

X_train_v4_enc = preprocessor_v4.fit_transform(X_train_v4)
X_val_v4_enc = preprocessor_v4.transform(X_val_v4)

hgb_v4 = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.07,
    max_leaf_nodes=7,
    min_samples_leaf=80,
    l2_regularization=0.5,
    random_state=42
)

hgb_v4.fit(X_train_v4_enc, y_train_v4)

train_prob_v4 = hgb_v4.predict_proba(X_train_v4_enc)[:, 1]
val_prob_v4 = hgb_v4.predict_proba(X_val_v4_enc)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train_v4, train_prob_v4), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val_v4, val_prob_v4), 4))


/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/3780964084.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_v4 = X_train_v4.select_dtypes(


Train ROC-AUC: 0.689
Validation ROC-AUC: 0.6426


In [79]:
# Cell 51 — LeadIQ ranking evaluation

from sklearn.metrics import roc_auc_score

val_scores = best_hgb.predict_proba(X_val_hgb)[:, 1]

ranking_df = pd.DataFrame({
    "actual": y_val.reset_index(drop=True),
    "score": val_scores
})

ranking_df = ranking_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

n_top = int(len(ranking_df) * 0.10)

top_10 = ranking_df.iloc[:n_top]

precision_10 = top_10["actual"].mean()
recall_10 = (
    top_10["actual"].sum()
    / ranking_df["actual"].sum()
)

overall_conversion = ranking_df["actual"].mean()

lift_10 = precision_10 / overall_conversion

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_scores), 4))

print("Precision@10%:",
      round(precision_10, 4))

print("Recall@10%:",
      round(recall_10, 4))

print("Lift@10%:",
      round(lift_10, 4))

print("Top 10% leads:",
      n_top)

Validation ROC-AUC: 0.645
Precision@10%: 0.4067
Recall@10%: 0.188
Lift@10%: 1.8798
Top 10% leads: 300


In [80]:
from sklearn.ensemble import ExtraTreesClassifier

extra = ExtraTreesClassifier(
    n_estimators=700,
    max_depth=8,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

extra.fit(X_train_hgb, y_train)

extra_train_prob = extra.predict_proba(X_train_hgb)[:, 1]
extra_val_prob = extra.predict_proba(X_val_hgb)[:, 1]

print("Train ROC-AUC:",
      round(
          roc_auc_score(y_train, extra_train_prob),
          4
      ))

print("Validation ROC-AUC:",
      round(
          roc_auc_score(y_val, extra_val_prob),
          4
      ))


Train ROC-AUC: 0.7143
Validation ROC-AUC: 0.6384


In [81]:
# Cell 53 — historical buyer inquiry velocity

df_work = df_work.sort_values("timestamp").reset_index(drop=True)

df_work["buyer_30d_inquiries"] = 0

for buyer, idx in df_work.groupby("buyer_id").groups.items():
    times = df_work.loc[idx, "timestamp"].values

    counts = []

    for i, current_time in enumerate(times):
        cutoff = current_time - np.timedelta64(30, "D")
        counts.append(
            np.sum(
                (times[:i] >= cutoff) &
                (times[:i] < current_time)
            )
        )

    df_work.loc[idx, "buyer_30d_inquiries"] = counts

print(
    df_work["buyer_30d_inquiries"].describe().round(2)
)

print("\nSample:")
print(
    df_work[
        [
            "buyer_id",
            "timestamp",
            "previous_inquiries",
            "buyer_30d_inquiries"
        ]
    ].head(20)
)


count    20000.00
mean         0.21
std          0.46
min          0.00
25%          0.00
50%          0.00
75%          0.00
max          4.00
Name: buyer_30d_inquiries, dtype: float64

Sample:
    buyer_id                  timestamp  previous_inquiries  \
0        447 2025-01-01 00:00:00.000000                   0   
1       3870 2025-01-01 00:39:14.517725                   0   
2       3273 2025-01-01 01:18:29.035451                   0   
3       2195 2025-01-01 01:57:43.553177                   0   
4       2166 2025-01-01 02:36:58.070903                   0   
5       4293 2025-01-01 03:16:12.588629                   0   
6        430 2025-01-01 03:55:27.106355                   0   
7       3487 2025-01-01 04:34:41.624081                   0   
8       1008 2025-01-01 05:13:56.141807                   0   
9        471 2025-01-01 05:53:10.659532                   0   
10      2633 2025-01-01 06:32:25.177258                   0   
11      4879 2025-01-01 07:11:39.694984          

In [82]:
# Cell 54

feature_cols_v5 = feature_cols + [
    "buyer_30d_inquiries"
]

train_v5 = df_work.iloc[:14000].copy()
validation_v5 = df_work.iloc[14000:17000].copy()

X_train_v5 = train_v5[feature_cols_v5]
X_val_v5 = validation_v5[feature_cols_v5]

y_train_v5 = train_v5["converted"]
y_val_v5 = validation_v5["converted"]

categorical_v5 = X_train_v5.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_v5 = [
    c for c in feature_cols_v5
    if c not in categorical_v5
]

preprocessor_v5 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_v5
        ),
        (
            "num",
            "passthrough",
            numeric_v5
        )
    ]
)

X_train_v5_enc = preprocessor_v5.fit_transform(X_train_v5)
X_val_v5_enc = preprocessor_v5.transform(X_val_v5)

hgb_v5 = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.07,
    max_leaf_nodes=7,
    min_samples_leaf=80,
    l2_regularization=0.5,
    random_state=42
)

hgb_v5.fit(X_train_v5_enc, y_train_v5)

train_prob_v5 = hgb_v5.predict_proba(X_train_v5_enc)[:, 1]
val_prob_v5 = hgb_v5.predict_proba(X_val_v5_enc)[:, 1]

print("Train ROC-AUC:",
      round(roc_auc_score(y_train_v5, train_prob_v5), 4))

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val_v5, val_prob_v5), 4))

/var/folders/ls/tjlznqp52vj3hx5v1zb_8md80000gn/T/ipykernel_69370/1857771602.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_v5 = X_train_v5.select_dtypes(


Train ROC-AUC: 0.6913
Validation ROC-AUC: 0.6442


In [83]:
# Cell 55 — ranking stability across validation segments

ranking_eval = validation_df.copy()

ranking_eval["score"] = best_hgb.predict_proba(
    X_val_hgb
)[:, 1]

ranking_eval["score_decile"] = pd.qcut(
    ranking_eval["score"],
    10,
    labels=False,
    duplicates="drop"
)

decile_performance = (
    ranking_eval
    .groupby("score_decile", observed=True)["converted"]
    .agg(
        conversion_rate="mean",
        leads="count"
    )
    .sort_index(ascending=False)
)

overall_rate = validation_df["converted"].mean()

decile_performance["lift"] = (
    decile_performance["conversion_rate"]
    / overall_rate
)

print(decile_performance.round(4))

              conversion_rate  leads    lift
score_decile                                
9                      0.4080    299  1.8861
8                      0.2776    299  1.2832
7                      0.2919    298  1.3495
6                      0.2442    303  1.1289
5                      0.2067    300  0.9553
4                      0.2126    301  0.9829
3                      0.1833    300  0.8475
2                      0.1000    300  0.4622
1                      0.1367    300  0.6317
0                      0.1033    300  0.4777


In [84]:
# Cell 56 — focused ranking model search

ranking_configs = [
    {
        "learning_rate": 0.03,
        "max_leaf_nodes": 5,
        "min_samples_leaf": 100,
        "l2_regularization": 1.0
    },
    {
        "learning_rate": 0.05,
        "max_leaf_nodes": 5,
        "min_samples_leaf": 100,
        "l2_regularization": 1.0
    },
    {
        "learning_rate": 0.07,
        "max_leaf_nodes": 5,
        "min_samples_leaf": 100,
        "l2_regularization": 0.5
    },
    {
        "learning_rate": 0.05,
        "max_leaf_nodes": 7,
        "min_samples_leaf": 120,
        "l2_regularization": 0.5
    },
    {
        "learning_rate": 0.07,
        "max_leaf_nodes": 7,
        "min_samples_leaf": 120,
        "l2_regularization": 1.0
    }
]

ranking_results = []

for i, params in enumerate(ranking_configs):

    model = HistGradientBoostingClassifier(
        max_iter=500,
        random_state=42,
        **params
    )

    model.fit(X_train_hgb, y_train)

    scores = model.predict_proba(
        X_val_hgb
    )[:, 1]

    auc = roc_auc_score(y_val, scores)

    temp = pd.DataFrame({
        "actual": y_val.reset_index(drop=True),
        "score": scores
    }).sort_values(
        "score",
        ascending=False
    )

    top_n = int(len(temp) * 0.10)

    precision_10 = temp.head(top_n)["actual"].mean()
    lift_10 = precision_10 / y_val.mean()

    ranking_results.append({
        "model": i + 1,
        "validation_auc": auc,
        "precision_at_10": precision_10,
        "lift_at_10": lift_10,
        **params
    })

ranking_results_df = pd.DataFrame(
    ranking_results
).sort_values(
    "validation_auc",
    ascending=False
)

print(
    ranking_results_df.round(4).to_string(index=False)
)

 model  validation_auc  precision_at_10  lift_at_10  learning_rate  max_leaf_nodes  min_samples_leaf  l2_regularization
     4          0.6444           0.4133      1.9106           0.05               7               120                0.5
     5          0.6442           0.4100      1.8952           0.07               7               120                1.0
     1          0.6441           0.3867      1.7874           0.03               5               100                1.0
     2          0.6436           0.3900      1.8028           0.05               5               100                1.0
     3          0.6436           0.3867      1.7874           0.07               5               100                0.5


In [85]:
# Cell 57 — Final future-period evaluation

final_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_leaf_nodes=7,
    min_samples_leaf=120,
    l2_regularization=0.5,
    max_iter=500,
    random_state=42
)

final_model.fit(X_train_hgb, y_train)

future_scores = final_model.predict_proba(X_test_hgb)[:, 1]

future_auc = roc_auc_score(y_test, future_scores)

future_eval = pd.DataFrame({
    "actual": y_test.reset_index(drop=True),
    "score": future_scores
}).sort_values(
    "score",
    ascending=False
)

top_n = int(len(future_eval) * 0.10)

future_precision_10 = future_eval.head(top_n)["actual"].mean()
future_lift_10 = future_precision_10 / y_test.mean()

print("Future Test ROC-AUC:", round(future_auc, 4))
print("Future Test Precision@10%:", round(future_precision_10, 4))
print("Future Test Lift@10%:", round(future_lift_10, 4))
print("Future Test Conversion Rate:", round(y_test.mean(), 4))

Future Test ROC-AUC: 0.6528
Future Test Precision@10%: 0.3967
Future Test Lift@10%: 1.7761
Future Test Conversion Rate: 0.2233


In [86]:
# Cell 58 — Production scoring function

def score_leads(leads_df, model=final_model):
    
    scored = leads_df.copy()

    # Generate model probabilities
    probabilities = model.predict_proba(
        preprocessor.transform(scored[production_features])
    )[:, 1]

    scored["conversion_probability"] = probabilities

    # 0–100 LeadIQ priority score
    scored["priority_score"] = (
        scored["conversion_probability"] * 100
    ).round(1)

    # Rank-based priority tiers
    scored["priority_tier"] = pd.qcut(
        scored["priority_score"].rank(method="first"),
        q=[0, 0.10, 0.30, 1.0],
        labels=["Low", "Medium", "High"]
    )

    return scored


print("Scoring function created successfully.")

Scoring function created successfully.


In [87]:
# Cell 59 — Score future leads

future_scored = score_leads(
    validation_df.copy(),
    model=final_model
)

display(
    future_scored[
        [
            "lead_id",
            "conversion_probability",
            "priority_score",
            "priority_tier",
            "urgency",
            "buyer_company_size",
            "inquiry_value",
            "previous_orders",
            "repeat_buyer",
            "converted"
        ]
    ]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(20)
)

NameError: name 'production_features' is not defined

In [88]:
production_features = [
    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "product_category",
    "inquiry_value",
    "quantity",
    "urgency",
    "lead_source",
    "seller_rating",
    "seller_experience",
    "previous_conversion_rate",
    "buyer_seller_previous_inquiries"
]

print("Production features:", len(production_features))
print(production_features)

Production features: 15
['buyer_industry', 'buyer_location', 'buyer_company_size', 'previous_orders', 'previous_inquiries', 'repeat_buyer', 'product_category', 'inquiry_value', 'quantity', 'urgency', 'lead_source', 'seller_rating', 'seller_experience', 'previous_conversion_rate', 'buyer_seller_previous_inquiries']


In [89]:
# Cell 60 — inspect relevant model objects

for name in globals():
    if any(x in name.lower() for x in [
        "preprocess",
        "pipeline",
        "transform",
        "encoder"
    ]):
        print(name, type(globals()[name]))

RuntimeError: dictionary changed size during iteration

In [90]:
print(type(final_model))
print(type(X_train_hgb))
print(X_train_hgb.shape)

<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>
<class 'numpy.ndarray'>
(14000, 46)


In [91]:
# Cell 61 — find preprocessing objects safely

names = list(globals().keys())

for name in names:
    obj = globals()[name]
    if hasattr(obj, "transform") and hasattr(obj, "fit"):
        print(name, type(obj))

ColumnTransformer <class 'abc.ABCMeta'>
OneHotEncoder <class 'type'>
StandardScaler <class 'type'>
Pipeline <class 'abc.ABCMeta'>
SimpleImputer <class 'type'>
preprocessor <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_hgb <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_base <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_fit <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_v2 <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_v3 <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_v4 <class 'sklearn.compose._column_transformer.ColumnTransformer'>
preprocessor_v5 <class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [92]:
X_check = preprocessor_hgb.transform(
    validation_df[production_features]
)

print("Transformed shape:", X_check.shape)
print("Expected shape:", X_train_hgb.shape)

Transformed shape: (3000, 46)
Expected shape: (14000, 46)


In [93]:
# Cell 63 — Final production scoring function

def score_leads(leads_df, model=final_model):
    
    scored = leads_df.copy()

    # Transform raw production features
    X = preprocessor_hgb.transform(
        scored[production_features]
    )

    # Predict conversion probability
    scored["conversion_probability"] = (
        model.predict_proba(X)[:, 1]
    )

    # LeadIQ score: 0–100
    scored["priority_score"] = (
        scored["conversion_probability"] * 100
    ).round(1)

    # Rank leads by score
    scored["rank"] = (
        scored["priority_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Product priority tiers
    scored["priority_tier"] = pd.cut(
        scored["rank"],
        bins=[
            0,
            int(len(scored) * 0.10),
            int(len(scored) * 0.30),
            len(scored)
        ],
        labels=["High", "Medium", "Low"],
        include_lowest=True
    )

    return scored


print("LeadIQ scoring pipeline ready.")

LeadIQ scoring pipeline ready.


In [94]:
# Cell 64 — Generate LeadIQ scores

leadiq_output = score_leads(
    validation_df.copy()
)

top_leads = (
    leadiq_output
    .sort_values("rank")
    .head(20)
)

display(
    top_leads[
        [
            "lead_id",
            "conversion_probability",
            "priority_score",
            "priority_tier",
            "urgency",
            "buyer_company_size",
            "inquiry_value",
            "quantity",
            "previous_orders",
            "previous_inquiries",
            "repeat_buyer",
            "seller_rating"
        ]
    ]
)

KeyError: "['lead_id'] not in index"

In [95]:
print(validation_df.columns.tolist())

['timestamp', 'buyer_id', 'seller_id', 'buyer_industry', 'buyer_location', 'buyer_company_size', 'product_category', 'urgency', 'lead_source', 'inquiry_hour', 'seller_rating', 'seller_experience', 'previous_inquiries', 'repeat_buyer', 'buyer_propensity', 'previous_orders', 'quantity', 'inquiry_value', 'buyer_seller_previous_inquiries', 'buyer_seller_previous_orders', 'seller_quality', 'converted', 'previous_conversion_rate']


In [96]:
# Cell 65 — Inspect top LeadIQ-ranked leads

top_leads = (
    leadiq_output
    .sort_values("rank")
    .head(20)
)

display(
    top_leads[
        [
            "buyer_id",
            "seller_id",
            "conversion_probability",
            "priority_score",
            "priority_tier",
            "urgency",
            "buyer_company_size",
            "inquiry_value",
            "quantity",
            "previous_orders",
            "previous_inquiries",
            "repeat_buyer",
            "seller_rating"
        ]
    ]
)

,buyer_id,seller_id,conversion_probability,priority_score,priority_tier,urgency,buyer_company_size,inquiry_value,quantity,previous_orders,previous_inquiries,repeat_buyer,seller_rating
15227,2077,606,0.446206,44.6,High,Critical,Large,24577.57,11,2,2,1,4.47
14940,1697,961,0.439382,43.9,High,Critical,Large,13717.86,10,4,5,1,4.55
14544,1726,480,0.434851,43.5,High,Critical,Large,16227.37,15,2,2,1,4.70
16882,3396,97,0.431441,43.1,High,Critical,Enterprise,44975.75,17,6,8,1,4.28
14049,503,125,0.427620,42.8,High,Critical,Large,6707.18,8,2,3,1,4.49
15862,367,343,0.425422,42.5,High,Critical,Enterprise,16754.57,14,2,3,1,4.02
16973,4710,874,0.423687,42.4,High,Critical,Large,14756.26,8,4,4,1,4.59
14100,4900,59,0.423461,42.3,High,Critical,Enterprise,28116.81,12,4,4,1,3.85
15992,1714,167,0.419876,42.0,High,Critical,Enterprise,25140.60,16,2,7,1,4.44
14728,2104,535,0.418692,41.9,High,Critical,Enterprise,29392.79,13,3,3,1,3.81


In [97]:
def generate_lead_explanation(row):
    
    reasons = []

    if row["urgency"] == "Critical":
        reasons.append("Critical urgency")
    elif row["urgency"] == "High":
        reasons.append("High urgency")

    if row["repeat_buyer"] == 1:
        reasons.append(
            f"Repeat buyer with {int(row['previous_orders'])} previous orders"
        )

    if row["buyer_company_size"] in ["Large", "Enterprise"]:
        reasons.append(
            f"{row['buyer_company_size']} buyer"
        )

    if row["inquiry_value"] >= 20000:
        reasons.append(
            f"High inquiry value (₹{row['inquiry_value']:,.0f})"
        )

    if row["quantity"] >= 10:
        reasons.append(
            f"Large quantity ({int(row['quantity'])} units)"
        )

    if row["seller_rating"] >= 4.5:
        reasons.append(
            f"Strong seller rating ({row['seller_rating']:.1f})"
        )

    return reasons[:4]


leadiq_output["why_prioritize"] = (
    leadiq_output.apply(
        generate_lead_explanation,
        axis=1
    )
)

print("Explanation layer created.")

Explanation layer created.


In [98]:
def recommend_action(row):
    
    if row["priority_tier"] == "High":
        if row["urgency"] == "Critical":
            return "Respond within 15 minutes"
        elif row["urgency"] == "High":
            return "Respond within 30 minutes"
        else:
            return "Respond within 1 hour"

    elif row["priority_tier"] == "Medium":
        return "Respond within 2 hours"

    else:
        return "Handle after higher-priority leads"


leadiq_output["recommended_action"] = (
    leadiq_output.apply(
        recommend_action,
        axis=1
    )
)

print("Recommended action layer created.")

Recommended action layer created.


In [99]:
# Cell 68 — Final LeadIQ seller view

seller_view = (
    leadiq_output
    .sort_values("rank")
    [
        [
            "buyer_id",
            "seller_id",
            "priority_score",
            "conversion_probability",
            "priority_tier",
            "why_prioritize",
            "recommended_action",
            "urgency",
            "buyer_company_size",
            "inquiry_value",
            "quantity"
        ]
    ]
)

display(seller_view.head(20))

,buyer_id,seller_id,priority_score,conversion_probability,priority_tier,why_prioritize,recommended_action,urgency,buyer_company_size,inquiry_value,quantity
15227,2077,606,44.6,0.446206,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes,Critical,Large,24577.57,11
14940,1697,961,43.9,0.439382,High,"[Critical urgency, Repeat buyer with 4 previou...",Respond within 15 minutes,Critical,Large,13717.86,10
14544,1726,480,43.5,0.434851,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes,Critical,Large,16227.37,15
16882,3396,97,43.1,0.431441,High,"[Critical urgency, Repeat buyer with 6 previou...",Respond within 15 minutes,Critical,Enterprise,44975.75,17
14049,503,125,42.8,0.427620,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes,Critical,Large,6707.18,8
15862,367,343,42.5,0.425422,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes,Critical,Enterprise,16754.57,14
16973,4710,874,42.4,0.423687,High,"[Critical urgency, Repeat buyer with 4 previou...",Respond within 15 minutes,Critical,Large,14756.26,8
14100,4900,59,42.3,0.423461,High,"[Critical urgency, Repeat buyer with 4 previou...",Respond within 15 minutes,Critical,Enterprise,28116.81,12
15992,1714,167,42.0,0.419876,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes,Critical,Enterprise,25140.60,16
14728,2104,535,41.9,0.418692,High,"[Critical urgency, Repeat buyer with 3 previou...",Respond within 15 minutes,Critical,Enterprise,29392.79,13


In [100]:
seller_view.to_csv(
    "leadiq_seller_view.csv",
    index=False
)

print("Saved:", len(seller_view), "leads")
print("File: leadiq_seller_view.csv")

Saved: 3000 leads
File: leadiq_seller_view.csv


In [101]:
import os

print(os.path.abspath("leadiq_seller_view.csv"))
print("File exists:", os.path.exists("leadiq_seller_view.csv"))

/Users/sameekshadongre/Documents/LeadIQ/notebooks/leadiq_seller_view.csv
File exists: True


In [102]:
import shutil
import os

source = os.path.abspath("leadiq_seller_view.csv")
destination = "/mnt/data/LeadIQ_Web/leadiq_seller_view.csv"

shutil.copy2(source, destination)

print("Copied successfully:")
print(destination)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/LeadIQ_Web/leadiq_seller_view.csv'

In [103]:
import os

matches = []

for root, dirs, files in os.walk("/mnt/data"):
    if "leadiq_seller_view.csv" in files:
        matches.append(os.path.join(root, "leadiq_seller_view.csv"))

print(matches)

[]


In [104]:
seller_view.to_csv(
    "/mnt/data/LeadIQ_Web/leadiq_seller_view.csv",
    index=False
)

print("Saved successfully.")

OSError: Cannot save file into a non-existent directory: '/mnt/data/LeadIQ_Web'

In [105]:
import os

print("Current notebook folder:")
print(os.getcwd())

seller_view.to_csv(
    "leadiq_seller_view.csv",
    index=False
)

print("\nCSV saved successfully.")
print(
    "Full path:",
    os.path.abspath("leadiq_seller_view.csv")
)

Current notebook folder:
/Users/sameekshadongre/Documents/LeadIQ/notebooks

CSV saved successfully.
Full path: /Users/sameekshadongre/Documents/LeadIQ/notebooks/leadiq_seller_view.csv


In [106]:
from pathlib import Path

print(Path.cwd())

/Users/sameekshadongre/Documents/LeadIQ/notebooks


In [110]:
from pathlib import Path

folder = Path("products")
print(folder.exists())
print(folder.absolute())

False
/Users/sameekshadongre/Documents/LeadIQ/notebooks/products


In [108]:
from pathlib import Path

matches = list(Path.home().rglob("LeadIQ_Web"))

print("Found:", len(matches))

for p in matches[:20]:
    print(p)

Found: 0


In [111]:
from pathlib import Path

project = Path("/Documents/LeadIQ/product")

print("Exists:", project.exists())
print("Files:")
print(list(project.iterdir()))

Exists: False
Files:


FileNotFoundError: [Errno 2] No such file or directory: '/Documents/LeadIQ/product'

In [112]:
from pathlib import Path

home = Path.home()

print("YOUR HOME FOLDER:")
print(home)

print("\nFOLDERS INSIDE HOME:")
for item in home.iterdir():
    print(item)

YOUR HOME FOLDER:
/Users/sameekshadongre

FOLDERS INSIDE HOME:
/Users/sameekshadongre/Music
/Users/sameekshadongre/.expo
/Users/sameekshadongre/.DS_Store
/Users/sameekshadongre/.CFUserTextEncoding
/Users/sameekshadongre/Untitled.ipynb
/Users/sameekshadongre/.streamlit
/Users/sameekshadongre/Pictures
/Users/sameekshadongre/.zprofile
/Users/sameekshadongre/.zsh_history
/Users/sameekshadongre/.ipython
/Users/sameekshadongre/Desktop
/Users/sameekshadongre/Library
/Users/sameekshadongre/.matplotlib
/Users/sameekshadongre/.codex
/Users/sameekshadongre/Public
/Users/sameekshadongre/Movies
/Users/sameekshadongre/Applications
/Users/sameekshadongre/.Trash
/Users/sameekshadongre/.ipynb_checkpoints
/Users/sameekshadongre/.jupyter
/Users/sameekshadongre/Documents
/Users/sameekshadongre/Downloads
/Users/sameekshadongre/.cache
/Users/sameekshadongre/.zsh_sessions


In [113]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

print("Path:", project)
print("Exists:", project.exists())

if project.exists():
    print("\nFiles inside product:")
    for item in project.iterdir():
        print(" -", item.name)

Path: /Users/sameekshadongre/Documents/LeadIQ/product
Exists: True

Files inside product:
 - index.html
 - styles.css
 - leadiq_seller_view.csv
 - .ipynb_checkpoints
 - app.js


In [114]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

html = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>LeadIQ — AI Lead Intelligence</title>
    <link rel="stylesheet" href="styles.css">
</head>

<body>

<header class="top-nav">

    <div class="brand">
        <div class="brand-icon">◆</div>

        <div>
            <div class="brand-name">LeadIQ</div>
            <div class="brand-subtitle">AI Lead Intelligence</div>
        </div>
    </div>

    <nav>
        <button class="nav-item active" data-page="home">
            Overview
        </button>

        <button class="nav-item" data-page="inbox">
            Priority Inbox
        </button>
    </nav>

</header>


<main class="app-container">

    <!-- ================= OVERVIEW ================= -->

    <section id="home-page" class="page active">

        <div class="hero">

            <div>

                <div class="eyebrow">
                    AI-POWERED SELLER INTELLIGENCE
                </div>

                <h1>
                    Focus on the leads<br>
                    most likely to convert.
                </h1>

                <p>
                    LeadIQ ranks incoming buyer inquiries by predicted
                    conversion potential so sellers know where to focus first.
                </p>

            </div>

            <div class="hero-badge">
                <span class="status-dot"></span>
                Model active
            </div>

        </div>


        <!-- METRICS -->

        <div class="metrics-grid">

            <div class="metric-card">

                <div class="metric-label">
                    INCOMING LEADS
                </div>

                <div id="total-leads" class="metric-value">
                    —
                </div>

                <div class="metric-sub">
                    Currently in your pipeline
                </div>

            </div>


            <div class="metric-card">

                <div class="metric-label">
                    NEED ATTENTION
                </div>

                <div id="high-leads" class="metric-value">
                    —
                </div>

                <div class="metric-sub">
                    Top 10% ranked by AI
                </div>

            </div>


            <div class="metric-card">

                <div class="metric-label">
                    OPPORTUNITY VALUE
                </div>

                <div id="high-value" class="metric-value">
                    —
                </div>

                <div class="metric-sub">
                    High-priority inquiries
                </div>

            </div>


            <div class="metric-card">

                <div class="metric-label">
                    AI PURPOSE
                </div>

                <div class="metric-value">
                    Prioritize
                </div>

                <div class="metric-sub">
                    Focus seller attention first
                </div>

            </div>

        </div>


        <!-- TOP LEADS -->

        <section class="section">

            <div class="section-heading">

                <div>

                    <h2>
                        Prioritize Now
                    </h2>

                    <p>
                        The buyer inquiries with the highest predicted
                        conversion potential.
                    </p>

                </div>

                <button class="text-button" id="open-inbox">
                    View all →
                </button>

            </div>


            <div id="top-leads-container"></div>

        </section>


        <!-- HOW IT WORKS -->

        <section class="section">

            <div class="section-heading">

                <div>

                    <h2>
                        How LeadIQ helps
                    </h2>

                    <p>
                        Turn a large stream of inquiries into a clear
                        seller decision.
                    </p>

                </div>

            </div>


            <div class="value-grid">

                <div class="value-card">

                    <span>
                        01 · RANK
                    </span>

                    <h3>
                        Prioritize
                    </h3>

                    <p>
                        Rank incoming inquiries by predicted
                        conversion potential.
                    </p>

                </div>


                <div class="value-card">

                    <span>
                        02 · EXPLAIN
                    </span>

                    <h3>
                        Understand
                    </h3>

                    <p>
                        Show sellers why a lead deserves attention.
                    </p>

                </div>


                <div class="value-card">

                    <span>
                        03 · ACT
                    </span>

                    <h3>
                        Respond
                    </h3>

                    <p>
                        Recommend the next sales action for the lead.
                    </p>

                </div>

            </div>

        </section>

    </section>


    <!-- ================= PRIORITY INBOX ================= -->

    <section id="inbox-page" class="page">

        <div class="page-header">

            <div>

                <div class="eyebrow">
                    SELLER WORKSPACE
                </div>

                <h1>
                    AI Priority Inbox
                </h1>

                <p>
                    LeadIQ tells you where to focus first.
                </p>

            </div>


            <div class="inbox-summary">

                <span id="filtered-count">
                    0
                </span>

                ranked opportunities

            </div>

        </div>


        <!-- FILTERS -->

        <div class="filters">

            <div class="filter-group">

                <label>
                    Priority
                </label>

                <select id="priority-filter">

                    <option value="All">
                        All
                    </option>

                    <option value="High">
                        High
                    </option>

                    <option value="Medium">
                        Medium
                    </option>

                    <option value="Low">
                        Low
                    </option>

                </select>

            </div>


            <div class="filter-group">

                <label>
                    Urgency
                </label>

                <select id="urgency-filter">

                    <option value="All">
                        All
                    </option>

                </select>

            </div>


            <div class="filter-group">

                <label>
                    Product Category
                </label>

                <select id="category-filter">

                    <option value="All">
                        All
                    </option>

                </select>

            </div>


            <div class="filter-group search-group">

                <label>
                    Search
                </label>

                <input
                    id="search-input"
                    type="text"
                    placeholder="Search buyer or seller..."
                >

            </div>

        </div>


        <!-- LEAD LIST -->

        <div id="lead-list"></div>

    </section>

</main>


<!-- ================= LEAD DETAIL DRAWER ================= -->

<div id="drawer-overlay" class="drawer-overlay"></div>


<aside id="lead-drawer" class="lead-drawer">

    <div class="drawer-header">

        <div>

            <div class="eyebrow">
                LEAD DETAIL
            </div>

            <h2 id="drawer-title">
                Lead
            </h2>

        </div>


        <button
            id="close-drawer"
            class="close-button"
        >
            ×
        </button>

    </div>


    <div id="drawer-content"></div>

</aside>


<script src="app.js"></script>

</body>
</html>
"""

file_path = project / "index.html"

file_path.write_text(html, encoding="utf-8")

print("✅ index.html created successfully!")
print(file_path)


✅ index.html created successfully!
/Users/sameekshadongre/Documents/LeadIQ/product/index.html


In [115]:
print((project / "index.html").exists())
print((project / "index.html").stat().st_size, "bytes")

True
8429 bytes


In [116]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

css = """
* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    background: #f7f8fa;
    color: #15171a;
    line-height: 1.5;
}

/* ================= NAVIGATION ================= */

.top-nav {
    height: 72px;
    background: #ffffff;
    border-bottom: 1px solid #e5e7eb;
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 48px;
    position: sticky;
    top: 0;
    z-index: 100;
}

.brand {
    display: flex;
    align-items: center;
    gap: 12px;
}

.brand-icon {
    width: 34px;
    height: 34px;
    background: #111827;
    color: white;
    display: flex;
    align-items: center;
    justify-content: center;
    border-radius: 8px;
    font-size: 15px;
}

.brand-name {
    font-size: 18px;
    font-weight: 700;
    letter-spacing: -0.3px;
}

.brand-subtitle {
    font-size: 11px;
    color: #8a9099;
    margin-top: -2px;
}

.top-nav nav {
    display: flex;
    gap: 8px;
}

.nav-item {
    border: none;
    background: transparent;
    padding: 9px 16px;
    border-radius: 7px;
    color: #69707a;
    font-size: 14px;
    font-weight: 500;
    cursor: pointer;
}

.nav-item:hover {
    background: #f3f4f6;
    color: #111827;
}

.nav-item.active {
    background: #111827;
    color: #ffffff;
}


/* ================= MAIN ================= */

.app-container {
    max-width: 1400px;
    margin: 0 auto;
    padding: 48px;
}

.page {
    display: none;
}

.page.active {
    display: block;
}


/* ================= HERO ================= */

.hero {
    background: #111827;
    color: white;
    border-radius: 16px;
    padding: 54px 56px;
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    margin-bottom: 28px;
}

.eyebrow {
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1.4px;
    color: #8b93a1;
    margin-bottom: 12px;
}

.hero .eyebrow {
    color: #aeb6c2;
}

.hero h1 {
    font-size: 42px;
    line-height: 1.08;
    letter-spacing: -1.5px;
    margin-bottom: 18px;
}

.hero p {
    max-width: 650px;
    color: #b9c0ca;
    font-size: 16px;
}

.hero-badge {
    border: 1px solid #374151;
    border-radius: 999px;
    padding: 9px 14px;
    font-size: 12px;
    color: #d1d5db;
    white-space: nowrap;
}

.status-dot {
    display: inline-block;
    width: 7px;
    height: 7px;
    background: #35b66f;
    border-radius: 50%;
    margin-right: 7px;
}


/* ================= METRICS ================= */

.metrics-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    margin-bottom: 48px;
}

.metric-card {
    background: white;
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 22px;
}

.metric-label {
    font-size: 10px;
    font-weight: 700;
    letter-spacing: 1px;
    color: #8b929c;
    margin-bottom: 10px;
}

.metric-value {
    font-size: 28px;
    font-weight: 700;
    letter-spacing: -0.7px;
}

.metric-sub {
    font-size: 12px;
    color: #8b929c;
    margin-top: 4px;
}


/* ================= SECTIONS ================= */

.section {
    margin-bottom: 52px;
}

.section-heading {
    display: flex;
    align-items: center;
    justify-content: space-between;
    margin-bottom: 20px;
}

.section-heading h2 {
    font-size: 21px;
    letter-spacing: -0.4px;
}

.section-heading p {
    font-size: 13px;
    color: #7a818b;
    margin-top: 3px;
}

.text-button {
    border: none;
    background: transparent;
    color: #374151;
    font-size: 13px;
    font-weight: 600;
    cursor: pointer;
}

.text-button:hover {
    color: #000000;
}


/* ================= TOP LEADS ================= */

.lead-cards {
    display: grid;
    gap: 10px;
}

.lead-card {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 10px;
    padding: 18px 20px;
    display: grid;
    grid-template-columns: 1fr auto auto;
    align-items: center;
    gap: 24px;
    cursor: pointer;
    transition: 0.15s ease;
}

.lead-card:hover {
    border-color: #bfc4cb;
    transform: translateY(-1px);
}

.lead-main strong {
    display: block;
    font-size: 14px;
}

.lead-meta {
    font-size: 12px;
    color: #8a9099;
    margin-top: 3px;
}

.lead-score {
    text-align: right;
}

.score-value {
    font-size: 19px;
    font-weight: 700;
}

.score-label {
    font-size: 10px;
    color: #8a9099;
    text-transform: uppercase;
    letter-spacing: 0.7px;
}


/* ================= PRIORITY ================= */

.priority-badge {
    display: inline-flex;
    align-items: center;
    border-radius: 999px;
    padding: 5px 9px;
    font-size: 10px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

.priority-high {
    background: #f1f1f1;
    color: #111111;
}

.priority-medium {
    background: #e9eaec;
    color: #4b5058;
}

.priority-low {
    background: #f5f5f5;
    color: #858b93;
}


/* ================= VALUE CARDS ================= */

.value-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 16px;
}

.value-card {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 25px;
}

.value-card span {
    font-size: 10px;
    font-weight: 700;
    letter-spacing: 1px;
    color: #9298a1;
}

.value-card h3 {
    font-size: 17px;
    margin: 10px 0 6px;
}

.value-card p {
    color: #777e88;
    font-size: 13px;
}


/* ================= PAGE HEADER ================= */

.page-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-end;
    margin-bottom: 28px;
}

.page-header h1 {
    font-size: 32px;
    letter-spacing: -1px;
}

.page-header p {
    color: #7a818b;
    font-size: 14px;
    margin-top: 4px;
}

.inbox-summary {
    font-size: 13px;
    color: #7a818b;
}

.inbox-summary span {
    color: #111827;
    font-weight: 700;
}


/* ================= FILTERS ================= */

.filters {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 18px;
    display: grid;
    grid-template-columns: 180px 180px 220px 1fr;
    gap: 14px;
    margin-bottom: 18px;
}

.filter-group {
    display: flex;
    flex-direction: column;
    gap: 6px;
}

.filter-group label {
    font-size: 10px;
    text-transform: uppercase;
    font-weight: 700;
    letter-spacing: 0.8px;
    color: #8b929c;
}

.filter-group select,
.filter-group input {
    height: 38px;
    border: 1px solid #dfe2e6;
    border-radius: 7px;
    background: #ffffff;
    padding: 0 11px;
    font-size: 13px;
    color: #30343a;
    outline: none;
}

.filter-group select:focus,
.filter-group input:focus {
    border-color: #9ca3af;
}


/* ================= INBOX ================= */

.inbox-list {
    display: grid;
    gap: 8px;
}

.inbox-row {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 10px;
    padding: 16px 18px;
    display: grid;
    grid-template-columns: 70px 1.5fr 1fr 120px 120px 30px;
    align-items: center;
    gap: 14px;
    cursor: pointer;
    transition: 0.15s ease;
}

.inbox-row:hover {
    border-color: #bfc4cb;
}

.rank-number {
    font-size: 12px;
    color: #8b929c;
}

.inbox-buyer strong {
    display: block;
    font-size: 13px;
}

.inbox-buyer span,
.inbox-info span {
    display: block;
    font-size: 11px;
    color: #8b929c;
    margin-top: 2px;
}

.inbox-score {
    font-weight: 700;
    font-size: 15px;
}

.arrow {
    color: #9ca3af;
    font-size: 18px;
}


/* ================= DRAWER ================= */

.drawer-overlay {
    position: fixed;
    inset: 0;
    background: rgba(0, 0, 0, 0.28);
    opacity: 0;
    visibility: hidden;
    transition: 0.2s ease;
    z-index: 199;
}

.drawer-overlay.open {
    opacity: 1;
    visibility: visible;
}

.lead-drawer {
    position: fixed;
    top: 0;
    right: -520px;
    width: 520px;
    max-width: 100%;
    height: 100vh;
    background: #ffffff;
    z-index: 200;
    box-shadow: -12px 0 35px rgba(0, 0, 0, 0.12);
    transition: 0.25s ease;
    overflow-y: auto;
    padding: 32px;
}

.lead-drawer.open {
    right: 0;
}

.drawer-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    border-bottom: 1px solid #e5e7eb;
    padding-bottom: 20px;
    margin-bottom: 24px;
}

.drawer-header h2 {
    font-size: 22px;
    margin-top: 2px;
}

.close-button {
    width: 34px;
    height: 34px;
    border: none;
    background: #f3f4f6;
    border-radius: 7px;
    font-size: 22px;
    color: #555b63;
    cursor: pointer;
}


/* ================= DETAIL ================= */

.detail-score {
    background: #111827;
    color: white;
    border-radius: 12px;
    padding: 24px;
    margin-bottom: 18px;
}

.detail-score-label {
    font-size: 10px;
    color: #aeb6c2;
    text-transform: uppercase;
    letter-spacing: 1px;
}

.detail-score-value {
    font-size: 38px;
    font-weight: 700;
    margin-top: 3px;
}

.detail-score-sub {
    font-size: 12px;
    color: #aeb6c2;
}

.detail-section {
    border-bottom: 1px solid #e5e7eb;
    padding: 20px 0;
}

.detail-section:last-child {
    border-bottom: none;
}

.detail-section h3 {
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: 0.7px;
    color: #777e88;
    margin-bottom: 12px;
}

.detail-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 14px;
}

.detail-item label {
    display: block;
    font-size: 10px;
    color: #9298a1;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

.detail-item strong {
    display: block;
    font-size: 13px;
    margin-top: 2px;
}

.reason-list {
    display: grid;
    gap: 8px;
}

.reason-item {
    background: #f6f7f8;
    border-radius: 7px;
    padding: 9px 11px;
    font-size: 12px;
}

.action-box {
    background: #f0f1f3;
    border-radius: 9px;
    padding: 14px;
    font-size: 13px;
    font-weight: 600;
}


/* ================= AI RESPONSE ================= */

.ai-response {
    background: #f7f8fa;
    border: 1px solid #e1e4e8;
    border-radius: 10px;
    padding: 15px;
    font-size: 13px;
    line-height: 1.6;
}

.ai-label {
    font-size: 10px;
    font-weight: 700;
    color: #6f7680;
    text-transform: uppercase;
    letter-spacing: 0.8px;
    margin-bottom: 8px;
}


/* ================= RESPONSIVE ================= */

@media (max-width: 900px) {

    .app-container {
        padding: 28px;
    }

    .top-nav {
        padding: 0 28px;
    }

    .metrics-grid {
        grid-template-columns: repeat(2, 1fr);
    }

    .filters {
        grid-template-columns: 1fr 1fr;
    }

    .value-grid {
        grid-template-columns: 1fr;
    }

    .inbox-row {
        grid-template-columns: 50px 1fr 90px;
    }

    .inbox-info {
        display: none;
    }

    .hero {
        padding: 40px;
    }
}


@media (max-width: 600px) {

    .app-container {
        padding: 20px;
    }

    .top-nav {
        padding: 0 18px;
    }

    .brand-subtitle {
        display: none;
    }

    .hero {
        display: block;
        padding: 30px;
    }

    .hero h1 {
        font-size: 31px;
    }

    .hero-badge {
        display: inline-block;
        margin-top: 24px;
    }

    .metrics-grid {
        grid-template-columns: 1fr;
    }

    .filters {
        grid-template-columns: 1fr;
    }

    .page-header {
        display: block;
    }

    .inbox-summary {
        margin-top: 12px;
    }

    .lead-drawer {
        width: 100%;
        padding: 24px;
    }
}
"""

css_path = project / "styles.css"
css_path.write_text(css, encoding="utf-8")

print("✅ styles.css created successfully!")
print(css_path)

✅ styles.css created successfully!
/Users/sameekshadongre/Documents/LeadIQ/product/styles.css


In [117]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

js = r"""
let leads = [];
let filteredLeads = [];


/* ================= CSV LOADER ================= */

async function loadCSV() {

    try {

        const response = await fetch("leadiq_seller_view.csv");

        if (!response.ok) {
            throw new Error("Could not load CSV");
        }

        const text = await response.text();

        leads = parseCSV(text);

        leads = leads.map(normalizeLead);

        filteredLeads = [...leads];

        initializeApp();

    } catch (error) {

        console.error(error);

        document.getElementById("lead-list").innerHTML = `
            <div class="metric-card">
                <h3>Could not load lead data</h3>
                <p style="margin-top:8px;color:#777;">
                    Start the project using a local server and open LeadIQ
                    through http://localhost rather than opening the HTML file directly.
                </p>
            </div>
        `;
    }
}


/* ================= CSV PARSER ================= */

function parseCSV(text) {

    const rows = [];
    let row = [];
    let value = "";
    let insideQuotes = false;

    for (let i = 0; i < text.length; i++) {

        const char = text[i];
        const next = text[i + 1];

        if (char === '"' && insideQuotes && next === '"') {

            value += '"';
            i++;

        } else if (char === '"') {

            insideQuotes = !insideQuotes;

        } else if (char === "," && !insideQuotes) {

            row.push(value);
            value = "";

        } else if ((char === "\n" || char === "\r") && !insideQuotes) {

            if (char === "\r" && next === "\n") {
                i++;
            }

            row.push(value);

            if (row.some(cell => cell.trim() !== "")) {
                rows.push(row);
            }

            row = [];
            value = "";

        } else {

            value += char;
        }
    }

    if (value !== "" || row.length > 0) {

        row.push(value);

        if (row.some(cell => cell.trim() !== "")) {
            rows.push(row);
        }
    }

    if (rows.length === 0) {
        return [];
    }

    const headers = rows[0].map(h => h.trim());

    return rows.slice(1).map(row => {

        const obj = {};

        headers.forEach((header, index) => {
            obj[header] = (row[index] || "").trim();
        });

        return obj;
    });
}


/* ================= DATA NORMALIZATION ================= */

function normalizeLead(row) {

    const numericFields = [
        "buyer_id",
        "seller_id",
        "priority_score",
        "conversion_probability",
        "rank",
        "inquiry_value",
        "quantity",
        "previous_orders",
        "previous_inquiries",
        "repeat_buyer",
        "seller_rating",
        "seller_experience"
    ];

    numericFields.forEach(field => {

        if (row[field] !== undefined && row[field] !== "") {
            row[field] = Number(row[field]);
        }
    });

    return row;
}


/* ================= INITIALIZATION ================= */

function initializeApp() {

    setupNavigation();

    setupFilters();

    setupDrawer();

    renderOverview();

    renderInbox();
}


/* ================= NAVIGATION ================= */

function setupNavigation() {

    const navItems = document.querySelectorAll(".nav-item");

    navItems.forEach(button => {

        button.addEventListener("click", () => {

            const page = button.dataset.page;

            showPage(page);

        });
    });


    document
        .getElementById("open-inbox")
        .addEventListener("click", () => {

            showPage("inbox");

        });
}


function showPage(page) {

    document.querySelectorAll(".page").forEach(section => {
        section.classList.remove("active");
    });

    document.querySelectorAll(".nav-item").forEach(button => {
        button.classList.remove("active");
    });


    if (page === "home") {

        document
            .getElementById("home-page")
            .classList.add("active");

        document
            .querySelector('[data-page="home"]')
            .classList.add("active");

    } else {

        document
            .getElementById("inbox-page")
            .classList.add("active");

        document
            .querySelector('[data-page="inbox"]')
            .classList.add("active");

    }
}


/* ================= OVERVIEW ================= */

function renderOverview() {

    const total = leads.length;

    const highLeads = leads.filter(
        lead => lead.priority_tier === "High"
    );

    const highValue = highLeads.reduce(
        (sum, lead) => sum + Number(lead.inquiry_value || 0),
        0
    );


    document.getElementById("total-leads").textContent =
        total.toLocaleString();


    document.getElementById("high-leads").textContent =
        highLeads.length.toLocaleString();


    document.getElementById("high-value").textContent =
        formatINR(highValue);


    renderTopLeads();
}


/* ================= TOP LEADS ================= */

function renderTopLeads() {

    const container =
        document.getElementById("top-leads-container");

    const topLeads = [...leads]
        .sort((a, b) => Number(a.rank) - Number(b.rank))
        .slice(0, 5);


    container.innerHTML = `
        <div class="lead-cards">
            ${topLeads.map(createTopLeadCard).join("")}
        </div>
    `;


    container
        .querySelectorAll(".lead-card")
        .forEach(card => {

            card.addEventListener("click", () => {

                const index = Number(card.dataset.index);

                openLeadDrawer(topLeads[index]);

            });

        });
}


function createTopLeadCard(lead, index) {

    return `
        <div class="lead-card" data-index="${index}">

            <div class="lead-main">

                <strong>
                    Buyer ${lead.buyer_id || "Unknown"}
                </strong>

                <div class="lead-meta">
                    ${lead.product_category || "Product inquiry"}
                    · ${lead.urgency || "Normal"} urgency
                </div>

            </div>


            <div>
                <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                    ${lead.priority_tier || "—"}
                </span>
            </div>


            <div class="lead-score">

                <div class="score-value">
                    ${formatNumber(lead.priority_score)}%
                </div>

                <div class="score-label">
                    AI priority score
                </div>

            </div>

        </div>
    `;
}


/* ================= FILTERS ================= */

function setupFilters() {

    const urgencyFilter =
        document.getElementById("urgency-filter");

    const categoryFilter =
        document.getElementById("category-filter");


    const urgencies = uniqueValues("urgency");

    urgencies.forEach(value => {

        const option = document.createElement("option");

        option.value = value;
        option.textContent = value;

        urgencyFilter.appendChild(option);

    });


    const categories = uniqueValues("product_category");

    categories.forEach(value => {

        const option = document.createElement("option");

        option.value = value;
        option.textContent = value;

        categoryFilter.appendChild(option);

    });


    document
        .getElementById("priority-filter")
        .addEventListener("change", applyFilters);


    urgencyFilter.addEventListener(
        "change",
        applyFilters
    );


    categoryFilter.addEventListener(
        "change",
        applyFilters
    );


    document
        .getElementById("search-input")
        .addEventListener("input", applyFilters);
}


function uniqueValues(field) {

    return [
        ...new Set(
            leads
                .map(lead => lead[field])
                .filter(value => value !== undefined && value !== "")
        )
    ].sort();
}


/* ================= FILTER LOGIC ================= */

function applyFilters() {

    const priority =
        document.getElementById("priority-filter").value;

    const urgency =
        document.getElementById("urgency-filter").value;

    const category =
        document.getElementById("category-filter").value;

    const search =
        document
            .getElementById("search-input")
            .value
            .toLowerCase()
            .trim();


    filteredLeads = leads.filter(lead => {

        const matchesPriority =
            priority === "All" ||
            lead.priority_tier === priority;


        const matchesUrgency =
            urgency === "All" ||
            lead.urgency === urgency;


        const matchesCategory =
            category === "All" ||
            lead.product_category === category;


        const searchableText = [
            lead.buyer_id,
            lead.seller_id,
            lead.product_category,
            lead.buyer_industry,
            lead.buyer_location
        ]
            .join(" ")
            .toLowerCase();


        const matchesSearch =
            search === "" ||
            searchableText.includes(search);


        return (
            matchesPriority &&
            matchesUrgency &&
            matchesCategory &&
            matchesSearch
        );

    });


    renderInbox();
}


/* ================= PRIORITY INBOX ================= */

function renderInbox() {

    const container =
        document.getElementById("lead-list");


    document.getElementById("filtered-count").textContent =
        filteredLeads.length.toLocaleString();


    if (filteredLeads.length === 0) {

        container.innerHTML = `
            <div class="metric-card">
                <h3>No leads found</h3>
                <p style="margin-top:8px;color:#777;">
                    Try changing your filters.
                </p>
            </div>
        `;

        return;
    }


    const sorted = [...filteredLeads]
        .sort((a, b) => Number(a.rank) - Number(b.rank));


    container.innerHTML = `
        <div class="inbox-list">

            ${sorted.map((lead, index) =>
                createInboxRow(lead, index)
            ).join("")}

        </div>
    `;


    container
        .querySelectorAll(".inbox-row")
        .forEach((row, index) => {

            row.addEventListener("click", () => {

                openLeadDrawer(sorted[index]);

            });

        });
}


function createInboxRow(lead, index) {

    return `
        <div class="inbox-row">

            <div class="rank-number">
                #${lead.rank || index + 1}
            </div>


            <div class="inbox-buyer">

                <strong>
                    Buyer ${lead.buyer_id || "Unknown"}
                </strong>

                <span>
                    Seller ${lead.seller_id || "Unknown"}
                </span>

            </div>


            <div class="inbox-info">

                <strong>
                    ${lead.product_category || "Product inquiry"}
                </strong>

                <span>
                    ${lead.urgency || "Normal"} urgency
                </span>

            </div>


            <div>

                <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                    ${lead.priority_tier || "—"}
                </span>

            </div>


            <div class="inbox-score">

                ${formatNumber(lead.priority_score)}%

            </div>


            <div class="arrow">
                →
            </div>

        </div>
    `;
}


/* ================= DRAWER ================= */

function setupDrawer() {

    document
        .getElementById("close-drawer")
        .addEventListener("click", closeDrawer);


    document
        .getElementById("drawer-overlay")
        .addEventListener("click", closeDrawer);


    document.addEventListener("keydown", event => {

        if (event.key === "Escape") {
            closeDrawer();
        }

    });
}


function openLeadDrawer(lead) {

    const drawer =
        document.getElementById("lead-drawer");

    const overlay =
        document.getElementById("drawer-overlay");


    document.getElementById("drawer-title").textContent =
        `Buyer ${lead.buyer_id || "Unknown"}`;


    document.getElementById("drawer-content").innerHTML =
        createLeadDetail(lead);


    drawer.classList.add("open");

    overlay.classList.add("open");
}


function closeDrawer() {

    document
        .getElementById("lead-drawer")
        .classList.remove("open");


    document
        .getElementById("drawer-overlay")
        .classList.remove("open");
}


/* ================= LEAD DETAIL ================= */

function createLeadDetail(lead) {

    const reasons = generateReasons(lead);

    const action = getRecommendedAction(lead);

    const response = generateSmartResponse(lead);


    return `

        <div class="detail-score">

            <div class="detail-score-label">
                AI CONVERSION POTENTIAL
            </div>

            <div class="detail-score-value">
                ${formatNumber(lead.priority_score)}%
            </div>

            <div class="detail-score-sub">
                Predicted probability of conversion
            </div>

        </div>


        <div class="detail-section">

            <h3>
                Priority
            </h3>

            <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                ${lead.priority_tier}
            </span>

        </div>


        <div class="detail-section">

            <h3>
                Buyer & Requirement
            </h3>

            <div class="detail-grid">

                <div class="detail-item">

                    <label>
                        Buyer
                    </label>

                    <strong>
                        ${lead.buyer_id || "—"}
                    </strong>

                </div>


                <div class="detail-item">

                    <label>
                        Seller
                    </label>

                    <strong>
                        ${lead.seller_id || "—"}
                    </strong>

                </div>


                <div class="detail-item">

                    <label>
                        Product
                    </label>

                    <strong>
                        ${lead.product_category || "—"}
                    </strong>

                </div>


                <div class="detail-item">

                    <label>
                        Inquiry value
                    </label>

                    <strong>
                        ${formatINR(lead.inquiry_value)}
                    </strong>

                </div>


                <div class="detail-item">

                    <label>
                        Quantity
                    </label>

                    <strong>
                        ${lead.quantity || "—"}
                    </strong>

                </div>


                <div class="detail-item">

                    <label>
                        Urgency
                    </label>

                    <strong>
                        ${lead.urgency || "—"}
                    </strong>

                </div>

            </div>

        </div>


        <div class="detail-section">

            <h3>
                Why prioritize?
            </h3>

            <div class="reason-list">

                ${
                    reasons.length
                        ? reasons.map(reason => `
                            <div class="reason-item">
                                ${reason}
                            </div>
                        `).join("")
                        : `
                            <div class="reason-item">
                                Lead ranked highly based on its
                                overall feature profile.
                            </div>
                        `
                }

            </div>

        </div>


        <div class="detail-section">

            <h3>
                Recommended action
            </h3>

            <div class="action-box">
                ${action}
            </div>

        </div>


        <div class="detail-section">

            <h3>
                AI Response Assistant
            </h3>

            <div class="ai-response">

                <div class="ai-label">
                    Suggested seller response
                </div>

                ${response}

            </div>

        </div>

    `;
}


/* ================= EXPLANATIONS ================= */

function generateReasons(lead) {

    const reasons = [];


    if (lead.urgency === "Critical") {

        reasons.push(
            "Critical urgency requires immediate attention."
        );

    } else if (lead.urgency === "High") {

        reasons.push(
            "High urgency indicates a time-sensitive requirement."
        );

    }


    if (Number(lead.repeat_buyer) === 1) {

        reasons.push(
            `Repeat buyer with ${Number(lead.previous_orders || 0)} previous orders.`
        );

    }


    if (
        lead.buyer_company_size === "Large" ||
        lead.buyer_company_size === "Enterprise"
    ) {

        reasons.push(
            `${lead.buyer_company_size} buyer profile.`
        );

    }


    if (Number(lead.inquiry_value) >= 20000) {

        reasons.push(
            `High inquiry value of ${formatINR(lead.inquiry_value)}.`
        );

    }


    if (Number(lead.quantity) >= 10) {

        reasons.push(
            `Large requirement of ${Number(lead.quantity)} units.`
        );

    }


    if (Number(lead.seller_rating) >= 4.5) {

        reasons.push(
            `Strong seller rating of ${Number(lead.seller_rating).toFixed(1)}.`
        );

    }


    return reasons.slice(0, 4);
}


/* ================= RECOMMENDED ACTION ================= */

function getRecommendedAction(lead) {

    if (lead.priority_tier === "High") {

        if (lead.urgency === "Critical") {
            return "Respond within 15 minutes.";
        }

        if (lead.urgency === "High") {
            return "Respond within 30 minutes.";
        }

        return "Respond within 1 hour.";
    }


    if (lead.priority_tier === "Medium") {

        return "Respond within 2 hours.";
    }


    return "Handle after higher-priority leads and use a standardized response.";
}


/* ================= AI RESPONSE ASSISTANT ================= */

function generateSmartResponse(lead) {

    const product =
        lead.product_category || "the requested product";

    const quantity =
        lead.quantity
            ? `${lead.quantity} units`
            : "the requested quantity";

    let opening =
        "Thank you for your inquiry.";


    if (lead.urgency === "Critical") {

        opening =
            "Thank you for your urgent inquiry. We are prioritizing your requirement.";

    } else if (lead.urgency === "High") {

        opening =
            "Thank you for your inquiry. We understand that your requirement is time-sensitive.";

    }


    let buyerLine = "";

    if (Number(lead.repeat_buyer) === 1) {

        buyerLine =
            " We appreciate your continued business with us.";

    }


    return `
        ${opening}${buyerLine}

        We can assist with ${product} for
        ${quantity}. We would be happy to confirm availability,
        pricing and delivery timelines based on your requirement.

        Please share any additional specifications or delivery
        requirements, and we will get back to you promptly.
    `;
}


/* ================= HELPERS ================= */

function priorityClass(priority) {

    if (priority === "High") {
        return "priority-high";
    }

    if (priority === "Medium") {
        return "priority-medium";
    }

    return "priority-low";
}


function formatINR(value) {

    const number = Number(value || 0);

    if (number >= 10000000) {
        return "₹" + (number / 10000000).toFixed(1) + " Cr";
    }

    if (number >= 100000) {
        return "₹" + (number / 100000).toFixed(1) + " L";
    }

    return "₹" + Math.round(number).toLocaleString("en-IN");
}


function formatNumber(value) {

    const number = Number(value || 0);

    return number.toFixed(1);
}


/* ================= START APP ================= */

loadCSV();
"""

js_path = project / "app.js"

js_path.write_text(js, encoding="utf-8")

print("✅ app.js created successfully!")
print(js_path)

✅ app.js created successfully!
/Users/sameekshadongre/Documents/LeadIQ/product/app.js


In [118]:
import subprocess
import webbrowser
import time
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

server = subprocess.Popen(
    ["python3", "-m", "http.server", "8000"],
    cwd=project,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

webbrowser.open("http://localhost:8000")

print("✅ LeadIQ server started")
print("🌐 http://localhost:8000")

✅ LeadIQ server started
🌐 http://localhost:8000


In [119]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

css = r"""
:root {
    --bg: #080b14;
    --panel: #101522;
    --panel-2: #151b2b;
    --border: #252d42;
    --text: #f5f7fb;
    --muted: #8f9ab0;

    --purple: #8b5cf6;
    --blue: #38bdf8;
    --green: #34d399;
    --orange: #f59e0b;
    --red: #fb7185;
}

* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    background:
        radial-gradient(circle at 80% 0%, rgba(139,92,246,.12), transparent 30%),
        radial-gradient(circle at 10% 20%, rgba(56,189,248,.07), transparent 25%),
        var(--bg);
    color: var(--text);
    line-height: 1.5;
    min-height: 100vh;
}


/* ================= NAV ================= */

.top-nav {
    height: 72px;
    background: rgba(8,11,20,.92);
    border-bottom: 1px solid var(--border);
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 48px;
    position: sticky;
    top: 0;
    z-index: 100;
    backdrop-filter: blur(14px);
}

.brand {
    display: flex;
    align-items: center;
    gap: 12px;
}

.brand-icon {
    width: 36px;
    height: 36px;
    border-radius: 10px;
    background: linear-gradient(135deg, var(--purple), var(--blue));
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    box-shadow: 0 0 20px rgba(139,92,246,.25);
}

.brand-name {
    font-size: 18px;
    font-weight: 750;
}

.brand-subtitle {
    font-size: 10px;
    color: var(--muted);
}

.top-nav nav {
    display: flex;
    gap: 7px;
}

.nav-item {
    border: 1px solid transparent;
    background: transparent;
    color: var(--muted);
    padding: 9px 16px;
    border-radius: 8px;
    cursor: pointer;
    font-size: 13px;
    font-weight: 600;
}

.nav-item:hover {
    background: var(--panel);
    color: white;
}

.nav-item.active {
    background: #1b2235;
    border-color: #303a55;
    color: white;
}


/* ================= MAIN ================= */

.app-container {
    max-width: 1450px;
    margin: auto;
    padding: 42px 48px 70px;
}

.page {
    display: none;
}

.page.active {
    display: block;
}


/* ================= HERO ================= */

.hero {
    position: relative;
    overflow: hidden;
    background:
        radial-gradient(circle at 85% 20%, rgba(139,92,246,.28), transparent 30%),
        radial-gradient(circle at 65% 100%, rgba(56,189,248,.13), transparent 28%),
        #111827;
    border: 1px solid #283149;
    border-radius: 18px;
    padding: 52px 56px;
    display: flex;
    justify-content: space-between;
    margin-bottom: 22px;
}

.eyebrow {
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 1.5px;
    color: #7f8aa3;
    margin-bottom: 11px;
}

.hero .eyebrow {
    color: #a9b4c9;
}

.hero h1 {
    font-size: 43px;
    line-height: 1.08;
    letter-spacing: -1.6px;
    margin-bottom: 17px;
}

.hero p {
    color: #aab4c5;
    max-width: 650px;
    font-size: 15px;
}

.hero-badge {
    height: fit-content;
    border: 1px solid #34405a;
    background: rgba(255,255,255,.04);
    border-radius: 999px;
    padding: 8px 13px;
    font-size: 11px;
    color: #d6dbea;
}

.status-dot {
    width: 7px;
    height: 7px;
    background: var(--green);
    border-radius: 50%;
    display: inline-block;
    margin-right: 7px;
    box-shadow: 0 0 10px rgba(52,211,153,.8);
}


/* ================= METRICS ================= */

.metrics-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 13px;
    margin-bottom: 44px;
}

.metric-card {
    background: rgba(16,21,34,.88);
    border: 1px solid var(--border);
    border-radius: 13px;
    padding: 20px;
}

.metric-card:nth-child(1) {
    border-top: 2px solid var(--blue);
}

.metric-card:nth-child(2) {
    border-top: 2px solid var(--purple);
}

.metric-card:nth-child(3) {
    border-top: 2px solid var(--green);
}

.metric-card:nth-child(4) {
    border-top: 2px solid var(--orange);
}

.metric-label {
    color: #758199;
    font-size: 9px;
    font-weight: 800;
    letter-spacing: 1px;
}

.metric-value {
    font-size: 27px;
    font-weight: 750;
    margin-top: 7px;
}

.metric-sub {
    color: #778198;
    font-size: 11px;
    margin-top: 3px;
}


/* ================= SECTIONS ================= */

.section {
    margin-bottom: 48px;
}

.section-heading {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 17px;
}

.section-heading h2 {
    font-size: 20px;
}

.section-heading p {
    color: var(--muted);
    font-size: 12px;
    margin-top: 3px;
}

.text-button {
    border: none;
    background: transparent;
    color: var(--blue);
    cursor: pointer;
    font-size: 12px;
    font-weight: 700;
}


/* ================= LEAD CARDS ================= */

.lead-cards {
    display: grid;
    gap: 9px;
}

.lead-card {
    background: var(--panel);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 17px 19px;
    display: grid;
    grid-template-columns: 1fr auto auto;
    gap: 20px;
    align-items: center;
    cursor: pointer;
    transition: .16s ease;
}

.lead-card:hover {
    border-color: #46516c;
    background: #131a2a;
    transform: translateY(-1px);
}

.lead-main strong {
    font-size: 13px;
}

.lead-meta {
    color: var(--muted);
    font-size: 11px;
    margin-top: 3px;
}

.lead-score {
    text-align: right;
}

.score-value {
    color: var(--blue);
    font-size: 18px;
    font-weight: 750;
}

.score-label {
    color: #69748b;
    font-size: 9px;
    text-transform: uppercase;
    letter-spacing: .7px;
}


/* ================= PRIORITY BADGES ================= */

.priority-badge {
    display: inline-flex;
    border-radius: 999px;
    padding: 5px 9px;
    font-size: 9px;
    font-weight: 800;
    letter-spacing: .6px;
    text-transform: uppercase;
}

.priority-high {
    background: rgba(139,92,246,.16);
    color: #b99cff;
    border: 1px solid rgba(139,92,246,.3);
}

.priority-medium {
    background: rgba(245,158,11,.13);
    color: #fbbf54;
    border: 1px solid rgba(245,158,11,.25);
}

.priority-low {
    background: rgba(100,116,139,.12);
    color: #9aa6bb;
    border: 1px solid #30394d;
}


/* ================= VALUE CARDS ================= */

.value-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 13px;
}

.value-card {
    background: var(--panel);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 23px;
}

.value-card span {
    color: var(--purple);
    font-size: 9px;
    font-weight: 800;
    letter-spacing: 1px;
}

.value-card h3 {
    font-size: 16px;
    margin: 9px 0 5px;
}

.value-card p {
    color: var(--muted);
    font-size: 12px;
}


/* ================= PAGE HEADER ================= */

.page-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-end;
    margin-bottom: 25px;
}

.page-header h1 {
    font-size: 31px;
    letter-spacing: -1px;
}

.page-header p {
    color: var(--muted);
    font-size: 13px;
}

.inbox-summary {
    color: var(--muted);
    font-size: 12px;
}

.inbox-summary span {
    color: var(--blue);
    font-weight: 750;
}


/* ================= FILTERS ================= */

.filters {
    background: var(--panel);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 17px;
    display: grid;
    grid-template-columns: 170px 170px 210px 1fr;
    gap: 12px;
    margin-bottom: 14px;
}

.filter-group {
    display: flex;
    flex-direction: column;
    gap: 5px;
}

.filter-group label {
    color: #778299;
    font-size: 9px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: .8px;
}

.filter-group select,
.filter-group input {
    height: 37px;
    background: #0b101c;
    border: 1px solid #293248;
    color: #dce2ee;
    border-radius: 7px;
    padding: 0 10px;
    outline: none;
}

.filter-group select:focus,
.filter-group input:focus {
    border-color: var(--purple);
}


/* ================= INBOX ================= */

.inbox-list {
    display: grid;
    gap: 8px;
}

.inbox-row {
    background: var(--panel);
    border: 1px solid var(--border);
    border-radius: 11px;
    padding: 16px;
    display: grid;
    grid-template-columns: 55px 1.4fr 1fr 110px 100px 25px;
    gap: 13px;
    align-items: center;
    cursor: pointer;
    transition: .15s ease;
}

.inbox-row:hover {
    border-color: #46516c;
    background: #131a2a;
}

.rank-number {
    color: #69758c;
    font-size: 11px;
}

.inbox-buyer strong,
.inbox-info strong {
    font-size: 12px;
}

.inbox-buyer span,
.inbox-info span {
    display: block;
    color: #758198;
    font-size: 10px;
    margin-top: 2px;
}

.inbox-score {
    color: var(--blue);
    font-weight: 750;
    font-size: 15px;
}

.arrow {
    color: #67748d;
}


/* ================= DRAWER ================= */

.drawer-overlay {
    position: fixed;
    inset: 0;
    background: rgba(0,0,0,.65);
    opacity: 0;
    visibility: hidden;
    transition: .2s;
    z-index: 199;
}

.drawer-overlay.open {
    opacity: 1;
    visibility: visible;
}

.lead-drawer {
    position: fixed;
    top: 0;
    right: -570px;
    width: 570px;
    max-width: 100%;
    height: 100vh;
    background: #0d121f;
    border-left: 1px solid var(--border);
    z-index: 200;
    transition: .25s ease;
    overflow-y: auto;
    padding: 30px;
}

.lead-drawer.open {
    right: 0;
}

.drawer-header {
    display: flex;
    justify-content: space-between;
    border-bottom: 1px solid var(--border);
    padding-bottom: 18px;
    margin-bottom: 20px;
}

.drawer-header h2 {
    font-size: 21px;
}

.close-button {
    width: 33px;
    height: 33px;
    border: 1px solid var(--border);
    background: var(--panel);
    color: #b7c0d0;
    border-radius: 7px;
    font-size: 21px;
    cursor: pointer;
}


/* ================= SCORE ================= */

.detail-score {
    background:
        radial-gradient(circle at 90% 10%, rgba(139,92,246,.24), transparent 35%),
        #151b2b;
    border: 1px solid #303a55;
    border-radius: 13px;
    padding: 22px;
    margin-bottom: 17px;
}

.detail-score-label {
    color: #8490a7;
    font-size: 9px;
    font-weight: 800;
    letter-spacing: 1px;
}

.detail-score-value {
    color: var(--blue);
    font-size: 39px;
    font-weight: 800;
    letter-spacing: -1px;
}

.detail-score-sub {
    color: #7f8aa0;
    font-size: 11px;
}


/* ================= DETAIL ================= */

.detail-section {
    border-bottom: 1px solid var(--border);
    padding: 18px 0;
}

.detail-section:last-child {
    border-bottom: none;
}

.detail-section h3 {
    color: #7f8aa0;
    font-size: 10px;
    text-transform: uppercase;
    letter-spacing: .8px;
    margin-bottom: 12px;
}

.detail-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 13px;
}

.detail-item label {
    display: block;
    color: #68748a;
    font-size: 9px;
    text-transform: uppercase;
}

.detail-item strong {
    display: block;
    color: #e6eaf2;
    font-size: 12px;
    margin-top: 2px;
}

.reason-list {
    display: grid;
    gap: 7px;
}

.reason-item {
    background: #131a29;
    border: 1px solid #252f45;
    border-radius: 8px;
    padding: 9px 11px;
    font-size: 11px;
    color: #c6cede;
}

.reason-item::before {
    content: "✦";
    color: var(--purple);
    margin-right: 7px;
}

.action-box {
    background: rgba(52,211,153,.08);
    border: 1px solid rgba(52,211,153,.2);
    color: #7ee8bd;
    border-radius: 9px;
    padding: 13px;
    font-size: 12px;
    font-weight: 650;
}


/* ================= AI RESPONSE ================= */

.ai-response {
    background: #111827;
    border: 1px solid #2c3650;
    border-radius: 10px;
    padding: 15px;
    color: #cdd4e1;
    font-size: 12px;
    line-height: 1.65;
}

.ai-label {
    color: var(--purple);
    font-size: 9px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: .8px;
    margin-bottom: 9px;
}


/* ================= RESPONSIVE ================= */

@media (max-width: 950px) {

    .app-container {
        padding: 28px;
    }

    .top-nav {
        padding: 0 28px;
    }

    .metrics-grid {
        grid-template-columns: 1fr 1fr;
    }

    .filters {
        grid-template-columns: 1fr 1fr;
    }

    .inbox-row {
        grid-template-columns: 45px 1fr 90px 80px 20px;
    }

    .inbox-info {
        display: none;
    }
}

@media (max-width: 600px) {

    .app-container {
        padding: 20px;
    }

    .top-nav {
        padding: 0 16px;
    }

    .brand-subtitle {
        display: none;
    }

    .hero {
        display: block;
        padding: 30px;
    }

    .hero h1 {
        font-size: 31px;
    }

    .metrics-grid {
        grid-template-columns: 1fr;
    }

    .filters {
        grid-template-columns: 1fr;
    }

    .value-grid {
        grid-template-columns: 1fr;
    }

    .lead-drawer {
        width: 100%;
        padding: 22px;
    }
}
"""

css_path = project / "styles.css"
css_path.write_text(css, encoding="utf-8")

print("✅ Dark colourful LeadIQ theme applied.")
print(css_path)

✅ Dark colourful LeadIQ theme applied.
/Users/sameekshadongre/Documents/LeadIQ/product/styles.css


In [120]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

js = r"""
let leads = [];
let filteredLeads = [];
let currentLead = null;


/* ================= CSV ================= */

async function loadCSV() {

    try {

        const response = await fetch("leadiq_seller_view.csv");

        if (!response.ok) {
            throw new Error("CSV could not be loaded");
        }

        const text = await response.text();

        leads = parseCSV(text).map(normalizeLead);

        filteredLeads = [...leads];

        initializeApp();

    } catch (error) {

        console.error(error);

        document.getElementById("lead-list").innerHTML = `
            <div class="metric-card">
                <h3>Lead data could not be loaded</h3>
                <p style="margin-top:8px;color:#8f9ab0;">
                    Make sure LeadIQ is running through
                    http://localhost:8000
                </p>
            </div>
        `;
    }
}


function parseCSV(text) {

    const rows = [];
    let row = [];
    let value = "";
    let quotes = false;

    for (let i = 0; i < text.length; i++) {

        const char = text[i];
        const next = text[i + 1];

        if (char === '"' && quotes && next === '"') {
            value += '"';
            i++;
        }

        else if (char === '"') {
            quotes = !quotes;
        }

        else if (char === "," && !quotes) {
            row.push(value);
            value = "";
        }

        else if ((char === "\n" || char === "\r") && !quotes) {

            if (char === "\r" && next === "\n") {
                i++;
            }

            row.push(value);

            if (row.some(x => x.trim() !== "")) {
                rows.push(row);
            }

            row = [];
            value = "";
        }

        else {
            value += char;
        }
    }

    if (value !== "" || row.length > 0) {
        row.push(value);

        if (row.some(x => x.trim() !== "")) {
            rows.push(row);
        }
    }

    if (!rows.length) return [];

    const headers = rows[0].map(x => x.trim());

    return rows.slice(1).map(row => {

        const obj = {};

        headers.forEach((header, index) => {
            obj[header] = (row[index] || "").trim();
        });

        return obj;
    });
}


function normalizeLead(row) {

    const numericFields = [
        "buyer_id",
        "seller_id",
        "priority_score",
        "conversion_probability",
        "rank",
        "inquiry_value",
        "quantity",
        "previous_orders",
        "previous_inquiries",
        "repeat_buyer",
        "seller_rating",
        "seller_experience"
    ];

    numericFields.forEach(field => {

        if (row[field] !== undefined && row[field] !== "") {
            row[field] = Number(row[field]);
        }
    });

    return row;
}


/* ================= APP ================= */

function initializeApp() {

    setupNavigation();
    setupFilters();
    setupDrawer();

    renderOverview();
    renderInbox();
}


/* ================= NAVIGATION ================= */

function setupNavigation() {

    document.querySelectorAll(".nav-item").forEach(button => {

        button.addEventListener("click", () => {
            showPage(button.dataset.page);
        });

    });


    document
        .getElementById("open-inbox")
        .addEventListener("click", () => {

            showPage("inbox");

        });
}


function showPage(page) {

    document.querySelectorAll(".page").forEach(pageElement => {
        pageElement.classList.remove("active");
    });

    document.querySelectorAll(".nav-item").forEach(button => {
        button.classList.remove("active");
    });


    if (page === "home") {

        document
            .getElementById("home-page")
            .classList.add("active");

        document
            .querySelector('[data-page="home"]')
            .classList.add("active");

    } else {

        document
            .getElementById("inbox-page")
            .classList.add("active");

        document
            .querySelector('[data-page="inbox"]')
            .classList.add("active");

    }
}


/* ================= OVERVIEW ================= */

function renderOverview() {

    const highLeads = leads.filter(
        lead => lead.priority_tier === "High"
    );

    const highValue = highLeads.reduce(
        (sum, lead) => sum + Number(lead.inquiry_value || 0),
        0
    );


    document.getElementById("total-leads").textContent =
        leads.length.toLocaleString();


    document.getElementById("high-leads").textContent =
        highLeads.length.toLocaleString();


    document.getElementById("high-value").textContent =
        formatINR(highValue);


    renderTopLeads();
}


function renderTopLeads() {

    const topLeads = [...leads]
        .sort((a, b) => Number(a.rank) - Number(b.rank))
        .slice(0, 5);


    document.getElementById("top-leads-container").innerHTML = `
        <div class="lead-cards">
            ${topLeads.map((lead, index) =>
                createTopLeadCard(lead, index)
            ).join("")}
        </div>
    `;


    document.querySelectorAll(".lead-card").forEach(card => {

        card.addEventListener("click", () => {

            const index = Number(card.dataset.index);

            openLeadDrawer(topLeads[index]);

        });

    });
}


function createTopLeadCard(lead, index) {

    return `
        <div class="lead-card" data-index="${index}">

            <div class="lead-main">

                <strong>
                    Buyer ${lead.buyer_id || "Unknown"}
                </strong>

                <div class="lead-meta">
                    ${lead.product_category || "Product inquiry"}
                    · ${lead.buyer_industry || "Industry not available"}
                    · ${lead.buyer_location || "Location not available"}
                </div>

            </div>


            <div>

                <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                    ${lead.priority_tier || "—"}
                </span>

            </div>


            <div class="lead-score">

                <div class="score-value">
                    ${formatNumber(lead.priority_score)}%
                </div>

                <div class="score-label">
                    AI score
                </div>

            </div>

        </div>
    `;
}


/* ================= FILTERS ================= */

function setupFilters() {

    const urgencyFilter =
        document.getElementById("urgency-filter");

    const categoryFilter =
        document.getElementById("category-filter");


    uniqueValues("urgency").forEach(value => {

        urgencyFilter.insertAdjacentHTML(
            "beforeend",
            `<option value="${escapeHTML(value)}">${escapeHTML(value)}</option>`
        );

    });


    uniqueValues("product_category").forEach(value => {

        categoryFilter.insertAdjacentHTML(
            "beforeend",
            `<option value="${escapeHTML(value)}">${escapeHTML(value)}</option>`
        );

    });


    document
        .getElementById("priority-filter")
        .addEventListener("change", applyFilters);

    urgencyFilter
        .addEventListener("change", applyFilters);

    categoryFilter
        .addEventListener("change", applyFilters);

    document
        .getElementById("search-input")
        .addEventListener("input", applyFilters);
}


function uniqueValues(field) {

    return [
        ...new Set(
            leads
                .map(lead => lead[field])
                .filter(value => value !== undefined && value !== "")
        )
    ].sort();
}


function applyFilters() {

    const priority =
        document.getElementById("priority-filter").value;

    const urgency =
        document.getElementById("urgency-filter").value;

    const category =
        document.getElementById("category-filter").value;

    const search =
        document
            .getElementById("search-input")
            .value
            .toLowerCase()
            .trim();


    filteredLeads = leads.filter(lead => {

        const priorityMatch =
            priority === "All" ||
            lead.priority_tier === priority;

        const urgencyMatch =
            urgency === "All" ||
            lead.urgency === urgency;

        const categoryMatch =
            category === "All" ||
            lead.product_category === category;

        const searchable = [
            lead.buyer_id,
            lead.seller_id,
            lead.product_category,
            lead.buyer_industry,
            lead.buyer_location
        ]
        .join(" ")
        .toLowerCase();

        const searchMatch =
            !search ||
            searchable.includes(search);

        return (
            priorityMatch &&
            urgencyMatch &&
            categoryMatch &&
            searchMatch
        );
    });


    renderInbox();
}


/* ================= INBOX ================= */

function renderInbox() {

    document.getElementById("filtered-count").textContent =
        filteredLeads.length.toLocaleString();


    const container =
        document.getElementById("lead-list");


    if (!filteredLeads.length) {

        container.innerHTML = `
            <div class="metric-card">
                <h3>No leads found</h3>
                <p style="margin-top:8px;color:#8f9ab0;">
                    Try changing the filters.
                </p>
            </div>
        `;

        return;
    }


    const sorted = [...filteredLeads]
        .sort((a, b) => Number(a.rank) - Number(b.rank));


    container.innerHTML = `
        <div class="inbox-list">

            ${sorted.map((lead, index) =>
                createInboxRow(lead, index)
            ).join("")}

        </div>
    `;


    container
        .querySelectorAll(".inbox-row")
        .forEach((row, index) => {

            row.addEventListener("click", () => {
                openLeadDrawer(sorted[index]);
            });

        });
}


function createInboxRow(lead, index) {

    return `
        <div class="inbox-row">

            <div class="rank-number">
                #${lead.rank || index + 1}
            </div>


            <div class="inbox-buyer">

                <strong>
                    Buyer ${lead.buyer_id || "Unknown"}
                </strong>

                <span>
                    Seller ${lead.seller_id || "Unknown"}
                </span>

            </div>


            <div class="inbox-info">

                <strong>
                    ${lead.product_category || "—"}
                </strong>

                <span>
                    ${lead.buyer_industry || "Industry unavailable"}
                </span>

            </div>


            <div>

                <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                    ${lead.priority_tier || "—"}
                </span>

            </div>


            <div class="inbox-score">
                ${formatNumber(lead.priority_score)}%
            </div>


            <div class="arrow">
                →
            </div>

        </div>
    `;
}


/* ================= DRAWER ================= */

function setupDrawer() {

    document
        .getElementById("close-drawer")
        .addEventListener("click", closeDrawer);

    document
        .getElementById("drawer-overlay")
        .addEventListener("click", closeDrawer);


    document.addEventListener("keydown", event => {

        if (event.key === "Escape") {
            closeDrawer();
        }

    });
}


function openLeadDrawer(lead) {

    currentLead = lead;

    document.getElementById("drawer-title").textContent =
        `Buyer ${lead.buyer_id || "Unknown"}`;

    document.getElementById("drawer-content").innerHTML =
        createLeadDetail(lead);


    document
        .getElementById("lead-drawer")
        .classList.add("open");

    document
        .getElementById("drawer-overlay")
        .classList.add("open");


    const generateButton =
        document.getElementById("generate-response");


    if (generateButton) {

        generateButton.addEventListener(
            "click",
            () => generateResponseForLead(lead)
        );

    }


    const copyButton =
        document.getElementById("copy-response");


    if (copyButton) {

        copyButton.addEventListener(
            "click",
            copyGeneratedResponse
        );

    }
}


function closeDrawer() {

    document
        .getElementById("lead-drawer")
        .classList.remove("open");

    document
        .getElementById("drawer-overlay")
        .classList.remove("open");
}


/* ================= DETAIL VIEW ================= */

function createLeadDetail(lead) {

    const reasons = generateReasons(lead);

    const action = getRecommendedAction(lead);


    return `

        <div class="detail-score">

            <div class="detail-score-label">
                AI CONVERSION POTENTIAL
            </div>

            <div class="detail-score-value">
                ${formatNumber(lead.priority_score)}%
            </div>

            <div class="detail-score-sub">
                Predicted conversion probability
            </div>

            <div style="margin-top:10px;">

                <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                    ${lead.priority_tier} PRIORITY
                </span>

            </div>

        </div>


        <div class="detail-section">

            <h3>
                Buyer & Requirement
            </h3>

            <div class="detail-grid">

                ${detailItem("Product", lead.product_category)}

                ${detailItem("Industry", lead.buyer_industry)}

                ${detailItem("Location", lead.buyer_location)}

                ${detailItem("Company Size", lead.buyer_company_size)}

                ${detailItem("Quantity", lead.quantity ? `${lead.quantity} units` : "—")}

                ${detailItem("Inquiry Value", formatINR(lead.inquiry_value))}

                ${detailItem("Urgency", lead.urgency)}

                ${detailItem("Lead Source", lead.lead_source)}

            </div>

        </div>


        <div class="detail-section">

            <h3>
                AI Score & Seller Signals
            </h3>

            <div class="detail-grid">

                ${detailItem(
                    "Conversion Probability",
                    `${formatNumber(lead.conversion_probability * 100)}%`
                )}

                ${detailItem(
                    "Previous Orders",
                    lead.previous_orders
                )}

                ${detailItem(
                    "Previous Inquiries",
                    lead.previous_inquiries
                )}

                ${detailItem(
                    "Repeat Buyer",
                    Number(lead.repeat_buyer) === 1 ? "Yes" : "No"
                )}

                ${detailItem(
                    "Seller Rating",
                    lead.seller_rating
                        ? Number(lead.seller_rating).toFixed(1) + " / 5"
                        : "—"
                )}

                ${detailItem(
                    "Seller Experience",
                    lead.seller_experience
                        ? lead.seller_experience + " years"
                        : "—"
                )}

            </div>

        </div>


        <div class="detail-section">

            <h3>
                Why prioritize?
            </h3>

            <div class="reason-list">

                ${
                    reasons.length
                    ? reasons.map(reason => `
                        <div class="reason-item">
                            ${reason}
                        </div>
                    `).join("")
                    : `
                        <div class="reason-item">
                            Strong overall lead profile.
                        </div>
                    `
                }

            </div>

        </div>


        <div class="detail-section">

            <h3>
                Recommended action
            </h3>

            <div class="action-box">
                ${action}
            </div>

        </div>


        <div class="detail-section">

            <h3>
                ✦ AI Response Assistant
            </h3>

            <button
                id="generate-response"
                style="
                    width:100%;
                    border:none;
                    border-radius:8px;
                    padding:11px;
                    background:linear-gradient(135deg,#8b5cf6,#6366f1);
                    color:white;
                    font-weight:700;
                    cursor:pointer;
                    margin-bottom:10px;
                "
            >
                Generate AI Response
            </button>


            <div
                id="response-container"
                style="display:none;"
            >

                <div class="ai-response">

                    <div class="ai-label">
                        Suggested seller response
                    </div>

                    <div id="generated-response"></div>

                </div>


                <button
                    id="copy-response"
                    style="
                        margin-top:9px;
                        border:1px solid #303a55;
                        background:#151b2b;
                        color:#cdd4e1;
                        border-radius:7px;
                        padding:8px 12px;
                        cursor:pointer;
                        font-size:11px;
                    "
                >
                    Copy Response
                </button>

            </div>

        </div>

    `;
}


function detailItem(label, value) {

    return `
        <div class="detail-item">

            <label>
                ${label}
            </label>

            <strong>
                ${value !== undefined &&
                  value !== null &&
                  value !== ""
                  ? escapeHTML(String(value))
                  : "—"}
            </strong>

        </div>
    `;
}


/* ================= REASONS ================= */

function generateReasons(lead) {

    const reasons = [];


    if (lead.urgency === "Critical") {

        reasons.push(
            "Critical urgency — immediate response recommended."
        );

    } else if (lead.urgency === "High") {

        reasons.push(
            "High urgency — the buyer requirement is time-sensitive."
        );

    }


    if (Number(lead.repeat_buyer) === 1) {

        reasons.push(
            `Repeat buyer with ${lead.previous_orders || 0} previous orders.`
        );

    }


    if (
        lead.buyer_company_size === "Large" ||
        lead.buyer_company_size === "Enterprise"
    ) {

        reasons.push(
            `${lead.buyer_company_size} buyer profile.`
        );

    }


    if (Number(lead.inquiry_value) >= 20000) {

        reasons.push(
            `High inquiry value — ${formatINR(lead.inquiry_value)}.`
        );

    }


    if (Number(lead.quantity) >= 10) {

        reasons.push(
            `Large requirement — ${lead.quantity} units.`
        );

    }


    if (Number(lead.seller_rating) >= 4.5) {

        reasons.push(
            `Strong seller rating — ${Number(lead.seller_rating).toFixed(1)}/5.`
        );

    }


    return reasons.slice(0, 4);
}


/* ================= ACTION ================= */

function getRecommendedAction(lead) {

    if (lead.priority_tier === "High") {

        if (lead.urgency === "Critical") {
            return "Respond within 15 minutes and confirm availability, pricing and delivery.";
        }

        if (lead.urgency === "High") {
            return "Respond within 30 minutes and qualify the requirement.";
        }

        return "Respond within 1 hour and move the inquiry toward quotation.";
    }


    if (lead.priority_tier === "Medium") {

        return "Respond within 2 hours and qualify the buyer's requirement.";
    }


    return "Handle after higher-priority leads using a standardized response.";
}


/* ================= AI RESPONSE ================= */

function generateResponseForLead(lead) {

    const container =
        document.getElementById("response-container");

    const output =
        document.getElementById("generated-response");


    const product =
        lead.product_category || "the requested product";

    const quantity =
        lead.quantity
            ? `${lead.quantity} units`
            : "the requested quantity";

    const value =
        lead.inquiry_value
            ? formatINR(lead.inquiry_value)
            : "your requirement";


    let opening;

    if (lead.urgency === "Critical") {

        opening =
            "Thank you for your urgent inquiry. We are prioritizing your requirement.";

    } else if (lead.urgency === "High") {

        opening =
            "Thank you for your inquiry. We understand that your requirement is time-sensitive.";

    } else {

        opening =
            "Thank you for reaching out regarding your requirement.";

    }


    const repeatLine =
        Number(lead.repeat_buyer) === 1
            ? " We also appreciate your continued business with us."
            : "";


    const response = `
        ${opening}${repeatLine}

        We can assist with ${product} for ${quantity},
        with an estimated inquiry value of ${value}.

        We would be happy to confirm availability, pricing and
        delivery timelines based on your requirement.

        Please share any additional specifications or delivery
        requirements, and we will get back to you promptly.
    `.trim();


    output.textContent = response;

    container.style.display = "block";
}


async function copyGeneratedResponse() {

    const text =
        document.getElementById("generated-response").textContent;

    try {

        await navigator.clipboard.writeText(text);

        document.getElementById("copy-response").textContent =
            "✓ Copied";

        setTimeout(() => {

            document.getElementById("copy-response").textContent =
                "Copy Response";

        }, 1500);

    } catch (error) {

        console.error(error);

    }
}


/* ================= HELPERS ================= */

function priorityClass(priority) {

    if (priority === "High") {
        return "priority-high";
    }

    if (priority === "Medium") {
        return "priority-medium";
    }

    return "priority-low";
}


function formatINR(value) {

    const number = Number(value || 0);

    if (number >= 10000000) {
        return "₹" + (number / 10000000).toFixed(1) + " Cr";
    }

    if (number >= 100000) {
        return "₹" + (number / 100000).toFixed(1) + " L";
    }

    return "₹" + Math.round(number).toLocaleString("en-IN");
}


function formatNumber(value) {

    return Number(value || 0).toFixed(1);
}


function escapeHTML(value) {

    return String(value)
        .replace(/&/g, "&amp;")
        .replace(/</g, "&lt;")
        .replace(/>/g, "&gt;")
        .replace(/"/g, "&quot;")
        .replace(/'/g, "&#039;");
}


/* ================= START ================= */

loadCSV();
"""

js_path = project / "app.js"
js_path.write_text(js, encoding="utf-8")

print("✅ app.js upgraded successfully.")
print(js_path)

✅ app.js upgraded successfully.
/Users/sameekshadongre/Documents/LeadIQ/product/app.js


In [121]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

html_path = project / "index.html"

html = html_path.read_text(encoding="utf-8")

old = """<div class="metric-card">

                <div class="metric-label">
                    AI PURPOSE
                </div>

                <div class="metric-value">
                    Prioritize
                </div>

                <div class="metric-sub">
                    Focus seller attention first
                </div>

            </div>"""

new = """<div class="metric-card">

                <div class="metric-label">
                    TOP-DECILE LIFT
                </div>

                <div class="metric-value">
                    1.78×
                </div>

                <div class="metric-sub">
                    Conversion rate vs. overall leads
                </div>

            </div>"""

if old in html:
    html = html.replace(old, new)
    html_path.write_text(html, encoding="utf-8")
    print("✅ Model performance metric added.")
else:
    print("⚠️ The expected section was not found. No changes made.")

✅ Model performance metric added.


In [122]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
js_path = project / "app.js"

js = js_path.read_text(encoding="utf-8")

start = js.index("function createInboxRow")
end = js.index("\n\n\n/* ================= DRAWER ================= */", start)

new_function = r'''function createInboxRow(lead, index) {

    const urgencyClass =
        lead.urgency === "Critical"
            ? "urgency-critical"
            : lead.urgency === "High"
                ? "urgency-high"
                : "urgency-normal";

    return `
        <div class="inbox-row">

            <div class="rank-number">
                #${lead.rank || index + 1}
            </div>


            <div class="inbox-buyer">

                <strong>
                    Buyer ${lead.buyer_id || "Unknown"}
                </strong>

                <span>
                    ${lead.buyer_location || "Location unavailable"}
                </span>

            </div>


            <div class="inbox-info">

                <strong>
                    ${lead.product_category || "—"}
                </strong>

                <span>
                    ${lead.buyer_industry || "Industry unavailable"}
                </span>

            </div>


            <div class="inbox-info">

                <strong>
                    ${lead.quantity || "—"} units
                </strong>

                <span>
                    ${formatINR(lead.inquiry_value)}
                </span>

            </div>


            <div>

                <span class="priority-badge ${urgencyClass}">
                    ${lead.urgency || "Normal"}
                </span>

            </div>


            <div class="inbox-score">

                ${formatNumber(lead.priority_score)}%

                <span style="
                    display:block;
                    color:#69748b;
                    font-size:9px;
                    font-weight:500;
                    margin-top:1px;
                ">
                    AI score
                </span>

            </div>


            <div>

                <span class="priority-badge ${priorityClass(lead.priority_tier)}">
                    ${lead.priority_tier || "—"}
                </span>

            </div>


            <div class="arrow">
                →
            </div>

        </div>
    `;
}'''

js = js[:start] + new_function + js[end:]

js_path.write_text(js, encoding="utf-8")

print("✅ Priority Inbox upgraded.")

✅ Priority Inbox upgraded.


In [123]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
css_path = project / "styles.css"

css = css_path.read_text(encoding="utf-8")

css += r"""

/* ================= URGENCY COLORS ================= */

.urgency-critical {
    background: rgba(251,113,133,.13);
    color: #fb7185;
    border: 1px solid rgba(251,113,133,.28);
}

.urgency-high {
    background: rgba(245,158,11,.13);
    color: #fbbf54;
    border: 1px solid rgba(245,158,11,.25);
}

.urgency-normal {
    background: rgba(56,189,248,.10);
    color: #67d3ff;
    border: 1px solid rgba(56,189,248,.2);
}


/* Wider inbox */

.inbox-row {
    grid-template-columns:
        45px
        1.15fr
        1.2fr
        105px
        100px
        90px
        85px
        22px;
}

.inbox-info strong {
    color: #e3e8f2;
}
"""

css_path.write_text(css, encoding="utf-8")

print("✅ Inbox styling updated.")

✅ Inbox styling updated.


In [124]:
print(df_new.columns.tolist())


['timestamp', 'buyer_id', 'seller_id', 'buyer_industry', 'buyer_location', 'buyer_company_size', 'product_category', 'urgency', 'lead_source', 'inquiry_hour', 'seller_rating', 'seller_experience', 'previous_inquiries', 'repeat_buyer', 'buyer_propensity', 'previous_orders', 'quantity', 'inquiry_value', 'buyer_seller_previous_inquiries', 'buyer_seller_previous_orders', 'seller_quality', 'converted', 'previous_conversion_rate']


In [125]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"

# Start from the already-scored LeadIQ output
complete_view = leadiq_output.copy()

# AI score should be displayed as 0–100, e.g. 86.4
complete_view["ai_score"] = (
    complete_view["conversion_probability"] * 100
).round(1)

# Make sure the priority rank is integer
complete_view["rank"] = complete_view["rank"].astype(int)

# Columns needed by the product
product_columns = [
    "rank",
    "buyer_id",
    "seller_id",

    "buyer_industry",
    "buyer_location",
    "buyer_company_size",
    "product_category",

    "quantity",
    "inquiry_value",
    "urgency",
    "lead_source",

    "previous_orders",
    "previous_inquiries",
    "repeat_buyer",
    "previous_conversion_rate",

    "buyer_seller_previous_inquiries",
    "buyer_seller_previous_orders",

    "seller_rating",
    "seller_experience",

    "conversion_probability",
    "ai_score",
    "priority_score",
    "priority_tier",

    "why_prioritize",
    "recommended_action"
]

# Keep only columns that actually exist
available_columns = [
    col for col in product_columns
    if col in complete_view.columns
]

seller_view = (
    complete_view[available_columns]
    .sort_values("rank")
    .reset_index(drop=True)
)

# Save the complete product dataset
output_path = project / "leadiq_seller_view.csv"

seller_view.to_csv(
    output_path,
    index=False
)

print("✅ Complete LeadIQ seller dataset created")
print(output_path)

print("\nShape:", seller_view.shape)

print("\nColumns:")
print(seller_view.columns.tolist())

print("\nSample:")
display(seller_view.head(3))

✅ Complete LeadIQ seller dataset created
/Users/sameekshadongre/Documents/LeadIQ/product/leadiq_seller_view.csv

Shape: (3000, 25)

Columns:
['rank', 'buyer_id', 'seller_id', 'buyer_industry', 'buyer_location', 'buyer_company_size', 'product_category', 'quantity', 'inquiry_value', 'urgency', 'lead_source', 'previous_orders', 'previous_inquiries', 'repeat_buyer', 'previous_conversion_rate', 'buyer_seller_previous_inquiries', 'buyer_seller_previous_orders', 'seller_rating', 'seller_experience', 'conversion_probability', 'ai_score', 'priority_score', 'priority_tier', 'why_prioritize', 'recommended_action']

Sample:


,rank,buyer_id,seller_id,buyer_industry,buyer_location,buyer_company_size,product_category,quantity,inquiry_value,urgency,...,buyer_seller_previous_inquiries,buyer_seller_previous_orders,seller_rating,seller_experience,conversion_probability,ai_score,priority_score,priority_tier,why_prioritize,recommended_action
0,1,2077,606,Manufacturing,Mumbai,Large,Chemicals,11,24577.57,Critical,...,0,0,4.47,10.0,0.446206,44.6,44.6,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes
1,2,1697,961,Retail,Chennai,Large,Chemicals,10,13717.86,Critical,...,0,0,4.55,8.8,0.439382,43.9,43.9,High,"[Critical urgency, Repeat buyer with 4 previou...",Respond within 15 minutes
2,3,1726,480,Automotive,Mumbai,Large,Food Ingredients,15,16227.37,Critical,...,0,0,4.70,11.2,0.434851,43.5,43.5,High,"[Critical urgency, Repeat buyer with 2 previou...",Respond within 15 minutes


In [126]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
js_path = project / "app.js"

js = r"""
let leads = [];
let filteredLeads = [];
let currentLead = null;


/* ================= DATA ================= */

async function loadCSV() {

    try {

        const response = await fetch("leadiq_seller_view.csv");

        if (!response.ok) {
            throw new Error("CSV could not be loaded");
        }

        const text = await response.text();

        leads = parseCSV(text).map(normalizeLead);

        filteredLeads = [...leads];

        initializeApp();

    } catch (error) {

        console.error(error);

        document.getElementById("lead-list").innerHTML = `
            <div class="metric-card">
                <h3>Unable to load lead data</h3>
                <p style="margin-top:8px;color:#8f9ab0;">
                    Make sure LeadIQ is running at
                    http://localhost:8000
                </p>
            </div>
        `;
    }
}


function parseCSV(text) {

    const rows = [];
    let row = [];
    let value = "";
    let insideQuotes = false;

    for (let i = 0; i < text.length; i++) {

        const char = text[i];
        const next = text[i + 1];

        if (char === '"' && insideQuotes && next === '"') {
            value += '"';
            i++;
        }

        else if (char === '"') {
            insideQuotes = !insideQuotes;
        }

        else if (char === "," && !insideQuotes) {
            row.push(value);
            value = "";
        }

        else if (
            (char === "\n" || char === "\r")
            && !insideQuotes
        ) {

            if (char === "\r" && next === "\n") {
                i++;
            }

            row.push(value);

            if (row.some(x => x.trim() !== "")) {
                rows.push(row);
            }

            row = [];
            value = "";

        }

        else {
            value += char;
        }
    }


    if (value !== "" || row.length > 0) {

        row.push(value);

        if (row.some(x => x.trim() !== "")) {
            rows.push(row);
        }
    }


    if (!rows.length) {
        return [];
    }


    const headers = rows[0].map(
        header => header.trim()
    );


    return rows.slice(1).map(row => {

        const object = {};

        headers.forEach((header, index) => {

            object[header] =
                (row[index] || "").trim();

        });

        return object;
    });
}


function normalizeLead(row) {

    const numericFields = [

        "rank",
        "buyer_id",
        "seller_id",

        "quantity",
        "inquiry_value",

        "previous_orders",
        "previous_inquiries",

        "repeat_buyer",
        "previous_conversion_rate",

        "buyer_seller_previous_inquiries",
        "buyer_seller_previous_orders",

        "seller_rating",
        "seller_experience",

        "conversion_probability",
        "ai_score",
        "priority_score"

    ];


    numericFields.forEach(field => {

        if (
            row[field] !== undefined &&
            row[field] !== ""
        ) {

            row[field] = Number(row[field]);

        }

    });


    return row;
}


/* ================= INITIALIZE ================= */

function initializeApp() {

    setupNavigation();

    setupFilters();

    setupDrawer();

    renderOverview();

    renderInbox();

}


/* ================= NAVIGATION ================= */

function setupNavigation() {

    document
        .querySelectorAll(".nav-item")
        .forEach(button => {

            button.addEventListener(
                "click",
                () => showPage(button.dataset.page)
            );

        });


    document
        .getElementById("open-inbox")
        .addEventListener(
            "click",
            () => showPage("inbox")
        );

}


function showPage(page) {

    document
        .querySelectorAll(".page")
        .forEach(section =>
            section.classList.remove("active")
        );


    document
        .querySelectorAll(".nav-item")
        .forEach(button =>
            button.classList.remove("active")
        );


    const targetPage =
        page === "home"
            ? "home-page"
            : "inbox-page";


    document
        .getElementById(targetPage)
        .classList.add("active");


    document
        .querySelector(
            `[data-page="${page}"]`
        )
        .classList.add("active");

}


/* ================= OVERVIEW ================= */

function renderOverview() {

    const highPriority =
        leads.filter(
            lead => lead.priority_tier === "High"
        );


    const highValue =
        highPriority.reduce(
            (sum, lead) =>
                sum + Number(lead.inquiry_value || 0),
            0
        );


    document
        .getElementById("total-leads")
        .textContent =
            leads.length.toLocaleString();


    document
        .getElementById("high-leads")
        .textContent =
            highPriority.length.toLocaleString();


    document
        .getElementById("high-value")
        .textContent =
            formatINR(highValue);


    renderTopLeads();

}


/* ================= TOP LEADS ================= */

function renderTopLeads() {

    const topLeads =
        [...leads]
        .sort(
            (a, b) =>
                Number(a.rank) - Number(b.rank)
        )
        .slice(0, 5);


    document
        .getElementById("top-leads-container")
        .innerHTML = `

            <div class="lead-cards">

                ${
                    topLeads
                    .map(
                        (lead, index) =>
                            createTopLeadCard(
                                lead,
                                index
                            )
                    )
                    .join("")
                }

            </div>
        `;


    document
        .querySelectorAll(".lead-card")
        .forEach(card => {

            card.addEventListener(
                "click",
                () => {

                    const index =
                        Number(
                            card.dataset.index
                        );

                    openLeadDrawer(
                        topLeads[index]
                    );

                }
            );

        });

}


function createTopLeadCard(
    lead,
    index
) {

    return `

        <div
            class="lead-card"
            data-index="${index}"
        >

            <div class="lead-main">

                <strong>
                    Buyer ${lead.buyer_id}
                </strong>

                <div class="lead-meta">

                    ${lead.product_category}
                    ·
                    ${lead.buyer_industry}
                    ·
                    ${lead.buyer_location}

                </div>

            </div>


            <div>

                <span
                    class="priority-badge
                    ${priorityClass(lead.priority_tier)}"
                >

                    ${lead.priority_tier}

                </span>

            </div>


            <div class="lead-score">

                <div class="score-value">

                    ${formatScore(lead.ai_score)}

                </div>

                <div class="score-label">
                    AI SCORE
                </div>

            </div>

        </div>
    `;
}


/* ================= FILTERS ================= */

function setupFilters() {

    const urgencyFilter =
        document.getElementById(
            "urgency-filter"
        );


    const categoryFilter =
        document.getElementById(
            "category-filter"
        );


    uniqueValues("urgency")
        .forEach(value => {

            urgencyFilter.insertAdjacentHTML(
                "beforeend",
                `
                <option value="${escapeHTML(value)}">
                    ${escapeHTML(value)}
                </option>
                `
            );

        });


    uniqueValues("product_category")
        .forEach(value => {

            categoryFilter.insertAdjacentHTML(
                "beforeend",
                `
                <option value="${escapeHTML(value)}">
                    ${escapeHTML(value)}
                </option>
                `
            );

        });


    document
        .getElementById("priority-filter")
        .addEventListener(
            "change",
            applyFilters
        );


    urgencyFilter
        .addEventListener(
            "change",
            applyFilters
        );


    categoryFilter
        .addEventListener(
            "change",
            applyFilters
        );


    document
        .getElementById("search-input")
        .addEventListener(
            "input",
            applyFilters
        );

}


function uniqueValues(field) {

    return [
        ...new Set(
            leads
                .map(lead => lead[field])
                .filter(
                    value =>
                        value !== undefined &&
                        value !== ""
                )
        )
    ].sort();

}


/* ================= FILTER LOGIC ================= */

function applyFilters() {

    const priority =
        document
        .getElementById("priority-filter")
        .value;


    const urgency =
        document
        .getElementById("urgency-filter")
        .value;


    const category =
        document
        .getElementById("category-filter")
        .value;


    const search =
        document
        .getElementById("search-input")
        .value
        .toLowerCase()
        .trim();


    filteredLeads =
        leads.filter(lead => {

            const priorityMatch =
                priority === "All" ||
                lead.priority_tier === priority;


            const urgencyMatch =
                urgency === "All" ||
                lead.urgency === urgency;


            const categoryMatch =
                category === "All" ||
                lead.product_category === category;


            const searchable = [

                lead.buyer_id,
                lead.seller_id,
                lead.product_category,
                lead.buyer_industry,
                lead.buyer_location,
                lead.buyer_company_size

            ]
            .join(" ")
            .toLowerCase();


            const searchMatch =
                search === "" ||
                searchable.includes(search);


            return (
                priorityMatch &&
                urgencyMatch &&
                categoryMatch &&
                searchMatch
            );

        });


    renderInbox();

}


/* ================= PRIORITY INBOX ================= */

function renderInbox() {

    const container =
        document.getElementById(
            "lead-list"
        );


    document
        .getElementById(
            "filtered-count"
        )
        .textContent =
            filteredLeads.length.toLocaleString();


    if (!filteredLeads.length) {

        container.innerHTML = `

            <div class="metric-card">

                <h3>
                    No leads found
                </h3>

                <p style="margin-top:8px;color:#8f9ab0;">
                    Try changing your filters.
                </p>

            </div>

        `;

        return;
    }


    const sorted =
        [...filteredLeads]
        .sort(
            (a, b) =>
                Number(a.rank) -
                Number(b.rank)
        );


    container.innerHTML = `

        <div class="inbox-list">

            ${sorted
                .map(
                    (lead, index) =>
                        createInboxRow(
                            lead,
                            index
                        )
                )
                .join("")}

        </div>
    `;


    container
        .querySelectorAll(".inbox-row")
        .forEach((row, index) => {

            row.addEventListener(
                "click",
                () =>
                    openLeadDrawer(
                        sorted[index]
                    )
            );

        });

}


/* ================= INBOX ROW ================= */

function createInboxRow(
    lead,
    index
) {

    return `

        <div class="inbox-row">

            <div class="rank-number">
                #${lead.rank}
            </div>


            <div class="inbox-buyer">

                <strong>
                    Buyer ${lead.buyer_id}
                </strong>

                <span>
                    ${lead.buyer_location}
                </span>

            </div>


            <div class="inbox-info">

                <strong>
                    ${lead.product_category}
                </strong>

                <span>
                    ${lead.buyer_industry}
                </span>

            </div>


            <div class="inbox-info">

                <strong>
                    ${lead.quantity} units
                </strong>

                <span>
                    ${formatINR(
                        lead.inquiry_value
                    )}
                </span>

            </div>


            <div>

                <span
                    class="priority-badge
                    ${urgencyClass(lead.urgency)}"
                >

                    ${lead.urgency}

                </span>

            </div>


            <div class="inbox-score">

                ${formatScore(lead.ai_score)}

                <span style="
                    display:block;
                    color:#69748b;
                    font-size:9px;
                    font-weight:500;
                ">
                    AI SCORE
                </span>

            </div>


            <div>

                <span
                    class="priority-badge
                    ${priorityClass(lead.priority_tier)}"
                >

                    ${lead.priority_tier}

                </span>

            </div>


            <div class="arrow">
                →
            </div>

        </div>

    `;

}


/* ================= DRAWER ================= */

function setupDrawer() {

    document
        .getElementById(
            "close-drawer"
        )
        .addEventListener(
            "click",
            closeDrawer
        );


    document
        .getElementById(
            "drawer-overlay"
        )
        .addEventListener(
            "click",
            closeDrawer
        );


    document.addEventListener(
        "keydown",
        event => {

            if (event.key === "Escape") {
                closeDrawer();
            }

        }
    );

}


function openLeadDrawer(lead) {

    currentLead = lead;


    document
        .getElementById(
            "drawer-title"
        )
        .textContent =
            `Buyer ${lead.buyer_id}`;


    document
        .getElementById(
            "drawer-content"
        )
        .innerHTML =
            createLeadDetail(lead);


    document
        .getElementById(
            "lead-drawer"
        )
        .classList.add("open");


    document
        .getElementById(
            "drawer-overlay"
        )
        .classList.add("open");


    document
        .getElementById(
            "generate-response"
        )
        .addEventListener(
            "click",
            () =>
                generateResponseForLead(
                    lead
                )
        );


    document
        .getElementById(
            "copy-response"
        )
        .addEventListener(
            "click",
            copyGeneratedResponse
        );

}


function closeDrawer() {

    document
        .getElementById(
            "lead-drawer"
        )
        .classList.remove("open");


    document
        .getElementById(
            "drawer-overlay"
        )
        .classList.remove("open");

}


/* ================= DETAIL ================= */

function createLeadDetail(lead) {

    const reasons =
        generateReasons(lead);


    return `

        <div class="detail-score">

            <div class="detail-score-label">
                AI PRIORITY SCORE
            </div>

            <div class="detail-score-value">

                ${formatScore(lead.ai_score)}

            </div>

            <div class="detail-score-sub">

                Conversion probability:
                ${formatProbability(
                    lead.conversion_probability
                )}

            </div>

            <div style="margin-top:10px;">

                <span
                    class="priority-badge
                    ${priorityClass(
                        lead.priority_tier
                    )}"
                >

                    ${lead.priority_tier}
                    PRIORITY

                </span>

            </div>

        </div>


        <div class="detail-section">

            <h3>
                BUYER & REQUIREMENT
            </h3>


            <div class="detail-grid">

                ${detailItem(
                    "Product",
                    lead.product_category
                )}

                ${detailItem(
                    "Industry",
                    lead.buyer_industry
                )}

                ${detailItem(
                    "Location",
                    lead.buyer_location
                )}

                ${detailItem(
                    "Company Size",
                    lead.buyer_company_size
                )}

                ${detailItem(
                    "Quantity",
                    `${lead.quantity} units`
                )}

                ${detailItem(
                    "Inquiry Value",
                    formatINR(
                        lead.inquiry_value
                    )
                )}

                ${detailItem(
                    "Urgency",
                    lead.urgency
                )}

                ${detailItem(
                    "Lead Source",
                    lead.lead_source
                )}

            </div>

        </div>


        <div class="detail-section">

            <h3>
                BUYER HISTORY
            </h3>


            <div class="detail-grid">

                ${detailItem(
                    "Previous Orders",
                    lead.previous_orders
                )}

                ${detailItem(
                    "Previous Inquiries",
                    lead.previous_inquiries
                )}

                ${detailItem(
                    "Repeat Buyer",
                    Number(
                        lead.repeat_buyer
                    ) === 1
                        ? "Yes"
                        : "No"
                )}

                ${detailItem(
                    "Conversion History",
                    formatProbability(
                        lead.previous_conversion_rate
                    )
                )}

                ${detailItem(
                    "Buyer-Seller Inquiries",
                    lead.buyer_seller_previous_inquiries
                )}

                ${detailItem(
                    "Buyer-Seller Orders",
                    lead.buyer_seller_previous_orders
                )}

            </div>

        </div>


        <div class="detail-section">

            <h3>
                SELLER PROFILE
            </h3>


            <div class="detail-grid">

                ${detailItem(
                    "Seller Rating",
                    `${Number(
                        lead.seller_rating
                    ).toFixed(2)} / 5`
                )}

                ${detailItem(
                    "Seller Experience",
                    `${Number(
                        lead.seller_experience
                    ).toFixed(1)} years`
                )}

            </div>

        </div>


        <div class="detail-section">

            <h3>
                WHY PRIORITIZE?
            </h3>


            <div class="reason-list">

                ${
                    reasons.length
                    ?
                    reasons
                        .map(
                            reason => `
                                <div class="reason-item">
                                    ${escapeHTML(
                                        reason
                                    )}
                                </div>
                            `
                        )
                        .join("")
                    :
                    `
                        <div class="reason-item">
                            Strong overall lead profile.
                        </div>
                    `
                }

            </div>

        </div>


        <div class="detail-section">

            <h3>
                RECOMMENDED ACTION
            </h3>


            <div class="action-box">

                ${escapeHTML(
                    lead.recommended_action
                )}

            </div>

        </div>


        <div class="detail-section">

            <h3>
                ✦ AI RESPONSE ASSISTANT
            </h3>


            <button
                id="generate-response"
                class="generate-button"
            >
                ✦ Generate AI Response
            </button>


            <div
                id="response-container"
                style="display:none;"
            >

                <div class="ai-response">

                    <div class="ai-label">
                        SUGGESTED SELLER RESPONSE
                    </div>

                    <div
                        id="generated-response"
                    ></div>

                </div>


                <button
                    id="copy-response"
                    class="copy-button"
                >
                    Copy Response
                </button>

            </div>

        </div>

    `;

}


function detailItem(
    label,
    value
) {

    return `

        <div class="detail-item">

            <label>
                ${escapeHTML(label)}
            </label>

            <strong>
                ${
                    value !== undefined &&
                    value !== null &&
                    value !== ""
                    ?
                    escapeHTML(
                        String(value)
                    )
                    :
                    "—"
                }
            </strong>

        </div>

    `;

}


/* ================= EXPLANATIONS ================= */

function generateReasons(lead) {

    const reasons = [];


    if (lead.urgency === "Critical") {

        reasons.push(
            "Critical urgency — immediate response recommended."
        );

    }

    else if (lead.urgency === "High") {

        reasons.push(
            "High urgency — buyer requirement is time-sensitive."
        );

    }


    if (
        Number(
            lead.repeat_buyer
        ) === 1
    ) {

        reasons.push(
            `Repeat buyer with ${lead.previous_orders} previous orders.`
        );

    }


    if (
        lead.buyer_company_size === "Large" ||
        lead.buyer_company_size === "Enterprise"
    ) {

        reasons.push(
            `${lead.buyer_company_size} buyer profile.`
        );

    }


    if (
        Number(
            lead.inquiry_value
        ) >= 20000
    ) {

        reasons.push(
            `High inquiry value — ${formatINR(
                lead.inquiry_value
            )}.`
        );

    }


    if (
        Number(
            lead.quantity
        ) >= 10
    ) {

        reasons.push(
            `Large requirement — ${lead.quantity} units.`
        );

    }


    if (
        Number(
            lead.seller_rating
        ) >= 4.5
    ) {

        reasons.push(
            `Strong seller rating — ${
                Number(
                    lead.seller_rating
                ).toFixed(2)
            }/5.`
        );

    }


    return reasons.slice(0, 4);

}


/* ================= AI RESPONSE ================= */

function generateResponseForLead(lead) {

    const container =
        document.getElementById(
            "response-container"
        );


    const output =
        document.getElementById(
            "generated-response"
        );


    let opening;


    if (lead.urgency === "Critical") {

        opening =
            "Thank you for your urgent inquiry. We are prioritizing your requirement.";

    }

    else if (lead.urgency === "High") {

        opening =
            "Thank you for your inquiry. We understand that your requirement is time-sensitive.";

    }

    else {

        opening =
            "Thank you for reaching out regarding your requirement.";

    }


    const repeatLine =
        Number(
            lead.repeat_buyer
        ) === 1
        ?
        " We appreciate your continued business with us."
        :
        "";


    const response = `

        ${opening}${repeatLine}

        We can assist with
        ${lead.product_category}
        for ${lead.quantity} units.

        We would be happy to confirm availability,
        pricing and delivery timelines based on your
        requirement.

        Please share any additional specifications or
        delivery requirements, and we will get back
        to you promptly.

    `.trim();


    output.textContent = response;

    container.style.display = "block";

}


async function copyGeneratedResponse() {

    const text =
        document
        .getElementById(
            "generated-response"
        )
        .textContent;


    try {

        await navigator.clipboard.writeText(
            text
        );


        const button =
            document.getElementById(
                "copy-response"
            );


        button.textContent =
            "✓ Copied";


        setTimeout(
            () => {
                button.textContent =
                    "Copy Response";
            },
            1500
        );

    }

    catch (error) {

        console.error(error);

    }

}


/* ================= HELPERS ================= */

function priorityClass(
    priority
) {

    if (priority === "High") {
        return "priority-high";
    }

    if (priority === "Medium") {
        return "priority-medium";
    }

    return "priority-low";

}


function urgencyClass(
    urgency
) {

    if (urgency === "Critical") {
        return "urgency-critical";
    }

    if (urgency === "High") {
        return "urgency-high";
    }

    return "urgency-normal";

}


function formatINR(value) {

    const number =
        Number(value || 0);


    if (number >= 10000000) {

        return (
            "₹" +
            (
                number / 10000000
            ).toFixed(1) +
            " Cr"
        );

    }


    if (number >= 100000) {

        return (
            "₹" +
            (
                number / 100000
            ).toFixed(1) +
            " L"
        );

    }


    return (
        "₹" +
        Math.round(
            number
        ).toLocaleString(
            "en-IN"
        )
    );

}


function formatScore(value) {

    return Number(
        value || 0
    ).toFixed(1);

}


function formatProbability(value) {

    return (
        Number(
            value || 0
        ) * 100
    ).toFixed(1) + "%";

}


function escapeHTML(value) {

    return String(value)
        .replace(
            /&/g,
            "&amp;"
        )
        .replace(
            /</g,
            "&lt;"
        )
        .replace(
            />/g,
            "&gt;"
        )
        .replace(
            /"/g,
            "&quot;"
        )
        .replace(
            /'/g,
            "&#039;"
        );

}


/* ================= START ================= */

loadCSV();
"""

js_path.write_text(js, encoding="utf-8")

print("✅ Frontend updated to use all 25 dataset fields.")
print(js_path)

✅ Frontend updated to use all 25 dataset fields.
/Users/sameekshadongre/Documents/LeadIQ/product/app.js


In [127]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
css_path = project / "styles.css"

css = css_path.read_text(encoding="utf-8")

css += r"""

/* ================= RESPONSE BUTTONS ================= */

.generate-button {
    width: 100%;
    border: none;
    border-radius: 9px;
    padding: 11px 14px;
    background: linear-gradient(
        135deg,
        #8b5cf6,
        #6366f1
    );
    color: white;
    font-weight: 700;
    font-size: 12px;
    cursor: pointer;
    transition: .15s ease;
}

.generate-button:hover {
    transform: translateY(-1px);
    box-shadow:
        0 5px 20px rgba(139,92,246,.25);
}

.copy-button {
    margin-top: 9px;
    border: 1px solid #303a55;
    background: #151b2b;
    color: #cdd4e1;
    border-radius: 7px;
    padding: 8px 12px;
    cursor: pointer;
    font-size: 11px;
}

.copy-button:hover {
    border-color: #586580;
}
"""

css_path.write_text(css, encoding="utf-8")

print("✅ Response button styling added.")

✅ Response button styling added.


In [128]:
from pathlib import Path
import pandas as pd

project = Path.home() / "Documents" / "LeadIQ" / "product"

csv_path = project / "leadiq_seller_view.csv"

complete_view = pd.read_csv(csv_path)

# Rank-based seller-facing Priority Score
complete_view["priority_score"] = (
    100 * (
        1 - (complete_view["rank"] - 1) /
        (len(complete_view) - 1)
    )
).round(1)

# Keep the actual ML probability separately
complete_view["conversion_probability"] = (
    complete_view["conversion_probability"] * 100
).round(2)

complete_view.to_csv(csv_path, index=False)

print("Updated:", csv_path)
print("\nTop 5 leads:")
print(
    complete_view[
        [
            "rank",
            "priority_score",
            "conversion_probability",
            "priority_tier"
        ]
    ].head()
)

Updated: /Users/sameekshadongre/Documents/LeadIQ/product/leadiq_seller_view.csv

Top 5 leads:
   rank  priority_score  conversion_probability priority_tier
0     1           100.0                   44.62          High
1     2           100.0                   43.94          High
2     3            99.9                   43.49          High
3     4            99.9                   43.14          High
4     5            99.9                   42.76          High


In [129]:
# Recalculate Priority Score using percentile ranking
complete_view["priority_score"] = (
    100 * (
        1 - complete_view["rank"] / len(complete_view)
    )
).round(1)

# Ensure the top lead gets 100
complete_view.loc[complete_view["rank"] == 1, "priority_score"] = 100.0

# Save updated dataset
complete_view.to_csv(csv_path, index=False)

print("Top 10 leads:")
print(
    complete_view[
        [
            "rank",
            "priority_score",
            "conversion_probability",
            "priority_tier"
        ]
    ].head(10)
)

Top 10 leads:
   rank  priority_score  conversion_probability priority_tier
0     1           100.0                   44.62          High
1     2            99.9                   43.94          High
2     3            99.9                   43.49          High
3     4            99.9                   43.14          High
4     5            99.8                   42.76          High
5     6            99.8                   42.54          High
6     7            99.8                   42.37          High
7     8            99.7                   42.35          High
8     9            99.7                   41.99          High
9    10            99.7                   41.87          High


In [130]:
# Create a seller-facing Priority Score based on relative model strength

min_prob = complete_view["conversion_probability"].min()
max_prob = complete_view["conversion_probability"].max()

complete_view["priority_score"] = (
    100 * (
        (complete_view["conversion_probability"] - min_prob) /
        (max_prob - min_prob)
    )
).round(1)

# Save updated dataset
complete_view.to_csv(csv_path, index=False)

print("Top 10 leads:")
print(
    complete_view[
        [
            "rank",
            "priority_score",
            "conversion_probability",
            "priority_tier"
        ]
    ].head(10)
)

Top 10 leads:
   rank  priority_score  conversion_probability priority_tier
0     1           100.0                   44.62          High
1     2            98.2                   43.94          High
2     3            97.1                   43.49          High
3     4            96.2                   43.14          High
4     5            95.2                   42.76          High
5     6            94.6                   42.54          High
6     7            94.2                   42.37          High
7     8            94.1                   42.35          High
8     9            93.2                   41.99          High
9    10            92.9                   41.87          High


In [131]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
js_path = project / "app.js"

js = js_path.read_text()

# Use Priority Score as the main seller-facing score
js = js.replace(
    "lead.ai_score",
    "lead.priority_score"
)

# Add explicit definitions to the detail drawer
old = """<div class="score-label">AI PRIORITY SCORE</div>
<div class="score-value">${lead.priority_score}</div>"""

new = """<div class="score-label">AI PRIORITY SCORE</div>
<div class="score-value">${lead.priority_score}</div>
<div class="score-definition">
    How strongly LeadIQ recommends prioritizing this lead relative to other incoming inquiries.
</div>"""

if old in js:
    js = js.replace(old, new)

# Add probability definition
old_prob = """<div class="detail-stat">
    <span>Conversion Probability</span>
    <strong>${lead.conversion_probability}%</strong>
</div>"""

new_prob = """<div class="detail-stat">
    <span>Predicted Conversion Probability</span>
    <strong>${lead.conversion_probability}%</strong>
    <small>Estimated probability that this inquiry converts, based on the ML model.</small>
</div>"""

if old_prob in js:
    js = js.replace(old_prob, new_prob)

js_path.write_text(js)

print("Updated:", js_path)
print("Priority Score is now the main seller-facing score.")

Updated: /Users/sameekshadongre/Documents/LeadIQ/product/app.js
Priority Score is now the main seller-facing score.


In [132]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
css_path = project / "styles.css"

css = css_path.read_text()

css += """

/* Priority Score Definitions */
.score-definition {
    margin-top: 8px;
    max-width: 420px;
    font-size: 12px;
    line-height: 1.5;
    color: #94a3b8;
}

.detail-stat small {
    display: block;
    margin-top: 5px;
    font-size: 11px;
    line-height: 1.4;
    color: #64748b;
}

.score-label {
    letter-spacing: 0.08em;
    font-size: 11px;
    font-weight: 700;
    color: #a78bfa;
}

.score-value {
    margin-top: 4px;
    font-size: 42px;
    line-height: 1;
    font-weight: 800;
}
"""

css_path.write_text(css)

print("Updated:", css_path)
print("Priority Score styling added.")

Updated: /Users/sameekshadongre/Documents/LeadIQ/product/styles.css
Priority Score styling added.


In [133]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
css_path = project / "styles.css"

css = css_path.read_text()

css = css.replace(
    """font-size: 42px;
    line-height: 1;
    font-weight: 800;""",
    """font-size: 28px;
    line-height: 1.1;
    font-weight: 800;"""
)

css_path.write_text(css)

print("Priority Score font reduced to 28px.")

Priority Score font reduced to 28px.


In [134]:
from pathlib import Path

project = Path.home() / "Documents" / "LeadIQ" / "product"
css_path = project / "styles.css"

css = css_path.read_text()

css = css.replace(
    """font-size: 28px;
    line-height: 1.1;
    font-weight: 800;""",
    """font-size: 22px;
    line-height: 1.2;
    font-weight: 700;"""
)

css_path.write_text(css)

print("Priority Score font reduced to 22px.")

Priority Score font reduced to 22px.
